# Florence-2-large — DIMER E2E handwritten-line OCR adaptation tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/florence2-vision-language-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/florence2-vision-language-pipeline/blob/main/tutorials/florence2_vision_language_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-florence--community%2FFlorence--2--large-ffcc4d?style=flat)](https://huggingface.co/florence-community/Florence-2-large) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2FFlorence--2--large-181717?style=flat&logo=huggingface&logoColor=white)](https://huggingface.co/microsoft/Florence-2-large) [![arXiv](https://img.shields.io/badge/arXiv-2311.06242-b31b1b.svg)](https://arxiv.org/abs/2311.06242)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.2 — **standalone** (§4)  
**Capability:** prompt-selected vision-language tasks (caption, detection and OCR demonstrated) and bounded supervised fine-tuning of the `<OCR>` task's last decoder layers on transcribed text lines, using the pinned Florence-2-large weights

**This notebook is standalone.** It carries the repository's package (3 modules under `src/florence2_vision_language_pipeline/`, at revision `0d15dffb95c8`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned Python distributions and the Hugging Face Hub at the immutable revision `4271c66b88cdbc05735372ec13b2360108de5317` (~1559 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2.1); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime builds an isolated environment from the hash-locked pins (the kernel's own packages are left alone, so no restart is needed), stages and digest-verifies the pinned `florence-community/Florence-2-large` snapshot (a 1.54 GB `model.safetensors`; no pickle is opened anywhere), fetches the first eight row groups of the Belfort-line test shard from the Hugging Face Hub at an immutable revision with HTTPS range requests (about 44 MB; each row group refused on any SHA-256 or byte-total mismatch), validates the 800 line records and splits them by line image into 600 / 60 / 140, runs three capabilities (`<CAPTION>`, `<OD>`, `<OCR>`) on a synthetic drawing through the inference contract with a combined input manifest and a rejection probe, measures the frozen model's `<OCR>` character and word error rates over the 140 held-out lines beside an empty-string and a constant-transcript baseline, runs a bounded fine-tuning of the last four BART decoder layers on cached encoder outputs with validation-CER epoch selection, scores the held-out lines again, re-runs six held-out lines and the three capabilities on the drawing with the adapted model, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify transcript parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.2 §5). A CUDA runtime is used automatically when present, and **a GPU runtime is required in practice**: on a Tesla T4 the model stages of the previous notebook version took about 20 minutes of cell time (six epochs 822 s, frozen scoring of 140 lines 90 s), while on CPU the beam-search decoding of some 800 lines plus 600 cached forwards of a 777M-parameter model take hours. Building the isolated environment comes on top of this.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4, select that cell and choose **Runtime → Run after** (it re-runs Section 4 and every later cell; Sections 5, 6 and 7 reload the pretrained model first, so the earlier adaptation never carries over into a *frozen* number). Supply one zip through the upload dialog on Colab, or a zip or folder path in `BYOD_PATH` on any runtime. It holds line images plus a `transcripts.csv` (`file`, `text`, optional `id`; one row per image). The archive is read without its folders, so every file name must be unique. It needs **at least 12 distinct images**: the seeded split keeps 20 % for test and 15 % for validation and must leave at least 8 training, 2 validation and 2 test lines; Section 4 checks this before any model runs, names the split it refuses, and reports any pixel-duplicate images it drops. The records pass through the same validation, image-disjoint split, baselines, fine-tuning, held-out evaluation, artifact export and reload-parity cells as the Belfort sample; a result where the adapted model does not beat the frozen one is reported, not treated as an error. Uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

Florence-2 is one sequence-to-sequence model whose behaviour is selected by a **task prompt token**: the image is resized to 768 × 768 by the processor (aspect ratio is not preserved), encoded by a DaViT vision tower into 577 visual tokens that a BART encoder reads together with the prompt, and a 12-layer BART decoder generates text that the processor parses per task — plain text for captions and OCR, boxes plus labels in input-pixel coordinates for region tasks (776,505,344 parameters in all, published under the **MIT** licence). Decoding is **deterministic beam search** (`num_beams` = 3 from the snapshot generation config, `do_sample=False`). The output carries **no score**: captions and transcripts are plain generated text, boxes have no confidence.

What this notebook adds to inference is **adaptation of one task on transcribed lines**. Florence-2's `<OCR>` was trained on printed and scene text; nineteenth-century French council minutes in cursive handwriting — the Belfort-line dataset — are far outside that distribution, and on them the frozen model reads nothing: it emits a dash or an empty string for almost every line, a character error rate of **0.992** on the 140 held-out lines (the build record's Tesla T4 figure), the empty baseline's 1.0 in all but name. So the honest question is narrow: does a bounded fine-tuning of the last four decoder layers on 600 transcribed lines move the held-out **CER** and **WER** on a held-out split of line images past two **non-adapted baselines** and the frozen model — and what does it do to the other tasks the same decoder serves? Nothing here is a claim about your documents or your script: it is one seeded split of one small labelled set, and the split is by line image, not by page (Section 4 explains why that matters).

**Snapshot note:** the pin is the community "official transformers converted checkpoint" (`florence-community/Florence-2-large`), loadable by native `transformers` classes with `trust_remote_code=False`; the original `microsoft/Florence-2-large` snapshot needs remote code and was rejected (see the weight provenance document). The pinned revision ships `model.safetensors` (a 12-file manifest with the tokenizer files) — no pickle is opened anywhere in this notebook. Section 3 stages and digest-verifies those files before the processor or the model is constructed, and the loader refuses a checkpoint whose weights do not map cleanly onto the native architecture. The pipeline loads the checkpoint in **float32 on every device**: the adapter is trained in float32 and overlays without a cast, and CPU, Tesla-class and consumer GPUs then run the same arithmetic.

**Who this is for.** A learner who knows basic Python and PIL and has met the idea of an encoder–decoder model, and wants to see how a pretrained document model is measured on labelled handwriting and adapted to it. No prior experience with Florence-2, OCR metrics or fine-tuning is assumed; each term is explained where it is first used and again in the **Glossary** at the end. A **GPU runtime** (for example a Colab or Kaggle T4) is needed in practice (Prerequisites).

**Input → Model → Output.**

| | Inference | Fine-tuning |
|---|---|---|
| Input | one image (1 to 16,384 px a side) and a task prompt token such as `<OCR>`; `<CAPTION_TO_PHRASE_GROUNDING>` also takes a `text_input` | 600 training line images, each with its transcript as the target text |
| Model | Florence-2-large: DaViT vision tower → BART encoder over 577 visual tokens + the prompt → 12-layer BART decoder, 3-beam search | the same model with the last 4 decoder layers and the decoder's embedding layer norm trained, everything else frozen |
| Output | the task-parsed `result` (a caption or transcript string, or `{bboxes, labels}` in input pixels) and the raw generated text — no score | an adapter (safetensors) and held-out CER / WER / exact-match beside two baselines and the frozen model |

**How to use this notebook.** Choose a GPU runtime, then **Runtime → Run all**. Sections 1–3 are **infrastructure** — the isolated environment, the carried code and the model verification — and can be run without study; their code is collapsed. The learning path starts in Section 4. Form fields (`# @param`) are the only values meant to be edited; the defaults reproduce the recorded path. Each stage asks you to **predict** before it runs, and the next cell opens with **What to notice** and a collapsible **Check your reasoning** block with a worked answer from the recorded Tesla T4 run of the previous notebook version (same model stages; your GPU's numbers can differ in the second or third decimal). Each optional experiment names the cell to change and the cells to re-run. Section 10 is a **change-one-thing activity**. **Troubleshooting**, a **Glossary** and a **Conclusion** template are at the end. Your notes are optional and are not required submissions.

**Roadmap:** 4 fetch, validate and split the lines → 5 three capabilities on a drawing with the pretrained model → 6 two baselines and the pretrained model on the test lines → 7 fine-tune the last decoder layers → 8 compare on the held-out lines → 9 look at the lines, re-run the capabilities, export and reload → 10 **change one thing: a higher learning rate** → conclude. Core concepts are Sections 5 and 6 (what the model emits and how it is scored); evaluation practice is Sections 4, 7 and 8 (split, selection, held-out comparison); engineering and reproducibility are Sections 1–3 and 9.

**Learning objectives:** by the end you should be able to (1) explain *image + task token → encoder → decoder → task-parsed text or boxes* and say why none of the outputs carries a score (Section 5); (2) read corpus CER and WER against an empty and a constant-transcript baseline, and say why a rate above 1.0 means the model is generating rather than reading (Section 6); (3) explain how the validation split, never the test split, chooses the kept epoch (Section 7); (4) predict and then check how CER, WER and hypothesis length move after a bounded fine-tuning, and say why CER can fall while WER does not (Section 8); (5) say what a line-image split does and does not protect against when the data has no page field (Sections 4 and 8); (6) check that an exported adapter reproduces the evaluated model (reload parity, Section 9); and (7) predict and then observe what a higher learning rate changes (Section 10).

**This notebook does not demonstrate:** the other task tokens in `TASKS` are exposed but not demonstrated by default — `<DETAILED_CAPTION>`, `<MORE_DETAILED_CAPTION>`, `<DENSE_REGION_CAPTION>`, `<REGION_PROPOSAL>`, `<OCR_WITH_REGION>` and `<CAPTION_TO_PHRASE_GROUNDING>` (the only task that takes a `text_input`; an optional experiment runs it); segmentation of any kind, open-vocabulary detection, visual question answering, confidence scores for boxes or text, fine-tuning of the vision tower, the projector, the encoder, the embeddings or the first eight decoder layers, fine-tuning of any task but `<OCR>`, evaluation on an OCR benchmark proper (only one seeded 800-line sample is scored here), a page- or writer-disjoint split (the dataset carries no page or writer field), and any claim that French cursive minutes stand in for your documents. The repository exposes none of these.

## Prerequisites

- **Learner:** basic Python and PIL, and Colab or Jupyter familiarity; what a sequence-to-sequence model's task prompt and generated tokens are; what a bounding box in pixel coordinates is. No prior experience with Florence-2, OCR metrics or fine-tuning: the notebook explains task tokens, beam search, CER and WER, the medoid baseline, the frozen encoder cache, teacher forcing and validation-based epoch selection where they are first used, and the Glossary repeats them.
- **Runtime:** a fresh **Linux x86_64** runtime with a CUDA GPU — Google Colab, Kaggle or Linux Jupyter. Section 1 builds its own Python 3.12.12 environment from a hash-locked list of manylinux wheels, so the kernel's own Python version does not matter, and a Windows or macOS kernel is not supported (Section 1 stops with that message). Generation is batched for the corpus stages — every `<OCR>` prompt is the same 587 tokens (577 image tokens plus the task token), so a batch needs no padding — and the build record measured 90 s to score 140 lines with 3-beam search and 822 s for the six epochs (caching the encoder outputs for 600 lines took 371 s) on a Tesla T4, about 20 minutes of cell time for the model stages; a CPU runtime would take hours. The locked install (PyTorch 2.14.0 with its CUDA libraries, a few GB) and the 1.54 GB checkpoint are the large downloads of the run; the row groups are about 44 MB.
- **Data contract:** records are `{id, image, text}` — `image` a PIL image (or a file decodable by Pillow) with sides within 1..16,384 px and at most 4096² pixels, `text` its transcript (1..`MAX_TRANSCRIPT_CHARS` = 512 characters after whitespace runs are collapsed; case and punctuation kept). Ids match `[A-Za-z0-9_.:-]{1,64}` and are unique; a training split needs 8..5,000 records. Splitting de-duplicates by decoded pixels so no image lands in two splits, and the seeded BYOD split (65 / 15 / 20 %) must leave at least 8 training, 2 validation and 2 test lines, so **a BYOD dataset needs at least 12 distinct images** (Section 4 checks it before any model runs and names the split it refuses).
- **BYOD layout:** one zip (by upload on Colab, or by path in `BYOD_PATH` on any runtime; a folder path also works) holding the line images and a `transcripts.csv` with the columns `file` and `text` (optional `id`). The archive is read without its folders, so every file name must be unique; two files with one name are refused with a message saying so.
- **Validation is structural, not semantic:** every image is decoded and every transcript checked for length, but nothing checks that a transcript says what its image shows — a mislabelled set is fine-tuned on without complaint.
- **Expected output:** a "slow image processor" notice from `transformers` is expected and harmless. A `RuntimeError: checkpoint does not match the native Florence-2 architecture` means the staged weights are not the pinned converted checkpoint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there. The default path uploads nothing.
- **External access (data):** besides the model snapshot, the default path reads eight row groups of `default/test/0000.parquet` from `https://huggingface.co/datasets/Teklia/Belfort-line/resolve/<revision>/` at the immutable parquet-conversion revision `c4a74bbd…` with HTTPS range requests (the parquet footer plus about 44 MB of row-group bytes out of a 210 MB shard), each row group pinned by SHA-256 and byte total in the carried `samples.py` and refused on any mismatch. Belfort-line is published under the MIT licence (Teklia; Tarride et al. 2023); nothing is redistributed by this repository.
- **External access:** the Hugging Face Hub only, to fetch the pinned `florence-community/Florence-2-large` snapshot (~1559 MB in total) at revision `4271c66b88cd…`. No repository access and no credentials are required; nothing is installed from this repository. The isolated environment also needs PyPI (`pypi.org`, `files.pythonhosted.org`) for the pinned `uv` wheel and the 48 hash-locked packages, and the managed CPython 3.12.12 build (python-build-standalone) that `uv` downloads.

## 1. Install the pinned runtime (isolated environment)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

Hosted runtimes such as Colab import some packages, NumPy among them, before the first cell runs, and Python cannot swap a module that is already loaded. Installing the pins into the notebook's own Python would therefore leave mixed versions or require a manual restart. Instead, the next cell builds a separate environment (`dimer_isolated_env/`) and leaves the kernel's packages untouched: it downloads the pinned `uv` 0.12.15 wheel and refuses it unless its size and SHA-256 match, has `uv` install the managed CPython **3.12.12** (whatever Python the kernel itself runs), and installs the 48 packages of the carried hash-locked requirements (`tutorials/requirements-colab.lock.txt`, compiled from the pins below) with `--require-hashes --only-binary :all:`, so every package, direct or transitive, is the exact file that was locked. No repository code is installed. The lock holds manylinux x86_64 wheels, so the notebook supports **Linux x86_64 runtimes only** (Google Colab, Kaggle or Linux Jupyter); on any other platform the cell stops with that message. The printed dictionary names the isolated Python version and the kernel's.

In [ ]:
# @title Infrastructure: install the locked runtime into an isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import hashlib
import io
import os
import platform
import subprocess
import sys
import time
import urllib.error
import urllib.request
import zipfile
from pathlib import Path

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'pyarrow==25.0.1',
]
MANAGED_PYTHON = '3.12.12'
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_BYTES = 20081404
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
LOCK_NAME = 'requirements-colab.lock.txt'
LOCK_SHA256 = 'a55548d4e338fb504e610b965eb5cca22bd3017c35a22c7472a2550d75109d64'
LOCKED_PACKAGES = 48
# The hash-locked requirements, compiled from the PINS above with `uv pip compile --generate-hashes` for manylinux x86_64.
LOCK_TEXT = r'''# This file was autogenerated by uv via the following command:
#    uv pip compile pyproject.toml --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --only-binary :all: -o tutorials/requirements-colab.lock.txt
certifi==2026.7.22 \
    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \
    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55
    # via requests
charset-normalizer==3.5.2 \
    --hash=sha256:01077390b03f7988f11d700a2194e69b119741a86b1a638b1db88891e3eced8e \
    --hash=sha256:01b0c0d2262a9e28e8484a278c7e1b5d650e3ac8cf2683d2967e25899f208bdf \
    --hash=sha256:04851f73ae72b8413dddadb16a49dfee95263553741fd42d546f7d66907e6be5 \
    --hash=sha256:0521c5665880b33d603717defa76c094048900010897909952397feb3039da56 \
    --hash=sha256:0774bf9bf620249fee3e0b8b9fd3065de213be30f3aa94ce2494b3b638949e26 \
    --hash=sha256:0891b9d3903c5571c03771ca669a4b0ec5618ca722a5c957d3d29cd4e5062848 \
    --hash=sha256:0c951d5e6dd9c2ff60609476752bee49da4206adde960ebc247766937f72e718 \
    --hash=sha256:0fed1d06615f022ee3b13caf5e8b180cfea32bb2c5aded8a9d44277afc040f93 \
    --hash=sha256:114e4d0c92d618409ed82a99e22b5c5e768fe995f2973f78265f4524f49d4640 \
    --hash=sha256:11912e4bb14baae7c5d8791aa55ba0a3a03ec6729073307b0f57270abaa713d3 \
    --hash=sha256:11a4d68a6ecda3292cb1e50239e111543ba5d709bb62a6b4ea1afcfa729d8875 \
    --hash=sha256:124fbf1a8ff966d87ae05bb8bd45a71f966055ed8bba320d0c7cf450bc5f4d0e \
    --hash=sha256:1461ac396c4fdb983a675f20aa555624f0ee18ac83d832b9244ffff3d8055275 \
    --hash=sha256:1503bccbeb36d5527790c3930327704c39af22de3112f1b1666a9f3ce15ee204 \
    --hash=sha256:15bb4005af6320d259dc7593ca84a38d7fe06a421dbcf7b910ae23979101e787 \
    --hash=sha256:15c44f7edfd477b06f517a5cc317fc1707edb9de2c865f43d4b6513907473234 \
    --hash=sha256:16fa0eccf81304b79c5cd87f9271c3b85dd9dd99245e4422ae9c0dd45e0f99d3 \
    --hash=sha256:183b88127acdb4fabe59d951ab424faf1af7b63cdbb5f776186c1ea2ffcaed98 \
    --hash=sha256:195c26fb65950f8fce54e26349852b7bdd7c5f120aeefbcc440b8a20faaed4a3 \
    --hash=sha256:1afb975bd5d68d5ce9f6b6d44fdf2f7e34b895a35e95708a7a91b20a3b51d187 \
    --hash=sha256:1b4cbc7c3491ccb4aa17fcd8165649d01cf39f76de1696da8631b5f71b85401d \
    --hash=sha256:1bc0baf5ef96b6ede57d47f4b8fe4d9d84019c3bfcbeb20a41edc6a6ee341f1f \
    --hash=sha256:1c50fe28bbc2ced33386f298650d91218076c05420e6cbd790b913adc41659e7 \
    --hash=sha256:1db38f4c5496827c1a501846d64d14c3b80c7e6714e406cd7dc36a9899fa1011 \
    --hash=sha256:211d5a3eb6af8f513b8d4ca19a8c1b7accab1b5f0d3175f9826b03c1a920dc1f \
    --hash=sha256:23851fb4e1b85ed3f6c2a27b777cdfe2e19fb5b38429a8faf38c7542b7665869 \
    --hash=sha256:254eb48b9fa5ee9898a3c445825a1f340fe53712a098904b39b0bddba8ea3cb1 \
    --hash=sha256:2625388c6c754520c37abaf3b41eb34d1cc4a373f457898f08606c8e362b891d \
    --hash=sha256:281cb91036248400f4cc957495cccd44c275c2e0c5854f7e45ac5cf7dc193847 \
    --hash=sha256:28a15fdad492a99b6eccfaaed66ef3f74050680545ea61ec8b2f4c538f1f1320 \
    --hash=sha256:28b4f0d66fb834ff90f28209ac7bce77868c45d8c93e26f906709d9b7c2e1af9 \
    --hash=sha256:2a925889534b3748302dae5dead07cc13480de1dac3aea80a941b729b471ef93 \
    --hash=sha256:2b7b3bbfb4fe8ef40600792d762fbaa9057559f9d3fad209525b7a22b99e91fd \
    --hash=sha256:2c9ad19a6cfcd5ea5c0d41161d22f9df1dcc277e9bef2751391334546a314c00 \
    --hash=sha256:2cc961b171b3f3440f410489ab3573e86aea8736134ebbb40ea1338b7f0831bc \
    --hash=sha256:2ce45c6627b22c47e390bc91a41c3d13032192e699fa0bea96e9671b373d69b0 \
    --hash=sha256:2e06a3a98f916dd41d27f3105e02e7a40181c98c94b9158733d03a6f80506c09 \
    --hash=sha256:304d5463e65a35d7bb0850550e0780395395f6fcf452f04db7d5ca7cecc425ac \
    --hash=sha256:304d8e4d493af723536393eee0c689eb7813f4a474c8b479dee63f1fdd98f621 \
    --hash=sha256:30fcd120b732aa79317f08dee04d7de0847822e4cf7ee0e9f445bb958832252c \
    --hash=sha256:31f3930700408d211f13378ccbe1c40845d8da54bd0681fac3a9b5aae81c7aa8 \
    --hash=sha256:34276fd796040bf0993ab33a369aa572e6979c7aab225a88893667ad8eac8f7a \
    --hash=sha256:355ad8011081dec5412240c087a9a0c9d4d5039f3ed11a3f13e18c2b29b56c51 \
    --hash=sha256:38a873987f3be698494da8b2e3085e29da02da7b633dce73e79c699a113d7bf0 \
    --hash=sha256:39de2a259fc954455c57274dc94c79d5842774e1247a016aff30bc0efed0f4ef \
    --hash=sha256:3d14b50de6bf4d0edf857a9386836846f982b8f524e188e2e68b96d702bcf4aa \
    --hash=sha256:3d21b8b13c7592db2ac5e544a6d83187b995257472b0c9e8351b6d507ae37ed6 \
    --hash=sha256:3d31298449090ab8d47b7b1b2a555ff73cac7ed438a08b7ac160980c7ebed649 \
    --hash=sha256:3ddacd27458c45bdacd6bd6db644bfb730efbf9e830310186e3045c9c5be8fb2 \
    --hash=sha256:3df041de8887954562c9b261cba85ca0e9ded74048daf125f45edcfaa4832229 \
    --hash=sha256:40ab6bffa02ae10a0581e6c198be7d2d8ca5c2a0c64e4ed3465d766df457573e \
    --hash=sha256:4275811936e2f06feff5e598fb42a1b7ae852da8e39605211892b56b81a34efd \
    --hash=sha256:443eae2bf318abeaf6f15d785138f71fd6de770e99a92158b8b814265e079115 \
    --hash=sha256:447441e76ec720b15e64418d32e092297340387053047c7c694f579efb0ee1d9 \
    --hash=sha256:4495c5002a7b28557e7e222e77e0b661183e432b7d6d2e788101e3f240e05b8c \
    --hash=sha256:44bd4fbb29dfbeba60e7d2bd000c59e4b21ddb3cc53912b14048d37092706d7c \
    --hash=sha256:4685902cf26edf013ed7a3da0f426ebba7a00ebb9541386d835afbf002c11cab \
    --hash=sha256:498dc3188ca05a68231ac3fdbfc7f57eb67e1343c30e0fea17f8218c1599b253 \
    --hash=sha256:4c2b5031f63e331e3839b40aed2dd6f191e9c07edbde303e7876846ea1946995 \
    --hash=sha256:4d48f2d08b9de5864e2c8744d4461b862fb149a18274abc8b698c45975573438 \
    --hash=sha256:4f87960d57feabfb618e4e0af6e7371645fa26a277860739d6e5d6e0012c92f0 \
    --hash=sha256:50e3adfb96fc189eb27b1cf62d3b598b89b4bb0420d93a3d3e42e137409011be \
    --hash=sha256:51cf45226a9b588d0d2b4880c62d686934b63ab0bd79ca23ab0e9762eb27441b \
    --hash=sha256:52aa6992700996af31f375de0c6bacd402b0097fe40b53c426b9f51a90ebabc7 \
    --hash=sha256:55ea99acb17b9325618de155a0cd6a2e8f5d10be008113e1d433bbb58db543b2 \
    --hash=sha256:56bc200a365efb37383b7852e4cc5898d3b2da5987289b543956cf8cad71018a \
    --hash=sha256:588461c2e8384d309bd63e5826019b6977bc66d629b99ac8737bb795d7b2cb5a \
    --hash=sha256:58ca3755ee7ff7f59b57789ec9833c9de9ea275405cdd240eda1f193112e398a \
    --hash=sha256:58f361dcbab699cf8f42db3f47c8e7fd1036f138c23a5d08de9fde5f425a730c \
    --hash=sha256:598a11a2c7ebaa5334bf698bf29568c9c390abac6a154d8170fedecd1cea38c5 \
    --hash=sha256:59f63901b0031c3136cf64704dcb21de0bbae62ce2c9529bc39d27665463de37 \
    --hash=sha256:5cde776b7cc66e4f6c99612cea4aa7269aa65863f7a15841b2c264f103822f4e \
    --hash=sha256:5e2b6b57e9733d39f0c9fd3185efa6b8e29652c4cd8fe94180272cf6ed9a78c4 \
    --hash=sha256:5fb29fb8cd1a46c27a1bf9613ad5ec2599310d46b4025d9556404a6b6a292800 \
    --hash=sha256:6045373d5a89a5ec71afde535db987ca28e76dfa276c2d4c818265b375d4b055 \
    --hash=sha256:619799369eeef6366ed3e8755a5670f4f2f0fb6b30a0fd7264dc0fdc2357058e \
    --hash=sha256:62588a277bfb59def052abd940703fa35107152bf479781a878617d60faf8fb5 \
    --hash=sha256:62603db9a7caa0802eaa28c1c46fecd7b3a263a774069c24c3c28c302448721c \
    --hash=sha256:65cd72beeeca9d3aaea1201e5923859f308f952f9c71de93f06063c79f0f7a3b \
    --hash=sha256:68eb192d85ab8e5f6ec69c2bc6ac0179fbf04a5ac1569d12fbef74883fe102d0 \
    --hash=sha256:6bd128f206a7752ae1f2ab6c61bf8a24ba28913a10df8b14c2637b973ff97a80 \
    --hash=sha256:6be488a102b8cf28d0391d8c4ba7748938ae28b78ad901f8585520fca33ead1a \
    --hash=sha256:7218e8f32b0956cfcd048fd42d9d5779809745ca1d86113ca56f66e7ae1549c4 \
    --hash=sha256:7441d755b7ab94f8d4eb3e43ec05482d760842fd263d003a99102d742cd835e2 \
    --hash=sha256:749e97e1b32313717a565abbe321bc2190bc8b35f1a67e4cdbc7c56c8d8ffe58 \
    --hash=sha256:75a3ceed0724d625d64b86ca20aba182e4df462e04c2414fc941c0f523f06aac \
    --hash=sha256:780fbe7cab297b81dad9fb8dc5eb003c0468ffb0d9e5f65068c53a34661a96bc \
    --hash=sha256:78456a747de8dc58360ffa581f30a002baf5aa28cb262536545e91f113ed7639 \
    --hash=sha256:7967d08cf06dee78443b874f98c98036f624f3a4e73e11f9f64f5be4d25393cf \
    --hash=sha256:7a881931aa470808df94a8c380eed2bbbc76cd9dc622310f99665658c821eb6d \
    --hash=sha256:7dcd882da75ef9adf94903b1e3b9419e8aa8fb4c7396822b834b9ef7fb96954f \
    --hash=sha256:7e841fb9010836c992c9f12fcbd43a831de93a5f726fc1ccd8ca1d0268c5014c \
    --hash=sha256:7fdde2c9fd9e3eca40631e024664cf2584272cc8f96308cbe5fdfc930f51d8bc \
    --hash=sha256:8024d00c3faf3fc0c16e07a69f4405e8eac7cc0ab15f65fe6cf43827c4cf72b4 \
    --hash=sha256:80d02b6f04e92601a081dd97b23d3128033098bff5d35d392ddcc0476ea11253 \
    --hash=sha256:838dcc90063569a0448120554591a1d6c4a4ffe11babf048908793154ab86ade \
    --hash=sha256:849df64e889b2e17230d58410a03dba311a65b163508fd33679b2b737d4b7858 \
    --hash=sha256:87475fabc8d9996fd9c27debb395e642e8c838d78a00b6e932227a0e06b81e26 \
    --hash=sha256:87e50a3e7cb90af586b6c5faf23e302a970415ac73bd7bd90a515a04b427ef96 \
    --hash=sha256:89b53f3cda69831909888e0494f4fa0bcd3537e3e138dabeb620bd6ad946bae8 \
    --hash=sha256:8a893cc101149f80a653f82062ebc95b34525a2614382e1da5458fe7c6997249 \
    --hash=sha256:8b2bfab86aa71ae13aa41a6a26aab338e0db2b8bc75434b05aea89e011ff35a4 \
    --hash=sha256:8d86d6fc60743dc916eb79e2eb1ec4818e21e427731543af40a3021851174a13 \
    --hash=sha256:915563965d418f986e7e145accc592eae9e1a1be3566ff98a05d7a9ec42a76e1 \
    --hash=sha256:92888bb3187c5ba50500b00b3b310c9f2c651709d28036077680cb5255450a03 \
    --hash=sha256:93223adc95033dd47133a46ccfc316a0139176fd79085762e27202ec56018f03 \
    --hash=sha256:9373ad13ef0d2c0fb761e04e55bfdee5a08b52cef2c882c8fbe9935b1517152e \
    --hash=sha256:9409a8bf35cf78353942504b24a57de3d75b708997a1e4bd8db71ac8633ce364 \
    --hash=sha256:9b7f416ff0978e2f2249330527f0ad6fa02f4932e6199692d3b52da2048c19e4 \
    --hash=sha256:9bde855991b7e362c146535e3136a50bfaffc0487d38b33ca7e5edefc6e23849 \
    --hash=sha256:9cae88599c7219005d879f98e5ed53341e9a122af585e1091200358a3003d2a0 \
    --hash=sha256:9cf9b1a857e25c4baceeb3624e92a56df3668f398c4acba74e174d81fb4d1d3a \
    --hash=sha256:9f56f72050826f63dcee7a7f55b0a77168cb3bfc553fd405e7f8f9ece75a4036 \
    --hash=sha256:a090bb2c68df85450502e3e20d665e3a5af9c65a84d6508ed477badd49166fd3 \
    --hash=sha256:a192e2c40070d92c3ccf777e3a5c4ff515573cd2bb7ed0c537fdadbbec5bbf21 \
    --hash=sha256:a19a731138fc27d5682277d3b9df22855cea1239bce7fcec5f78f42ef2d1f3c3 \
    --hash=sha256:a66c3bc5ab1f0ff2164fc9965ddd611ff0802173f4b9d24554c563f6ab7e1d6e \
    --hash=sha256:a815775b6c38d4e0ff7bcffbeba67feded90202bb6a226b8dd35f1c855217413 \
    --hash=sha256:a89012d6d5476ee112d20d998570ed58df2260a852afb1758809cd6900411d21 \
    --hash=sha256:ae4f5fea5b8b8ccff88238cc8569303e5ee95efae67fa62922a311397a71f346 \
    --hash=sha256:b6856554c4f44d79fc2307d5768854310a8f0096e501c75637542c82292b0429 \
    --hash=sha256:b6b751274acb69d77b3323d6b7dbaa3c7fdfc1eb829b7eb61d262f32e1af9685 \
    --hash=sha256:b736353c0a625bbd5fcec108576e2385db3496f4f771f785ff32e108d3c3bc45 \
    --hash=sha256:b7fd005a73d9e657273b7a10dc71a9e03c8fb9ee6999798d6918ce095b81ac7f \
    --hash=sha256:b91363207bd9dc966a691e959bb47f64b30f7ac4b072be9968b366982f7db77c \
    --hash=sha256:ba0b1d2620edf869789c3879223f52bf2afc5d31b3cb47cc57b3a12c05e2aa9d \
    --hash=sha256:bbbfc8e28816f19d7c0f1816664980c0a9875d01b27cdf8eedddb639d9e108ad \
    --hash=sha256:bd16aabe4a02a297c23417aa17ac6299dbd8c49f673bcd645b4929b11f5a4400 \
    --hash=sha256:c0afc6800ba57ccc350374c5bd6150419915d95ce93cdbab2d783d75eaf30ecb \
    --hash=sha256:c6708715abcf3c73b99508253e961a9967f02fe536532834149574eda6de0d1c \
    --hash=sha256:c7c9ab723cde841fefb34efbad91e87f00a674b1fe1cd0784fde742bf2c154dc \
    --hash=sha256:c8f3d67aeaf55f017982b73683f0e7342ba2f6635a78f69ce89ebb26aa411e5c \
    --hash=sha256:c9790464842f85f437dbbb54417eda1e0e6bfc52dd8d22d6fd1c994b73b2dc74 \
    --hash=sha256:ca403d7e4798f525fdfc78e258820419cbbd0f0ecbab9de7840e3c017cf6b8cf \
    --hash=sha256:d008d90a7f2471519aef0c90dfbe73b3e6e4d5e66ac48e19154c17e89e98b604 \
    --hash=sha256:d19fbd981a488e22cd04883659ca6b08f50b5974f9fd7c95655ef6a043e5893f \
    --hash=sha256:d1befeed746d247c81127bb14de9dc3d30edb6e5976d34f83f86ed262b1d9105 \
    --hash=sha256:d2374b62878abb00cd8309b32af6c0b715cd02dec0ca74ef12e5069bdc64144a \
    --hash=sha256:d376bbd28b3a8999db1a103b3b388aee6f1ddeb3e51bc2172993efdcd86e064d \
    --hash=sha256:d4a7319f304a774bed22115bc891618e45f85065ab44ea6acd07d274e750519a \
    --hash=sha256:d6734d2ef8a50fbf8445c139477da401f50d62a0606bf00e20ec6d87773fefb1 \
    --hash=sha256:d760fe2a4d7c3b226cb9026d6a842868d52a7901bd98420e1baf14e80da85cf5 \
    --hash=sha256:d913de495d90407cd859d263bee2e5d1a4ed3eb6573c04e70d9ec619a7cbed7f \
    --hash=sha256:db19d07e2e0129e974a0e65d0064fc222a446cd5122c2fd4184d2af9fc734a9e \
    --hash=sha256:dca9ab98072a5a54ebacebdc45f53e645336b320c667410b061be1ca588ae709 \
    --hash=sha256:ddc7dacc8ece3a182e7f15cb862d1fd616b46d076cb1ae9dd232b2c38b655874 \
    --hash=sha256:ddf19c062bea7a0cc80f519243d2c01dd091be0cf952a0750d4ad576709559f5 \
    --hash=sha256:def79fa35ef0cef8d2accec024f4fdc7ead3012ff02f5215c783f39f03ef8cfc \
    --hash=sha256:df29a0a7107f7011e77f4eebdddec4c7331e24d787a0b21a46d63bdf7445da95 \
    --hash=sha256:e09a3942ecbdee5cce73ea9d42da82b81b72ac1bf031ce069b93b5adf4eac8cd \
    --hash=sha256:e242bb1c5e76e97dfa9e7f209a71e93a01d7f19ffdd5cfbb2e2d55b4f08f8ab0 \
    --hash=sha256:e243bd13217235fc7290c621941c3f5cc8b66e4872495be821d7436ba2fb838d \
    --hash=sha256:e2af3aad578aa6bd1384bcf4750fc285e5a9de53f40b7d41e5a0bf748edeb2b3 \
    --hash=sha256:e4e81e09c1578b8df602e3db08b0b3ea0a6947ad612f52bf8dc5ea8d47691f0c \
    --hash=sha256:e54da4baf05720032d527874d40b65fa4d7e5c6c6a43d0c3adbeffcaf275a2b3 \
    --hash=sha256:e80e6c2f55656b4824d72065abb4ddd6a525c74bd78a0aab5d9fc2cf4fb5af50 \
    --hash=sha256:ed2a239c0ea213acc1908150a3037257083c7c083128f1a4cec2ec4b97dca491 \
    --hash=sha256:ed905975ab14056a2e5eb1c376cb2e1ebc5396baf84163939c518556fccde9f5 \
    --hash=sha256:ee21e28f0430bd6dc9086c6e525d5e818a44a5ad19720c8a0ef766792f3eb5e5 \
    --hash=sha256:ee43c17b173d46a3212baa6ead3ae258eeabdae48c263a01ccf0218c366dd655 \
    --hash=sha256:ef4fcbf3327382cd4c9f540babd61248208af7b93eec4de397b4d5f58a09e288 \
    --hash=sha256:eff0ac9dbe711a4aee69bf04a83896aa9b85f19641264053a9f6d48573abb7dd \
    --hash=sha256:f0aa869112ef88429ae17820d99c3dd9504c9e9c671d3c246f3d7442cb051084 \
    --hash=sha256:f3c96f633825733f735c5a9cf21d21a257d8e1edf0b1cee0a064b9c424ca0f7d \
    --hash=sha256:f5833ad231be5eb6553de524a70f48d71b2c8563101750531e0b80184e175cd4 \
    --hash=sha256:f5ec61164adcec446f8969a3358ec3f9b26bbda3b9213e5586d219afa8df2915 \
    --hash=sha256:f7d486c83842422badd511868fd8a9a20e9407ace71564b6af47ce7e60a336c1 \
    --hash=sha256:fb9e68df06293761f9fe66ade60a9bc6d0f5e42b8acf2939a9158af86ab0e5bd \
    --hash=sha256:fc14a032f813bf5fe624d991960ea83e9715adc27e4c1830a2361eb1d02ac341 \
    --hash=sha256:fcff63213e8e6e47770541a4607175404f47cbb3ebea7b6058cc82d524a0e424 \
    --hash=sha256:fd1fbe0f116b6e55da77aca2c6ddcddcfac2186cbf78bdebf40fc156efca389d \
    --hash=sha256:fe9753dfee015c570d73df76f899f18444d41388bffcde097deba51c4fadbb9f
    # via requests
cuda-bindings==13.4.3 \
    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \
    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \
    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \
    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \
    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \
    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \
    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \
    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \
    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \
    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \
    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \
    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \
    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \
    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \
    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \
    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \
    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \
    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \
    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \
    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \
    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \
    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \
    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368
    # via torch
cuda-pathfinder==1.8.3 \
    --hash=sha256:e29e59829c297a7a5233bd9cc71094fc5bddbd076951482670178f9eade39b1f
    # via cuda-bindings
cuda-toolkit==13.0.3.0 \
    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f
    # via torch
filelock==4.0.9 \
    --hash=sha256:635e7d67fa92654eed444e75e9ca18426d34e77ad9c469bf4373f75a932f7b22 \
    --hash=sha256:9287fd61b99a806e5202be29a83034c0808a1b9830e820537c2f5773981f7eeb
    # via
    #   huggingface-hub
    #   torch
    #   transformers
fsspec==2026.9.0 \
    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \
    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f
    # via
    #   huggingface-hub
    #   torch
hf-xet==1.6.0 \
    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \
    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \
    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \
    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \
    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \
    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \
    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \
    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \
    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \
    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \
    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \
    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \
    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \
    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \
    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \
    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \
    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b
    # via huggingface-hub
huggingface-hub==0.36.2 \
    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \
    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270
    # via
    #   blip-captioning-pipeline (pyproject.toml)
    #   tokenizers
    #   transformers
idna==3.20 \
    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \
    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c
    # via requests
jinja2==3.1.6 \
    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \
    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67
    # via torch
markupsafe==3.0.4 \
    --hash=sha256:007e1ffd9bf65bb6ee96df7b258fc632a4868dd5566037986c64781f35a36e98 \
    --hash=sha256:02fa4acbc6a3fc5c693c34d4dd8c1130b7fe99cc915181b0ddd6f72aeb296002 \
    --hash=sha256:03470d1a8268e692ecf79ecd565593e59d44219377a7ead61f1f1b94c1f7ff6b \
    --hash=sha256:04e7902ba80ee4bac1d50a549606527a1dcf0476cd81403db41099d3b60ec653 \
    --hash=sha256:051417f74bcaaefa316276e0ff723f541616ca51043d070da00249d9bddd3e3c \
    --hash=sha256:05295589e619b9bed252a86b532b8e27350abc372d18ba89b59375325e91ec1e \
    --hash=sha256:06de8ef6331f6e822c28d577dc8bf43fe398800477c49498f38fc38b67ff33fc \
    --hash=sha256:0764a13d34cae40db7bbf3a09b7e9b491bf4603e20b263a7a9d6b8e324975d0a \
    --hash=sha256:077293e425f28ec737dbcad442a71752e28f8ae27cde3d68acd1fb212091cd92 \
    --hash=sha256:0930db9bdc62d22944e10b066448bb65dc9abe9112880c7cab8da54db4284d5f \
    --hash=sha256:0cee7cb0f9a1b6892ea482237d9403b3d1b4603aee057d0ff01f0fac2d019a97 \
    --hash=sha256:0d9c47709875fdb321452056622e930c52afbc07a7d780762fbb8b4d91ce6fa4 \
    --hash=sha256:11935df9bf455ed0c04eb87bcd720f02b1fe5e02128a9430f23aed6f93336fc7 \
    --hash=sha256:12a606a492de952afcb43b59a14aaaaad120e708d3663dd0fdf2d738d427a691 \
    --hash=sha256:14bd2d845d62ab678eaf81da89d7b621b51756c72346745c1a594c09d49207a2 \
    --hash=sha256:15ba9e28640feef770374b116a6f019c21f52404aeabe516aa7f800587b98cfc \
    --hash=sha256:18a801868a884f216e784d7d14db2a4077143ce7610440aee2ce8f734e7cfcde \
    --hash=sha256:1c0df495a977d10460a94941799c72d5b5ab03d3858d949b55b5a66c8f371c99 \
    --hash=sha256:1caa2fa5a6184fb233153b35f654e6687bd555476f6170f29d8ee9be1a8b0af9 \
    --hash=sha256:1e1451fab512d1bcc3dc26988ec1edb0b82c2db909132872cd9356070a6b63df \
    --hash=sha256:1f1f9477e174582b0a1b583d60b66e1f2cf5d3fe12cee985e4aedf44766600e5 \
    --hash=sha256:2628d3a8cb648ecebb3c5d6b0a1052d400e4d8b7ac0fb786be8d285b50040d17 \
    --hash=sha256:26e9867520db70d37f7fb421a7f0d8adb40171011fb84ce869afa1a83370dfa8 \
    --hash=sha256:2a6ef68ae94aed8721934072b27a3b654ea2100b97e4ab864cf1489c90926fbc \
    --hash=sha256:2b2b1e18af909b448bb3cf9e3433366f7a8726271fc214e8b10e0f62a78c724b \
    --hash=sha256:2cb3dd71fc6be918ad4264346a8ed69485f9b7ed7bf35495d8e22807cd6b8bea \
    --hash=sha256:2d1b7d9308288661f56672b1b157d75fc536714d3638487bbea17b6318a78248 \
    --hash=sha256:2dad610540cb2e6272855c178f08ae9a1c7ac258a7fb71660553a5f104b42741 \
    --hash=sha256:2e5a7cd7fdd14fcb1ae5d7d8bf23d24fbd1daefd1fbca2580132e1ea75f098b5 \
    --hash=sha256:2e9ad7dd851bf45fab9f75cbff4cb493fee9979e8d8c7c9c3ee119022518edd6 \
    --hash=sha256:340cbb1957ba99929cbf19a75626d36ba1ae21d1730b287d1cf7f824a20c4fc7 \
    --hash=sha256:34bdde374c5932765d7dc685c4a1d191a3207852d67e8e0a9eb6ea85156181f1 \
    --hash=sha256:353bd63081912ab8cfa6a0c7d185934cdf8426f04c618bba6bc4b394f2069b67 \
    --hash=sha256:387d8cd30e69b3f0a72877b9ae717033396404e19095b17fe89753a981fda44f \
    --hash=sha256:3882fb412298575bae3b9c46868251f15cc69307359f87bb1b382e53d6e5a2c9 \
    --hash=sha256:38fc55594dab834470b6733dead2ee9e3f657fb0608c769dcafa0ba5ab52f45c \
    --hash=sha256:396ec4e65cc889f69786b3b89478b471cee5a3bcf468b9d9bb03e1a30fb291fc \
    --hash=sha256:39dbacefc411633db5b4378b066a9aca70a3d7e2922c9e578d825f844026eeba \
    --hash=sha256:3a93d9616ddecfb393727a0041a562cf0b15a244e20f2bd25efc7949be4c4f17 \
    --hash=sha256:3d23795802fc8bd72534836d64489bbf0f67c088959091bdb22e10735a5107bf \
    --hash=sha256:434139499bb20b502ed3baa1f169e618f924a97e7a777fea1a49446d80106cf6 \
    --hash=sha256:436e3ffc6310d3c41878c601db29098102fe5d8a467c49da4a4125254e0980f2 \
    --hash=sha256:489505b03f692c3f376394e49194fa7a7f9e8558d6e293a7056a0032b0c38163 \
    --hash=sha256:4a540e2d3192792fc84eced57bef37851ccb2b41f73291bb17408eea77bcd278 \
    --hash=sha256:4a7cdc2a420ca01058182da4253329764d4bfa055564d1eced90e6ba1e8b1d3d \
    --hash=sha256:4bced6e2a6dba6a28f7dd3c6ce14df1b2dd495923f16ea484cad03decd463b2b \
    --hash=sha256:4cf3468d5ec187ffffcaca8e61929a37448f215dafc1386a12c750a72fe53634 \
    --hash=sha256:4e2c4809c14559aa7ef426f27fb35afbb38104c349a903bf8f3600456764bb38 \
    --hash=sha256:4ed644d75aa94a2baf7ec3a96eaa160ea58c742eb9d27c6506053c5c40fc84ed \
    --hash=sha256:4f6e0852a0283b1b1fd776eeb7b766a5f440b3e2bd31ab51af3b400585f3965c \
    --hash=sha256:5066b244f576f91afc8ee3ba029a89f99d39c79b1853fe9d39bea9f0afbec148 \
    --hash=sha256:5086f9975abb1ab531ee6afca1761e4b59a19b446f3f6522ed776963228cfe5a \
    --hash=sha256:50b5bedc9ed8a94fc8857a42ef4f84a81ea88f8d4f05dc8705fb23ee6d8dcca7 \
    --hash=sha256:52704c5d36eb6dda8866493decd61111fff86244c9b1ad225ca01b9e91e5970f \
    --hash=sha256:55ffd6ce583d97dc71dc92e930324c8c0d25aea7e3ade6ae54ef77cedb096811 \
    --hash=sha256:569d65055d367e3dcdf30c3f41119467b73d9ee9faf332bdf40402644f5ac08e \
    --hash=sha256:57f9947a7e57a081c1e3e0a2dd0d2dcf290a4531450e6f611e30084c222a7295 \
    --hash=sha256:5989cb26b2e1efc6a42216a9f6b5ee495ce5ace2e5b352a9af489976b32d1ee2 \
    --hash=sha256:5c22873ad1f0532ba40fa1727f3c0fc1bbbaab6d373d4cbe3f0dc74b2e2521c7 \
    --hash=sha256:5e8b3d0b18fd623afa12ecb2ce8d8becef69f9b5440c6330c7972200e0bb84b0 \
    --hash=sha256:61631e08084be9e21a8967ec3139c7616ed7c5e9368e05c86d1b39562c8a57b6 \
    --hash=sha256:64511c54db4e4987aef4c41923235927428729e8174c5dba488429be70a998ed \
    --hash=sha256:6669c1bf34080161ce49c589cc512ef24d4c704ac9d2b2d3667f519c60418378 \
    --hash=sha256:672d207103e6b16ca098611b0f9efad6bc00afd47c03d6ef62186495ca677dc0 \
    --hash=sha256:6768d67d1bce64270e0fdc2e69309d68b9b18ae56ddf6c711d168e9d051c2cac \
    --hash=sha256:6a45c3d514f2436064db00d7fc8778d888f0236ebfed649b53d13a59e69ad51b \
    --hash=sha256:6bd9e1788e15bfcf6a9082de42e30387e7b85d211ab21e57a939bb8cfaaf8d96 \
    --hash=sha256:6d2a9efe686f9de00d0d1ea32a4a5a86d558a2277501bd78d964214eab625e59 \
    --hash=sha256:6da83a088f8ef93b2d483a8232a4dbf4d69d3d8496b568a03c56becac43e1808 \
    --hash=sha256:7018d4af1cd272e847aa5917983ab5e83e4f6579f9dbfecd4a79c0ca80b144c2 \
    --hash=sha256:71f88e749ea29f67f21f3b36433c1dc54c7729ed2a6d9e2da2e0d9e0d7b224eb \
    --hash=sha256:737c9c3981998eba27f11786f84fddcbabc74068b72a4a1f454ea02094b57b65 \
    --hash=sha256:73e77980c7207854f00fc4e71fb1626868d5740ab4012623d55c7a99ad122a72 \
    --hash=sha256:799c39bdf5e2f1292fedd3009f7b3c9e760f10b2420cb9638d56920840ff6db8 \
    --hash=sha256:7a83aa6e4805df46fed18e989d3d16f86ef60cb50bbc8d9ce3a6be89165fbf6e \
    --hash=sha256:7d3391b2188d18737cb2fa147028b1096236eaa7e156446c650a489fa2cadc91 \
    --hash=sha256:7e1636da3d8dfc220b6dd10264db5f2b165e4888c4518594898fbe381049af8a \
    --hash=sha256:805c8b84534fa10891890f0e4be39f3a99e94615d93e8836bf9fa1fdca2feeb2 \
    --hash=sha256:811d02d5122171c1941357efd8f9bf4ffe907b7f0a1a4e729a880e4be3f46e3e \
    --hash=sha256:8138eb83940ec7299024d92d4dee45f601b9e6c5ffde9d25f4e35e326203c707 \
    --hash=sha256:83b3944fea42a8400edf92fd1770fb8d0d4f7de651353bd2d8525a92dba69a21 \
    --hash=sha256:849dd2bb0e5e4ab2b71c7191726a4a8d5aa8a610daa584728cbee0b710ddc4ef \
    --hash=sha256:8698d70a8081ee8c090dbb394768b5789a1da8b131b5499f89d071dd3cfaf6be \
    --hash=sha256:8781a792a070cf2bd1b86d3aa943894115faaba6e88122a7bf32d62072742453 \
    --hash=sha256:88d59b473bfb03259722600839af9bbd7fa13a2eb514beefeedb95997882f69a \
    --hash=sha256:8909c2f1c6dd65e054ac4b573a91c8384d1492281e55d82d159d653f7a13adf6 \
    --hash=sha256:8965520ac587c94a4ac48b729be3d8b8de00af39699b17585dfb599babe77977 \
    --hash=sha256:8b5d563170ff8ba3181caa967c99a3c804d1dedb702c7cb93a6a7c32247da978 \
    --hash=sha256:8e124f974786f831d6043728e38296969d3579db8896fe004682f5758e613581 \
    --hash=sha256:8f0fac8b13d14bb06c68195f849371924ae53dd7b1c00fed24650f704383b692 \
    --hash=sha256:9240187afb63d2f9ddc3e032c670356fe941f6e20662ea168a5dc3f1f317e1b3 \
    --hash=sha256:925f929d6b59a8b3f8b8c6ac363cd0af7eecc81efb3071770b3c6717c450a369 \
    --hash=sha256:9348cbb300d224fe3b89793262cb093504d4ae927004468463f745188a193e4a \
    --hash=sha256:9388003072b95f2f1e3fd908604194d653ba21330d811961a78b7da1a77e9e36 \
    --hash=sha256:9438a2648b2195980cb2dd8e53ed7b8df91319e2d0b70ae61a9e1d1bc8d3bec9 \
    --hash=sha256:94e4c421742086aeee4c32a506eec8859d7634aad943f7e6aacf70f813478768 \
    --hash=sha256:94f5407f7bc64fa6463906b896f9904beeeb7dd8dc116ee8e9056c8714ff9916 \
    --hash=sha256:971a3bbb75d97ae4e2e8f7d4834236f86f85f0c85e04ab2e191db1123b04f80b \
    --hash=sha256:9e227f3dbe6bde7491cf0a9965d00b88c6b1a4a95d11480ddf88bb96d397c19f \
    --hash=sha256:9e25feb9e330b63edb0278a0acdf85e50d0cb0fbf49c3084abbe4e24ae195346 \
    --hash=sha256:9f098115c247e11d138ab83a28fa0323c77015007ea2df73ba5fd714dfefd67c \
    --hash=sha256:a18f38cafc329bac5e3c2b96c765b4c96d3d103421ed22ab7988c1e3fce27464 \
    --hash=sha256:a4bbd2d87dd233b9fc5812160c3d0ffbe42edc22a26ce0469f58479ede633fe9 \
    --hash=sha256:a5fcffb37e602b0b3c1638a97746b9b96125caa9bcf6fa41d337a9261de231ee \
    --hash=sha256:a8e9f292fcda89b324f2f5c91d13f1424a153e40fc2756f38ee23b15835ff300 \
    --hash=sha256:a9f54054101545a9a9cccefddf54316aa6e4491611fcbef9e91b3b6bebec04f6 \
    --hash=sha256:aa2c838cc024642cc04c6854232f32b43e5e22833dd11119c1766c7873b8370d \
    --hash=sha256:ac0c7c9f1609b0c4c114feb1d7a3409564c7fb77e360bed9e97e5d25dfeaf868 \
    --hash=sha256:add96447a86d205ab616665d53b2950ee81083757f56e6ea833c8b2917646b46 \
    --hash=sha256:ae9dcb8fbe244cb82f8a6458b455b927a03685e383d9bacf1ea5ce180b96dc97 \
    --hash=sha256:b4a635a0487774f841cb1fb62e907e7195cc95bc761e053184b8acc3ceb20733 \
    --hash=sha256:b4d12837e0203bbace818ff4a7461afdcd78bcd782351cea148139180d7bcffe \
    --hash=sha256:b61687d0828e72bf5cda24a2690188f37170bd31c9359ac97e4e66569f120a16 \
    --hash=sha256:b807e598953730f82e4eae3bd30f6a122cf6b31c398c6b504c0e04c13c170429 \
    --hash=sha256:b8cd1f918b26fd7b1832ece557cc18f2d8747309ff8b3f0ef9d4250c5ad67a39 \
    --hash=sha256:b91cc9d336957239ff200f30097e6fea2dc6d6fb3c81e853eaa09eac904fd894 \
    --hash=sha256:bd3ce56ae2cbae3ba82b683bc425cd7e48d2ed8b10f3e818186b6f5646d9271c \
    --hash=sha256:be6cb0c799abb0e2ba3e618e6d28ddddf7e485f6c2ce938dfa237daf3905072c \
    --hash=sha256:befb4158af32106b9a93db8d6d1d1cbbd418c0d5aca0cabb7b1780abf0c89169 \
    --hash=sha256:bf053da3c97a4bc5ecfbb218cdd2983febd91c617be8367d139882aa11e490aa \
    --hash=sha256:c02e8f18bdedba082cef725942ac823b9b60656db07f7e265cb31618dfd00d77 \
    --hash=sha256:c1bc67752d5f21013cfe430df4062441714eab79f65a6a05e01505957e9c35fe \
    --hash=sha256:c61750fadcd119d0825bcb7d7d675dd264dcc89cc05292aab5be68ebdbb374ad \
    --hash=sha256:c90d5b3d4e944e065a301d741b3c1d784f6bd1f503aa68b4967e32b2ba313d85 \
    --hash=sha256:c9a7f43c0b202b334cc9184af09bb8f21d3a209e038efaf106936fb69e6b026e \
    --hash=sha256:cb96e6e088d6cf71c1ea977510948320234824cf226e32f6f6e044f7a9c82b34 \
    --hash=sha256:cf63c214fe879a65e69a386f915e36104fc84254ab141240f8854602d8e0be2a \
    --hash=sha256:d1aca03ede943eb80ab3d63bb082c84b7aab85ea83bd0fd0c200260945fb49d9 \
    --hash=sha256:d2e56fd3b00222722abfb3f5f0759ddbae4b90811b5ad4343c64030ad1bde70c \
    --hash=sha256:d5f93ebbeb8032d47e349328ec8662d973d9b05a70b3c35df1f91fe419b84749 \
    --hash=sha256:d882a373d8093c2941e01291b7ced96e9cbe4781da9a7751ca7e6c70385e5214 \
    --hash=sha256:d920abdfa61279ba1a2ef9484aab07bf03331f8c08a10120fa332353d06e6932 \
    --hash=sha256:da2af0d7aebfc2074080d72efa6ab8317c62481ef1f896f65d9999c1c01f4494 \
    --hash=sha256:dd8ea6ebee7aedbf7c749fa80521d9ccf1ba473e0d1e14805caafbaad281c889 \
    --hash=sha256:de8b364c423ef0a4bad9069657d617f9a5d2b2062457a89b1fa16ee199c399c1 \
    --hash=sha256:df1ae86ff54725a01fa1a0510b914ca53a161b7050be74f6204e24aded5971d0 \
    --hash=sha256:dff05cb7016dff1e9fd68f4122c127b65dfc59de5306cfb7ad92f956f230bee2 \
    --hash=sha256:e1a622f13970d81f95d0c72f9dc090dce9085fccfa4c9f2174377ee32bd15786 \
    --hash=sha256:e49fb0d1ce92cfa0cb198cc5b1b11cdf9d0638658e2a2db2687e39db7c87fc78 \
    --hash=sha256:e5c802729725bd07e2bc3ab7b76dc7e0bbfc53129d8f1eb1c002c24cf774717e \
    --hash=sha256:e841068dc0be4cb6dfb5c890eb88cbdcff2f4a332393c7ec94e8e618bd32c1a8 \
    --hash=sha256:e916035e3e9930cbdfdd10abf48861340221857f45509565898e012263f7b289 \
    --hash=sha256:eba154571c16e032112afac0dc2dfe9e63c2ceb7aedd07bb7eecf2ce26d4dd4c \
    --hash=sha256:f03460ff076f70ab595bb45a0205ccea1971443575b6920c52e755dec2b3fbfe \
    --hash=sha256:f0ec3b750b59375eab5b0fb2b9254810c00a3375be6d789899f1055a1d556237 \
    --hash=sha256:f291bcf42ae98eb5107edb162c3c998b4a89648fd8e99ed4cbd12705292788cd \
    --hash=sha256:f61efe1d2fe0de16158a5fe1d1cf3c14bdb6aecd54d8938fd26512c525c1f624 \
    --hash=sha256:f68edfc67aabac33708941f26f22a7b8e9f81429bc0cf249fcf7d66b23af8d19 \
    --hash=sha256:fa95848c929b6a75f6848d3c9793e59db365ee436776e57db835cdbfa79ba977 \
    --hash=sha256:fd9f8797427910198f95bced71ddfed61130d7e349213bfb8466c9c99e2c46a8 \
    --hash=sha256:fdb4ca07ab75ffadab4a8b135ad59cdbb3156b99310f3d565370da74a15d6bd3
    # via jinja2
mpmath==1.3.0 \
    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \
    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c
    # via sympy
networkx==3.7 \
    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \
    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a
    # via torch
numpy==2.5.3 \
    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \
    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \
    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \
    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \
    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \
    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \
    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \
    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \
    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \
    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \
    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \
    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \
    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \
    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \
    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \
    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \
    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \
    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \
    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \
    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \
    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \
    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \
    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \
    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \
    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \
    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \
    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \
    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \
    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \
    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \
    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \
    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \
    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \
    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \
    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \
    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \
    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \
    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \
    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \
    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \
    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \
    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \
    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \
    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \
    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \
    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \
    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \
    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \
    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \
    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \
    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \
    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \
    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \
    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \
    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \
    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \
    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \
    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \
    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \
    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \
    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \
    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \
    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \
    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \
    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \
    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab
    # via
    #   blip-captioning-pipeline (pyproject.toml)
    #   torchvision
    #   transformers
nvidia-cublas==13.1.1.3 \
    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \
    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \
    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5
    # via
    #   cuda-toolkit
    #   nvidia-cudnn-cu13
    #   nvidia-cusolver
nvidia-cuda-cupti==13.0.85 \
    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \
    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \
    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151
    # via cuda-toolkit
nvidia-cuda-nvrtc==13.0.88 \
    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \
    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \
    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b
    # via
    #   cuda-toolkit
    #   nvidia-cublas
nvidia-cuda-runtime==13.0.96 \
    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \
    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \
    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492
    # via cuda-toolkit
nvidia-cudnn-cu13==9.24.0.43 \
    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \
    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \
    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f
    # via torch
nvidia-cufft==12.0.0.61 \
    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \
    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \
    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3
    # via cuda-toolkit
nvidia-cufile==1.15.1.6 \
    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \
    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1
    # via cuda-toolkit
nvidia-curand==10.4.0.35 \
    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \
    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \
    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f
    # via cuda-toolkit
nvidia-cusolver==12.0.4.66 \
    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \
    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \
    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65
    # via cuda-toolkit
nvidia-cusparse==12.6.3.3 \
    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \
    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \
    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79
    # via
    #   cuda-toolkit
    #   nvidia-cusolver
nvidia-cusparselt-cu13==0.8.1 \
    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \
    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \
    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215
    # via torch
nvidia-nccl-cu13==2.30.7 \
    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \
    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50
    # via torch
nvidia-nvjitlink==13.4.92 \
    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \
    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \
    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \
    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323
    # via
    #   cuda-toolkit
    #   nvidia-cufft
    #   nvidia-cusolver
    #   nvidia-cusparse
nvidia-nvshmem-cu13==3.4.5 \
    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \
    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9
    # via torch
nvidia-nvtx==13.0.85 \
    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \
    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \
    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519
    # via cuda-toolkit
packaging==26.3 \
    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \
    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c
    # via
    #   huggingface-hub
    #   transformers
pillow==11.3.0 \
    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \
    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \
    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \
    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \
    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \
    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \
    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \
    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \
    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \
    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \
    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \
    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \
    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \
    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \
    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \
    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \
    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \
    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \
    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \
    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \
    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \
    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \
    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \
    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \
    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \
    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \
    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \
    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \
    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \
    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \
    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \
    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \
    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \
    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \
    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \
    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \
    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \
    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \
    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \
    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \
    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \
    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \
    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \
    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \
    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \
    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \
    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \
    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \
    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \
    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \
    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \
    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \
    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \
    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \
    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \
    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \
    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \
    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \
    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \
    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \
    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \
    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \
    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \
    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \
    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \
    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \
    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \
    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \
    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \
    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \
    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \
    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \
    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \
    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \
    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \
    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \
    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \
    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \
    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \
    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \
    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \
    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \
    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \
    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \
    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \
    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \
    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \
    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \
    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \
    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \
    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \
    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \
    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \
    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \
    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \
    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \
    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \
    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \
    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \
    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \
    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \
    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \
    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \
    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \
    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \
    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3
    # via
    #   blip-captioning-pipeline (pyproject.toml)
    #   torchvision
pyarrow==25.0.1 \
    --hash=sha256:0b1edbb2f385a6a65e9711b62ba86ac54a7816a3f8d17bb3e8a5929d65fb2485 \
    --hash=sha256:0b726ad7e7b669be982b0c71c07fe4b037d654354130da79a7902a669e93a66b \
    --hash=sha256:0befcf816e45a1af33ac775a9970b749e4868a230c7372f0ae5e932bee27039f \
    --hash=sha256:0fe7c8b6c03969b49c8c66182e4a18e3819ab92d07cfab5d8370c531b9369ef0 \
    --hash=sha256:119297a6dc197e45d9c6d4415f7814a67ffa36c180d26f68c154c58067ae782d \
    --hash=sha256:169d3429d5be7c752125890620f75a60776d38b0035eddae939651640822332e \
    --hash=sha256:25f8720bf6387d5dc2ebd2622112de630760419e4b66134405dd24110d15f37e \
    --hash=sha256:31e49a7888fcdf3a835da33ae777f6bb9a866334e5a789282fc26dcf426f7f15 \
    --hash=sha256:35935cd5de130aa5cf4dea052a63e6bf2e17006c35c3a468194242b9b2bf5956 \
    --hash=sha256:38a9a4b4b9613380e200641891495a56c3d5a98a092db4a870af9975e220471d \
    --hash=sha256:3f89685964f46e4216103c75483aac0c0692a5f72212d7ca835adba5ede56ce3 \
    --hash=sha256:4288f27577352d608ca08553b0865e4a9b3aa14820c5d95b53337218d609835b \
    --hash=sha256:4340f0ba6c1d2e13f21658de1d7c662ca2545018568d0030a1e9afca159d87e3 \
    --hash=sha256:44a9120ce5bd81936b8ab9a88076e3fd47c2c6838e0e43630fed83626aca81d9 \
    --hash=sha256:4facd65742a024a4a366328a1d2292062d72d6e023c1b7dda8d4c37544933a25 \
    --hash=sha256:51093dd9e10325fbdb3c10a2ae7c4806e5c822d94e74ae4938b26524a3323fee \
    --hash=sha256:514ddb60285631af068875550c90eddc181db3e8e63a032b1559be189e82f056 \
    --hash=sha256:5389cdf79447ed1515c9e31620e6e1e2302249564d603f2ad727d4f6d313e4c3 \
    --hash=sha256:59a2de54c0cbd954da861eee4d1d330f8e909c45b53455baef696380f2c55033 \
    --hash=sha256:60e89d8f13861a1f7f8d950fa54aebb8023b30734d0ac51ffa80beabe2df4bba \
    --hash=sha256:6109c94d8b9f3b17a041daca16cacb2f651ad8f1ef70a4232c2c0f37a23da2a8 \
    --hash=sha256:62cd0d785b8aa6675ee355f9fc02252a340f4441257c42674937826fd7594325 \
    --hash=sha256:6943e2fe7954d29d84de45d29d34c8dc36ce96570e67d89aa9976e650a4a9138 \
    --hash=sha256:6a1fdfc6659b6b19022f2e50627fb5cf7156a66c46bf4299379955cbe742382a \
    --hash=sha256:880523be3d29efcf83d3998835d206118ccf35e3871dbd2fb60408cf6b007a80 \
    --hash=sha256:8858d7bfc22e3f51529aeaa4077225029724623e4595dc9eff8c793935c34140 \
    --hash=sha256:9150a83248bfed9813ea3c3af74c3856c1984d444aa28e58bf7733b9750ddf6a \
    --hash=sha256:9171748cdf796972d85a4b60157c279913e242992e350c90c7450182a9838b2a \
    --hash=sha256:a4d6d5e9a3d1879a97c08ded0c797579b7965eafd0f0c26c30b45ccc06db939b \
    --hash=sha256:a4dd8bf99a8fac133efc0ed6a92f5fddbe2adba0d0f6dd720e39ba9855cea85c \
    --hash=sha256:aa0559502e1cd6254d6814614085dd9c5a3dd0419362978a936a3f68a9e5c3df \
    --hash=sha256:b7a296aac7a71fa0886c08e155ddb6c636a50013f801f6178daafa0f9e726188 \
    --hash=sha256:bddd0c4f7630c2a3ddf6347c1bdaa79d97bcf6bd445f9e60c816b7d77c85a5ae \
    --hash=sha256:bf0b672390cdcb640d7288f96b826d71ff4e9abb254a86c89890baf51a29cee6 \
    --hash=sha256:c7c534ec03c358a76ea3e505e74c1b6aef290af90c444dfd092dbfe23e755b85 \
    --hash=sha256:cab40b1edfef0262e0e5251aa2c58d75630f24d06dd7794480243acc001a1d7d \
    --hash=sha256:cc4aa407fde9fc660be3939e49ea31f50f3e9fec17c0ec63159f7711edd3efc9 \
    --hash=sha256:d51592cb7561e87877c506113e7adbf1342ab579e6c21f0ef44b8ba41cb74c80 \
    --hash=sha256:dda9470024204d7bbf2042b47c6e8a0e47a3eeb8e34405882dfaea6577e0c153 \
    --hash=sha256:df961f2e7ae9cf496459259d798652c70625f6c080650d6952f8c04053c58ee9 \
    --hash=sha256:eb6203482ff3746a5632303a7279ae0b5a304c46985b49ed1378cb350ea6728d \
    --hash=sha256:f3831aaa25c67a99f99dc8b05873cb9d64560390372e2aa197ce9dd4a3f06a44 \
    --hash=sha256:f729cfdbd36fd99d543b67a914d2de044c84ebe45be8b34902b299b608c15c8f
    # via blip-captioning-pipeline (pyproject.toml)
pyyaml==6.0.3 \
    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \
    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \
    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \
    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \
    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \
    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \
    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \
    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \
    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \
    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \
    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \
    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \
    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \
    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \
    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \
    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \
    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \
    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \
    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \
    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \
    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \
    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \
    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \
    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \
    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \
    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \
    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \
    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \
    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \
    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \
    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \
    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \
    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \
    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \
    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \
    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \
    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \
    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \
    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \
    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \
    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \
    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \
    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \
    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \
    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \
    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \
    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \
    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \
    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \
    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \
    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \
    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \
    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \
    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \
    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \
    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \
    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \
    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \
    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \
    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \
    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \
    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \
    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \
    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \
    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \
    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \
    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \
    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \
    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \
    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \
    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \
    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \
    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0
    # via
    #   huggingface-hub
    #   transformers
regex==2026.9.29 \
    --hash=sha256:01000ddf0e3ffef97f2413ceb514f6313040106b6d18a03ee00a4fe35c1eb1db \
    --hash=sha256:0166844493626c5015c6088ee15c9ca2fd060ca15b7641d1657da6a58432ae33 \
    --hash=sha256:044265d77d94f5e3cb2fd72c76723807c429cb8c533e9d4672d0334a6f14f588 \
    --hash=sha256:0476e5bcbe6e1ba3d1c4cc7bbb1c3ba78e3b979b5c8a88d0a6a8cdd4992b8c84 \
    --hash=sha256:066d0e3dbfdd739bce2bf8c2a41dd16f73e3d8adc2eb06dd803a36a307f56075 \
    --hash=sha256:0b65c72739f981377c9c22e0c5c3cd7f42da7bd8a3c9209330fac772c7d893ed \
    --hash=sha256:0c992c19cd45058a4b92f68f139c93db168b48fb1f322c9a7cd620806afb6b51 \
    --hash=sha256:0cc63b5e47c12a48d90c7e9d7de6a035dd14f62868aaedbb4e0ff8ba2b8bfe7b \
    --hash=sha256:0dd8af32e9f7b56b7f95cc1fd79b23054c3bdc172392ae560acc24d57b7ffe71 \
    --hash=sha256:0def9fb6abac55492d6d51cddb7225d07d6f279e774e0adc08569a54a5fc8d46 \
    --hash=sha256:0fd2c901cc307a745ad4bc87f20060d7a0825a3371d1e93488af22e7a387f78f \
    --hash=sha256:1043aedf5917caa861bcb25a9c11460049656bdf0017a90a309fa8f255467725 \
    --hash=sha256:121a76a0985db80ceae9e171c337f8c927868e37d01b54e3ce87bc87f9c6a208 \
    --hash=sha256:143533cc4b6fbc5b95aca0a5b8d541088d374831593def000ec89322c220221d \
    --hash=sha256:14e953ff3607c92d7675bf79c4d4509ef6782aa8c08509f179f9b3d6d0679e86 \
    --hash=sha256:18ae8eed4526e35bdb754d61562b90bf5c00a67fdcf3cc1380dd59597486631b \
    --hash=sha256:19959129885356df0e97556856f77eb2888380dac18bed075a7c05c5128c618d \
    --hash=sha256:1ba8c6a416569ce0d37e83e28a254a61dc99a419084dfb6476cea02d997f74fa \
    --hash=sha256:1c2a0026062abcc321a53db4a185ceba0b59a66b5d37b0808917a88b55a5257f \
    --hash=sha256:1d9fe8091b2e89d470df68a9331111ed008ae8aae6bf1e8e1fba4086a495c84e \
    --hash=sha256:2089fe39c406784d90101c726755ffa1497bb74638fd434300d2b88006186de8 \
    --hash=sha256:23ae6fdad9e63e54038f5ef78aba2933faca61e24d432786589e737bc5522ebb \
    --hash=sha256:26ec4ccce55aa533fbd603d08911b01101a8fcfec987845ac3ae2c7087b2bde3 \
    --hash=sha256:2f7f7aa47b229f2b39a2ae2596d2ad5625d77b5eb9856fac2dab3eb506cdd0a0 \
    --hash=sha256:31b003f9a070335e2a8233ee9b14a3ca8e6d792012ae011f741bf0aaf11744c5 \
    --hash=sha256:32ab11df9677ca80bcbb5fe4eb1da9109a5019239a054836efc6fa1c64e683cf \
    --hash=sha256:33026515aebc0e70d1c89978e53e8d695d35d9e472f8d5b34465ba3c74028650 \
    --hash=sha256:34b6925af9853bf461950e6508910f179fd6e9b1a7ec8548e069606b7e51a26b \
    --hash=sha256:352cf115a810b357caa35193ab656ecf5ef41056855e82f292c99e8514f8d954 \
    --hash=sha256:39ab5894d971f9ac68baa6eca5c50387db579cfcacf36ae8df3feceb1815e6d0 \
    --hash=sha256:3a21a9509d0ee88e7a70e1ad228cd2f0e0fd1e187458db132e8a8d18c97daf9d \
    --hash=sha256:3c5c2ef13797466aa64170cbb66ad98a32351dd4127694cea7199f80f213750d \
    --hash=sha256:3e778bfccd63075167709136afbc251c1f683758d5bf49c803c60ac3f894ce6b \
    --hash=sha256:3f1e6cb402a89457582cd696f982559217d13484a193202c394015297968c86d \
    --hash=sha256:42e82e578c904445d4c8a35b8f28052cf567593215fa5db06266fbc6f77aaa2e \
    --hash=sha256:4408b2b27a95ca8cc48b7411945753773353b5c93b307754781086c99d3a576f \
    --hash=sha256:446654b29bfaa30500d80947eda42cef1449dc8a87f4e3cf061cc8485d3a1f0b \
    --hash=sha256:45010bcfe66df41522d56c9b6114e87ecc597a08970ff6a2ced24415c141ae5f \
    --hash=sha256:49ee178ca31c94621294bf9b8b676a92a2e6bba8af0529591753719e57edb621 \
    --hash=sha256:4d7d93613b01b0199961330e49cfc52d479b3d5776c56c691db31130c0a07d91 \
    --hash=sha256:4fb41211d2333eb930a51e0546a65999761cf1f572a4da56ef9b8a62966c06f2 \
    --hash=sha256:4fe97894d1b306c919b4e50def1e6f6c522f4d03a7283811f4d108f1ce5d3ac2 \
    --hash=sha256:554bffadcbcb6d5f4e5fb10a61cc52084b9a63d1dab5f10bcd2c4343972e8e2c \
    --hash=sha256:59b49507f47479e299a9e1bc41b5cb83a7afda0540625f1dbae886615978acbf \
    --hash=sha256:5eeb8edc6110d9194a4d0d54610f64c37a31c605b5dbb7e407fc6ec7fa34a4a1 \
    --hash=sha256:612b709381c0355b70d89cdb51b7f670591ed5cbbc0e3b5337488019dc667b65 \
    --hash=sha256:61956f074ecd123f55adca68ee3eab46e6a07ad3f8e64e6db95dfacb444f55c4 \
    --hash=sha256:6398d5145689503412cc1748895242598d8846b8967b851133b20dc2ed1e21e8 \
    --hash=sha256:65b408d8fcb273e3499e7ef2ce796810da1becd208c7fb4373692a242d79d461 \
    --hash=sha256:686ac5350fceae63830bb98805fcb8039325bf4c06d9f6f048ff65229d5bffa5 \
    --hash=sha256:6a1a824fbed817e0a891103886b68f063b1e83cc51bc97192a90a60195a9291f \
    --hash=sha256:6abb75ab16bc3281714a5b99548a2225db70dba1f995f6d7f7419b76eb5a8fbe \
    --hash=sha256:6f7121a8914ed13fcfe2099f895341bfb789f004d4c5a0bdece8fa667da10849 \
    --hash=sha256:7020ed44df30b3aa492c00ee3b52d0548c1f30c2c6c5bb13ae897680900d3413 \
    --hash=sha256:720537c7ea6f80dc61913184edb0ce2497a306b39ef19f28505b322553d52bdb \
    --hash=sha256:724184b4aafed865e4f13ca313fdcb43024300c028ec67319cfa16847d84685e \
    --hash=sha256:7c03031610e3e6ed1768a2b7a8fc84637c1257b50c5eacaf094c6e17a84fc563 \
    --hash=sha256:80a5ea3b4fd9d6a5b9a44f7976a9acaaab35aa3c1f6b29e5bd857dfabaded223 \
    --hash=sha256:80c7cadd3fd2bfde5df8aa0787e315812cad0c313a753095d02f4c2b6c01677b \
    --hash=sha256:80ea96f5c1a30bf09007d48466521d9c294bebe197c708c3359096e3e3691632 \
    --hash=sha256:864e9b87ac33c3fb9fb4ad48166d4fdb579c351d5c77deb0d34bccb36a775cd9 \
    --hash=sha256:87fb80cbe3557e27e7b28b995c2b2eedf689b8886f941ab93e0e288f0976518a \
    --hash=sha256:8873c4a11c50b9989168881aeb3f08859f469d809941866aa1feefd8be5431f6 \
    --hash=sha256:888d60953908dcf761aa320c3e390ab8556efbdb551ace63921de90f6ae0848d \
    --hash=sha256:8b5fcc4771732191b2b7d1dd68d8f0353f47f8d90b6150f6dce58bf1112442cb \
    --hash=sha256:8f39588af4731c8923c26810eb3b33f76f17633985e40f59c3cd45a33805a895 \
    --hash=sha256:9173db3be74a35cb6731701094b98120f7ee4876a287882a59cdea1fa7da342f \
    --hash=sha256:92f05c9c42bde5785dc48770bc2194d9f7442544156f951e19cd31b096cec562 \
    --hash=sha256:951733b1bbdb71e377cec567b409f1a7881b47cfcad84121aa74cb575fa425ea \
    --hash=sha256:957bb708e8057ab1649ba566456429d691ec9b90d1c9ad1af1ba7ffbbeaf05f2 \
    --hash=sha256:9916fda742cd4eede63b286f58c06718324265d727ce0856eb1aac86d0d150d6 \
    --hash=sha256:9e1d3a4cb7993b708f0ada8d0c84590efd853f169e7147d2202c9da503180242 \
    --hash=sha256:9e4482589065c8ecd761cff522dcd85f2d39e62f551e37e025d1c7d54772def3 \
    --hash=sha256:a5300757f8a68f5b6cc33f57338d72a0e3589c5cc9ad5f8504ea06f028be582a \
    --hash=sha256:a540abfab208e1b7ef2df231c40ef3b6cbb30a0aad6204e9b6a81c10a6794628 \
    --hash=sha256:a5758353650079898dc1b2b0e95aa51fa23a30d020e06f62c430dd08ee56cdd8 \
    --hash=sha256:a64b85a4760337cfefdb27d42da6ed8b58e8cde3f2d57b6ef43e76ef6ea9ef47 \
    --hash=sha256:a655d34b2a6943af32401f3d94f72e9d731f6ad16285815550bf2b4ee69d420a \
    --hash=sha256:a714befaacbd10092ffe4cea0d3c5f008fb9efe9bc322c715bcdfdee414b9a3d \
    --hash=sha256:a760da040b47767b4b873adfb7c3b691e9ba2fc60f113f9d0b88f1a62f323e85 \
    --hash=sha256:addd736a0547d553283adaf4e05d7104e7f2c7b0b092e9b4d28756825f14531f \
    --hash=sha256:ae4613d7d9dda60fcba95f846cc6f808017f1843f392cf9daad14a6534493d71 \
    --hash=sha256:b11b589e00095ec69cf79841a76360f9b079e95b0368a25b5ebb951ab0c157ff \
    --hash=sha256:b3e445b66c80b4eb4234e855ce94d9adc183eedbd632816228d89930b91b2c5b \
    --hash=sha256:b7b893976e7fe42053da64f2aa27239c24252fd2ec6df471e1be197c0addc3b1 \
    --hash=sha256:b84f186a7f0536fe4ff9a9fa12d06d007b9b71d4b5352ddcc41f59ad6522a312 \
    --hash=sha256:b89efc38431793d28b7cd91227e2f952ad7c48df19132b17f43a5fec3c14143b \
    --hash=sha256:b97a38fb4c732b6832db6bf108963adbcd82ef1268ba2025dce390f45af75efa \
    --hash=sha256:b9d74e4eee9ddb64c2e92d5d61472c59c21684c059eb7b68767be9628e977859 \
    --hash=sha256:bb90e7177944b6684738c1fc36aabd2dd00d1de3be7dbe09f91e196f1bc0dc81 \
    --hash=sha256:bec37990e3d6121f29ecfb594bd8f1bf009e9f7926daba2e50e3b27d3892a783 \
    --hash=sha256:bf3c49863c23a1ad6da9c30351aed6cff8d5ddbeb63c5c8420ae54e98c7d0138 \
    --hash=sha256:bf48516e35cf848390ea68850aba53e7c333720d2945b4d2c25b69fc5171723f \
    --hash=sha256:bfc71e6d970419c1309b3640305298643e2a734cad3f7cfb6d2ddee4175ab53d \
    --hash=sha256:c0094897d7d01f184b2d7fe8c56c66d64efe01b31f4b7d34205b391387df1111 \
    --hash=sha256:c03c6eb6ece86dfdcbb34799efaa339b093132e1aceed491ba5e08fe06cdf699 \
    --hash=sha256:c1a9a6651197fbed6f0212591418b9def774fc3f8324f78d1bf0e6a63e5f8aa1 \
    --hash=sha256:c3589f40749acce747510bf5d589d54e376cb0930ea58b35effac97e5312b0c1 \
    --hash=sha256:c4e38dd8f39c43a91d2410ad2b85610701b0979342c3df1d69eaf8e838c757d8 \
    --hash=sha256:c6c8fabf1dafc1f1ddcbb67896d3f93efb092e8c4b6322d7389b944e76a484e5 \
    --hash=sha256:c90fcf7804ea0a54b896ce0f2b9565350220b8d4890fd0db461a476a4c687963 \
    --hash=sha256:c9b602fae1e00b7c035d661ce85575365719192a7b46784bd71cf64c68053aa0 \
    --hash=sha256:ccb64d887a9db1cd76dbc0f92051a1a478a2a67e7f56c62d915cb881d7734704 \
    --hash=sha256:d06fcdecc10fc7954d7c8f27a03c96055fe525274dc84a7b0dbdc3d6b9e03dab \
    --hash=sha256:d0c3082bf79bcd6a614d55916590ad4b8f93200e10b97f463ea5d9d07c9b5f23 \
    --hash=sha256:d49c18f1ea294cf4adde2e5ac256e98c82ea9d708462ce4bf799dffa7cfe8a2c \
    --hash=sha256:d60030baaa7bfbb02d650c126cdcddcb6e33dbff14d819434c8fa2fdcaeeeba5 \
    --hash=sha256:d7cab119d0df0b9413f106b4d7fc34f2872d3574ed3806fb48959c830b1537da \
    --hash=sha256:d9b77b25b4f395f92de6099ab08e8ae2bc7e51dfe157f22900902243a5cc90c7 \
    --hash=sha256:dabee8f4935e731fb46b2a3091bdda0d3d94b3bbfb907d2b4f12eefce4009619 \
    --hash=sha256:db5e82ba15c142425b8406690032df89e39cca4a2e8afbbb9a3d84edc2373ac3 \
    --hash=sha256:dc79d36d0618752265f0d575915bdc5c5130ecb9c9f6b3bcefeae32e4bdfafcf \
    --hash=sha256:ddfa987262763c3c22a8367d2a49c244b018a74c3a8e3ab1a864119ad45c5633 \
    --hash=sha256:e1172147d28d8fbcf8cb8d26c41506169f5ad8fe9ec969cb116835a19d4d8eca \
    --hash=sha256:e11edba5bc344a32b029a7af9d4b3173982dd79eeafa0b9dbd787364414b0509 \
    --hash=sha256:e2c89e9b762c57f59d5e99ee8b20202adb892e35f8d3485741340999ca55058e \
    --hash=sha256:e31f72490b7c12f7790e1e25c3afffd20503ee1bfb43461d7838b871ff244b19 \
    --hash=sha256:e8c65ef3862a8ad6e86492b6ed9327805dd66904c012bd3649dc67d822ed6c34 \
    --hash=sha256:ebb8912f565b8cdbbf27debfe00df04202c20e2f651b9e32767930c5eace3621 \
    --hash=sha256:ed511a0708e2297e1d6431e7fb217e3402791e491e02da800658ace4973df1bb \
    --hash=sha256:edf06545875f3efa31560d94121e95c7fd70d98b1dfedc0157097d79b13b52ea \
    --hash=sha256:f0fe9834e5aeccaf19a0d8feb296d66a24be1a7c9922002f842a682cd5abb787 \
    --hash=sha256:f1a0d5117230dd46b399a30a38afa44f79c99f3168988fdc4f425c3f928b39df \
    --hash=sha256:f37964e4a5e993d2fd45147741e9dff7f34a2d8c00ab94c4ea0514a4677f959e \
    --hash=sha256:f57dc6b8fef170f105d2cf5cdce254f47b137d7755086cf7050f47e16582abba \
    --hash=sha256:f93bc1c3486ef3747e07c9d7c1d0a147b8fbaab975f80e348aed6f71309dfaca \
    --hash=sha256:fb00027a09a8f9f08028b40dce4c933cf73e4833240ed356583fdc9cfa721566 \
    --hash=sha256:fb99cc9d45f48895d9d67f6a0b8a57f08d39c174d9f25ad97a313e0470267b1c \
    --hash=sha256:fdd88ed5e20b1bcdd234421e454962c971aa44b653bdb7f1ea9ef683e90fb649 \
    --hash=sha256:fe3fa1dd453ed5c7f5ea23a26218329790ed7197a99b90e94330e313959a7f52
    # via transformers
requests==2.34.2 \
    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \
    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed
    # via
    #   huggingface-hub
    #   transformers
safetensors==0.8.0 \
    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \
    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \
    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \
    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \
    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \
    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \
    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \
    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \
    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \
    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \
    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \
    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \
    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \
    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \
    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \
    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \
    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774
    # via
    #   blip-captioning-pipeline (pyproject.toml)
    #   transformers
setuptools==84.0.0 \
    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \
    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73
    # via torch
sympy==1.14.0 \
    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \
    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5
    # via torch
tokenizers==0.22.2 \
    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \
    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \
    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \
    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \
    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \
    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \
    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \
    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \
    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \
    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \
    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \
    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \
    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \
    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \
    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \
    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \
    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \
    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \
    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \
    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \
    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \
    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \
    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \
    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5
    # via transformers
torch==2.14.0 \
    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \
    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \
    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \
    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \
    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \
    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \
    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \
    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \
    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \
    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \
    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \
    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \
    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \
    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \
    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \
    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \
    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \
    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \
    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \
    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \
    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \
    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \
    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \
    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963
    # via
    #   blip-captioning-pipeline (pyproject.toml)
    #   torchvision
torchaudio==2.11.0 \
    --hash=sha256:00e9f71ab9c656f0abdb40c515bd65d4658ab0ad380dee27a2efd7d51dabd3d6 \
    --hash=sha256:13cff988697ccbad539987599f9dc672f40c417bed67570b365e4e5002bbd096 \
    --hash=sha256:1424638adb8bb40087bc7b6eb103e8e4fe398210f09076f33b7b5e61501b5d66 \
    --hash=sha256:1a07ec72fd6f26a588c39b5f029e0130d16bb40bc4221635580bf8fb18fcbc80 \
    --hash=sha256:1be3767064364ae82705bdf2b15c1e8b41fea82c4cd04d47428a8684b634b6ed \
    --hash=sha256:1c1101c1243ef0e4063ec63298977e2d3655c15cf88d9eb0a1bd4fe2db9f47ea \
    --hash=sha256:478110f981e5d40a8d82221732c57a56c85a1d5895fb8fe646e86ee15eded3bd \
    --hash=sha256:492dd64645e9d0bb843e94f1d9a4d1e31426262ffc594fafecc1697df9df5eb9 \
    --hash=sha256:5847fe2022b17c6580aeb39c8797a443411cc09edfd9183cd50ac1a3b8ccf97c \
    --hash=sha256:6503c0bdb29daf2e6281bb70ea2dfe2c3553b782b619eb5d73bdadd8a3f7cecf \
    --hash=sha256:67f6edac29ed004652c11db5c19d9debb5d835695930574f564efc8bdd061bba \
    --hash=sha256:6ebb59c694909eccb5d61b7cc199d297692012c43286e36d92983aa7bad7586d \
    --hash=sha256:73dab4841f94d888bc7c2aed7b5547c643edc974306919fe1adfb65d57cccf4b \
    --hash=sha256:79fb3cb99169fd41bd9719647261402a164da0d105a4d81f42a3260844ec5e79 \
    --hash=sha256:7e2da1df4f6fe885c46db350a0dc90a0dff4b54541dff8846faa904d255e2bfe \
    --hash=sha256:88fb5e29f670a33d9bac6aabb1d2734460cf6e461bde5cdc352826035851b16d \
    --hash=sha256:986f4df5ed17b003dc52489468601720090e65f964f8bebccf90eb45bba75744 \
    --hash=sha256:9fe3083c62e035646483a14e180d33561bdc2eed436c9ab1259c137fb7120b4a \
    --hash=sha256:a1cf1acc883bee9cb906a933572fed6a8a933f86ef34e9ea7d803f72317e8c1b \
    --hash=sha256:b034d7672f1c415434f48ef17807f2cce47f29e8795338c751d4e596c9fbe8b5 \
    --hash=sha256:bb59ba4452bbbe95d75ad3ef18df9824955625f36698ce9a5998a4a9f3c1ba1d \
    --hash=sha256:bc653defca1c16154398517a1adc98d0fb7f1dd08e58ced217558d213c2c6e29 \
    --hash=sha256:bda09ea630ae7207384fb0f28c35e4f8c0d82dd6eba020b6b335ad0caa9fed49 \
    --hash=sha256:be7ad472acb16d16e98c005f0219b0db06a47dfe8f7b4d177062e1638f871e3b \
    --hash=sha256:cc09cd1f6015b8549e7fe255fb1be5346b57e7fee06541d3f3dbb012d8c4715f \
    --hash=sha256:da2725e250866da42a12934c9a6552f65a18b7187fd7a6221387f0e605fb3b96 \
    --hash=sha256:e3f9696a9ef1d49acc452159b052370c636406d072e9d8f10895fda87b591ea9 \
    --hash=sha256:ed404c4399ad7f172c86a47c1b25293d322d1d58e26b10b0456a86cf67d37d84
    # via blip-captioning-pipeline (pyproject.toml)
torchvision==0.29.0 \
    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \
    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \
    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \
    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \
    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \
    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \
    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \
    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \
    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \
    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \
    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \
    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \
    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \
    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \
    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \
    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \
    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \
    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \
    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \
    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \
    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \
    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \
    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \
    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e
    # via blip-captioning-pipeline (pyproject.toml)
tqdm==4.70.1 \
    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \
    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4
    # via
    #   huggingface-hub
    #   transformers
transformers==4.57.6 \
    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \
    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3
    # via blip-captioning-pipeline (pyproject.toml)
triton==3.8.0 \
    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \
    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \
    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \
    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \
    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \
    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \
    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \
    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \
    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \
    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \
    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \
    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3
    # via torch
typing-extensions==4.16.0 \
    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \
    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5
    # via
    #   huggingface-hub
    #   torch
urllib3==2.8.0 \
    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \
    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63
    # via requests
'''

SKIP_INSTALL = os.environ.get("DIMER_NOTEBOOK_CI_PREINSTALLED") == "1"
ISOLATED_ENV = Path(os.environ.get("DIMER_ISOLATED_ENV", "dimer_isolated_env")).resolve()
ISOLATED_PYTHON = ISOLATED_ENV / "bin" / "python"
ISOLATED_TOOLS = ISOLATED_ENV.with_name(ISOLATED_ENV.name + "_tools")

if SKIP_INSTALL:
    print("DIMER_NOTEBOOK_CI_PREINSTALLED=1: the pins are already installed; the notebook runs in this kernel.")
else:
    if platform.system() != "Linux" or platform.machine() != "x86_64":
        raise RuntimeError("This notebook needs a Linux x86_64 runtime (Google Colab, Kaggle or Linux Jupyter): its locked environment is built for manylinux x86_64.")
    setup_started = time.perf_counter()
    if hashlib.sha256(LOCK_TEXT.encode("utf-8")).hexdigest() != LOCK_SHA256:
        raise RuntimeError("The carried lock does not match its digest: regenerate the notebook from the repository instead of editing this cell.")
    ISOLATED_TOOLS.mkdir(parents=True, exist_ok=True)
    lock_path = ISOLATED_TOOLS / LOCK_NAME
    lock_path.write_text(LOCK_TEXT, encoding="utf-8", newline="\n")
    for attempt in range(3):
        try:
            with urllib.request.urlopen(UV_URL, timeout=90) as response:
                wheel = response.read(UV_BYTES + 1)
            break
        except (urllib.error.URLError, TimeoutError, ConnectionError):
            if attempt == 2:
                raise
            time.sleep(2**attempt)
    if len(wheel) != UV_BYTES or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
        raise RuntimeError("The pinned uv wheel failed its size/SHA-256 check: refusing to run it. Run this cell again; if it repeats, the download is being altered.")
    with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
        member = next(name for name in archive.namelist() if name.endswith(".data/scripts/uv"))
        uv = ISOLATED_TOOLS / "uv"
        uv.write_bytes(archive.read(member))
    uv.chmod(0o700)
    # uv gets no kernel Python path; the managed interpreter is downloaded once and reused on a re-run.
    uv_env = dict(os.environ)
    for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP"):
        uv_env.pop(name, None)
    if not ISOLATED_PYTHON.is_file():
        subprocess.run([str(uv), "venv", "--quiet", "--managed-python", "--python", MANAGED_PYTHON, str(ISOLATED_ENV)], env=uv_env, check=True)
    isolated_version = subprocess.run([str(ISOLATED_PYTHON), "-c", "import platform; print(platform.python_version())"], env=uv_env, check=True, capture_output=True, text=True).stdout.strip()
    if isolated_version != MANAGED_PYTHON:
        raise RuntimeError(f"{ISOLATED_ENV} holds Python {isolated_version}, not {MANAGED_PYTHON}: delete that folder (or start a fresh runtime) and run this cell again.")
    subprocess.run([str(uv), "pip", "install", "--quiet", "--python", str(ISOLATED_PYTHON), "--require-hashes", "--only-binary", ":all:", "--index-url", "https://pypi.org/simple", "-r", str(lock_path)], env=uv_env, check=True)
    print({"isolated_environment": str(ISOLATED_ENV), "isolated_python": isolated_version, "kernel_python": platform.python_version(), "locked_packages": LOCKED_PACKAGES, "setup_seconds": round(time.perf_counter() - setup_started)})

{'isolated_environment': '/content/dimer_isolated_env', 'isolated_python': '3.12.12', 'kernel_python': '3.13.15', 'locked_packages': 48, 'setup_seconds': 52}


The next cell starts one Python process in that environment and routes **every later code cell** to it. Printed output comes back to the notebook as usual, variables persist from cell to cell, and an error stops **Run all** as it would in the kernel. These two cells are marked `# dimer: kernel cell` and are the only cells that run in the kernel. If you re-run a single cell later, it still runs in the isolated environment with the variables created so far; to start over, restart the session and choose **Run all**. `DIMER_NOTEBOOK_CI_PREINSTALLED=1` lets an executor that has already installed exactly these pins run every cell in its own kernel instead.

In [ ]:
# @title Infrastructure: route the remaining cells to the isolated environment
# dimer: kernel cell (runs in the notebook kernel, not in the isolated environment)
import signal
from multiprocessing.connection import Connection

# The worker runs in the isolated environment. It executes each routed cell in one persistent namespace and sends
# back printed text, displayed objects and matplotlib figures, so every cell behaves as it would in the kernel.
_WORKER_SOURCE = r"""
import ast, base64, builtins, io, linecache, os, signal, sys, traceback, types
from multiprocessing.connection import Connection

_send = Connection(int(sys.argv[1]), readable=False)
_recv = Connection(int(sys.argv[2]), writable=False)


class _Stream(io.TextIOBase):
    def __init__(self, name):
        self._name = name

    @property
    def encoding(self):
        return "utf-8"

    def writable(self):
        return True

    def isatty(self):
        return False

    def write(self, text):
        if text:
            _send.send(("stream", self._name, str(text)))
        return len(text)


sys.stdout, sys.stderr = _Stream("stdout"), _Stream("stderr")


def _figure_bundle(fig):
    buffer = io.BytesIO()
    fig.savefig(buffer, format="png", bbox_inches="tight")
    return {"image/png": base64.b64encode(buffer.getvalue()).decode("ascii"), "text/plain": repr(fig)}


def _flush_figures():
    plt = sys.modules.get("matplotlib.pyplot")
    if plt is None:
        return
    for number in plt.get_fignums():
        _send.send(("display", _figure_bundle(plt.figure(number))))
    plt.close("all")


def _mimebundle(obj):
    if hasattr(obj, "savefig"):
        return _figure_bundle(obj)
    data = {"text/plain": repr(obj)}
    for method, mime in (("_repr_html_", "text/html"), ("_repr_markdown_", "text/markdown"), ("_repr_png_", "image/png")):
        render = getattr(obj, method, None)
        if callable(render):
            try:
                value = render()
            except Exception:
                value = None
            if isinstance(value, bytes):
                value = base64.b64encode(value).decode("ascii")
            if value is not None:
                data[mime] = value
    return data


def display(*objects, **kwargs):
    for obj in objects:
        _send.send(("display", _mimebundle(obj)))


try:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot

    matplotlib.pyplot.show = lambda *args, **kwargs: _flush_figures()
except ImportError:
    pass

if os.environ.get("DIMER_KERNEL_IS_COLAB") == "1":
    # google.colab only exists in the kernel; forward the BYOD upload dialog to it.
    def _upload():
        _send.send(("upload",))
        reply = _recv.recv()
        if reply[1] is None:
            raise RuntimeError("The notebook kernel could not open the upload dialog.")
        return reply[1]

    import importlib.machinery

    def _stub(name, package):
        # A spec on every stub: importlib.util.find_spec("google.colab") (accelerate does this) raises on a None __spec__.
        module = types.ModuleType(name)
        module.__spec__ = importlib.machinery.ModuleSpec(name, None, is_package=package)
        if package:
            module.__path__ = []
        return module

    try:
        import google
    except ImportError:
        google = _stub("google", True)
        sys.modules["google"] = google
    _colab = _stub("google.colab", True)
    _files = _stub("google.colab.files", False)
    _files.upload = _upload
    _colab.files = _files
    google.colab = _colab
    sys.modules["google.colab"] = _colab
    sys.modules["google.colab.files"] = _files

_main = types.ModuleType("__main__")
_main.__dict__.update(__builtins__=builtins, display=display)
sys.modules["__main__"] = _main
_count = 0
while True:
    # An interrupt only lands inside a running cell; between cells it is ignored.
    signal.signal(signal.SIGINT, signal.SIG_IGN)
    try:
        message = _recv.recv()
    except EOFError:
        break
    if message[0] != "run":
        continue
    _count += 1
    filename = f"<isolated cell {_count}>"
    source = message[1]
    linecache.cache[filename] = (len(source), None, source.splitlines(True), filename)
    try:
        signal.signal(signal.SIGINT, signal.default_int_handler)
        tree = ast.parse(source, filename)
        tail = ast.Expression(tree.body.pop().value) if tree.body and isinstance(tree.body[-1], ast.Expr) else None
        exec(compile(tree, filename, "exec"), _main.__dict__)
        if tail is not None:
            value = eval(compile(tail, filename, "eval"), _main.__dict__)
            if value is not None:
                display(value)
        _flush_figures()
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        _send.send(("done",))
    except BaseException as exc:
        signal.signal(signal.SIGINT, signal.SIG_IGN)
        frames = exc.__traceback__.tb_next if exc.__traceback__ is not None else None
        _send.send(("error", "".join(traceback.format_exception(type(exc), exc, frames)), f"{type(exc).__name__}: {exc}"))
"""


class IsolatedCellError(RuntimeError):
    """A routed cell raised inside the isolated environment; its traceback is printed above."""


class IsolatedRuntime:
    """One persistent worker process in the isolated environment, fed one cell at a time."""

    def __init__(self, python, display=None):
        to_kernel_r, to_kernel_w = os.pipe()
        to_worker_r, to_worker_w = os.pipe()
        env = dict(os.environ, MPLBACKEND="Agg", PYTHONUNBUFFERED="1", DIMER_NOTEBOOK_CI_PREINSTALLED="1", HF_HUB_DISABLE_IMPLICIT_TOKEN="1")
        for name in ("PYTHONPATH", "PYTHONHOME", "PYTHONSTARTUP", "HF_TOKEN", "HUGGING_FACE_HUB_TOKEN"):
            env.pop(name, None)
        env["DIMER_KERNEL_IS_COLAB"] = "1" if "google.colab" in sys.modules else "0"
        self.proc = subprocess.Popen(
            [str(python), "-c", _WORKER_SOURCE, str(to_kernel_w), str(to_worker_r)],
            pass_fds=(to_kernel_w, to_worker_r),
            env=env,
            start_new_session=True,  # interrupts reach the worker only through run(), exactly once
        )
        os.close(to_kernel_w)
        os.close(to_worker_r)
        self._recv = Connection(to_kernel_r, writable=False)
        self._send = Connection(to_worker_w, readable=False)
        if display is None:
            from IPython.display import display
        self._display = display

    def _exited(self):
        return RuntimeError(
            f"The isolated environment's Python process exited (code {self.proc.wait()}); a crash of this kind is "
            "usually running out of memory. Restart the session and choose Run all again."
        )

    def run(self, source):
        try:
            self._send.send(("run", source))
        except OSError:
            raise self._exited() from None
        while True:
            try:
                message = self._recv.recv()
            except EOFError:
                raise self._exited() from None
            except KeyboardInterrupt:
                self.proc.send_signal(signal.SIGINT)
                continue
            kind = message[0]
            if kind == "stream":
                (sys.stdout if message[1] == "stdout" else sys.stderr).write(message[2])
            elif kind == "display":
                self._display(message[1], raw=True)
            elif kind == "upload":
                self._send.send(("upload", self._colab_upload()))
            elif kind == "error":
                sys.stderr.write(message[1])
                raise IsolatedCellError(message[2]) from None
            elif kind == "done":
                return

    @staticmethod
    def _colab_upload():
        try:
            from google.colab import files
        except ImportError:
            return None
        return files.upload()

    def close(self):
        self._send.close()
        self.proc.wait(timeout=30)


def _is_user_cell():
    # ipykernel transforms a cell before executing it; its caller knows whether this is a silent frontend request.
    frame = sys._getframe(2)
    while frame is not None:
        local = frame.f_locals
        if "silent" in local and "store_history" in local:
            return bool(local["store_history"]) and not bool(local["silent"])
        frame = frame.f_back
    return True


def _route_to_isolated_runtime(lines):
    source = "".join(lines)
    if not source.strip() or "# dimer: kernel cell" in source or not _is_user_cell():
        return lines
    return [f"_DIMER_ISOLATED_RUNTIME.run({source!r})\n"]


if SKIP_INSTALL:
    print("Routing disabled: the notebook runs in this kernel.")
else:
    from IPython import get_ipython as _kernel_shell

    _ip = _kernel_shell()
    _ip.input_transformers_cleanup[:] = [
        t for t in _ip.input_transformers_cleanup if getattr(t, "__name__", "") != "_route_to_isolated_runtime"
    ]
    if isinstance(globals().get("_DIMER_ISOLATED_RUNTIME"), IsolatedRuntime):
        _DIMER_ISOLATED_RUNTIME.close()
    _DIMER_ISOLATED_RUNTIME = IsolatedRuntime(ISOLATED_PYTHON)
    _ip.input_transformers_cleanup.append(_route_to_isolated_runtime)
    print(f"Every later code cell now runs in {ISOLATED_PYTHON} (pid {_DIMER_ISOLATED_RUNTIME.proc.pid}).")

Every later code cell now runs in /content/dimer_isolated_env/bin/python (pid 879).


### Record the runtime

This is the first cell that runs in the isolated environment. Nothing is installed here: the locked environment already holds the pins listed in the cell. It records the notebook's source revision and the versions actually imported. Look for a dictionary reporting the notebook's source revision, the isolated Python (3.12.12), `torch`, `transformers`, `PIL` versions, and whether CUDA is available. (The pip install and stale-import guard in the cell are skipped: the isolated worker, like an executor that has pre-installed exactly these pins, sets `DIMER_NOTEBOOK_CI_PREINSTALLED=1`.)

In [ ]:
# @title Infrastructure: record the runtime
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'torchaudio==2.11.0',
    'transformers==4.57.6',
    'huggingface-hub==0.36.2',
    'safetensors==0.8.0',
    'numpy==2.5.3',
    'pillow==11.3.0',
    'pyarrow==25.0.1',
]
NOTEBOOK_SOURCE = {
    'repository': 'florence2-vision-language-pipeline',
    'repository_revision': '0d15dffb95c87c956190e17a76981518c5c5146e',
    'embedded_module': 'src/florence2_vision_language_pipeline/pipeline.py',
    'embedded_modules': ['src/florence2_vision_language_pipeline/pipeline.py', 'src/florence2_vision_language_pipeline/metrics.py', 'src/florence2_vision_language_pipeline/samples.py'],
    'module_sha256': '52d02425351f82224c1471c19f4666beb432f511253b466d093e25ec7d05fc61',
    'generator': 'build_notebook.py/2.1',
    'notebook_spec': '2.2',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, transformers, PIL
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'PIL': PIL.__version__, 'cuda': torch.cuda.is_available()})

{'notebook_source': {'repository': 'florence2-vision-language-pipeline', 'repository_revision': '0d15dffb95c87c956190e17a76981518c5c5146e', 'embedded_module': 'src/florence2_vision_language_pipeline/pipeline.py', 'embedded_modules': ['src/florence2_vision_language_pipeline/pipeline.py', 'src/florence2_vision_language_pipeline/metrics.py', 'src/florence2_vision_language_pipeline/samples.py'], 'module_sha256': '52d02425351f82224c1471c19f4666beb432f511253b466d093e25ec7d05fc61', 'generator': 'build_notebook.py/2.1', 'notebook_spec': '2.2'}, 'python': '3.12.12', 'torch': '2.14.0+cu130', 'transformers': '4.57.6', 'PIL': '11.3.0', 'cuda': True}


## 2. Pipeline code (carried verbatim from `src/florence2_vision_language_pipeline/` @ `0d15dffb95c8`)

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/florence2_vision_language_pipeline/pipeline.py`

In [ ]:
# @title Infrastructure: carried module `src/florence2_vision_language_pipeline/pipeline.py` (you may run it without reading it)
"""Prompt-driven vision-language tasks with the pinned ``florence-community/Florence-2-large`` snapshot, plus the
adaptation contract for the ``<OCR>`` task: corpus-level evaluation on labelled text lines, bounded fine-tuning of the
last decoder layers on cached encoder outputs, and a verified adapter artifact.

The class loads weights only from a digest-verified local snapshot (``weights/<key>/``) or, when explicitly
allowed, from the Hugging Face Hub at the pinned revision, through the native ``transformers`` Florence-2
classes with ``trust_remote_code=False``. Pin history: the original ``microsoft/Florence-2-large`` pin was
rejected on 2026-09-12 because loading it requires executing custom code bundled in the model repository.
"""

# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

import hashlib
import json
import random
import time
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any

from PIL import Image

MODEL_ID = "florence-community/Florence-2-large"
MODEL_REVISION = "4271c66b88cdbc05735372ec13b2360108de5317"
MODEL_LICENSE = "mit"
MODEL_KEY = "florence-2-large-community"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"
WEIGHTS_FILE = "model.safetensors"
CONFIG_FILE = "config.json"

# Task prompts documented upstream for Florence-2; the last one needs a caption as text input.
TASKS_WITHOUT_TEXT = (
    "<CAPTION>",
    "<DETAILED_CAPTION>",
    "<MORE_DETAILED_CAPTION>",
    "<OD>",
    "<DENSE_REGION_CAPTION>",
    "<REGION_PROPOSAL>",
    "<OCR>",
    "<OCR_WITH_REGION>",
)
TASKS_WITH_TEXT = ("<CAPTION_TO_PHRASE_GROUNDING>",)
TASKS = TASKS_WITHOUT_TEXT + TASKS_WITH_TEXT

# The processor resizes every image to 768x768 regardless (preprocessor_config.json), so the side and pixel
# ceilings only guard memory while decoding and resizing (a 9,000 px wide text line is fine, a 4096x4096 page is
# the largest area accepted).
MAX_IMAGE_SIDE = 16_384
MAX_IMAGE_PIXELS = 4096 * 4096
MIN_IMAGE_SIDE = 1
MAX_TEXT_CHARS = 1000  # characters of caption text accepted for phrase grounding
MAX_NEW_TOKENS = 1024  # hard ceiling for `max_new_tokens` (upstream examples use 1024)
DEFAULT_MAX_NEW_TOKENS = 256
DEFAULT_LINE_MAX_NEW_TOKENS = 128  # the corpus stages' budget per text line
NUM_BEAMS = 3  # snapshot generation_config.json; decoding is deterministic beam search (do_sample=False)
OCR_PROMPT_TOKENS = 587  # 577 image tokens + the <OCR> prompt; identical for every image
# Model facts (measured on the pinned snapshot; tests pin them).
PARAMETER_COUNT = 776_505_344
DECODER_LAYERS = 12
TRAINABLE_LAYERS = 4  # the last decoder layers + the decoder's embedding layer norm are the adapter
ADAPTER_PARAMETERS = 67_188_736
# Adaptation contract.
ARTIFACT_FORMAT = f"org.valcorza.{MODEL_KEY}.adapter.v1"
ARTIFACT_VERSION = "1.0"
ADAPTER_WEIGHTS = "adapter.safetensors"
ADAPTER_MANIFEST = "manifest.json"
MIN_SCORED_RECORDS = 50  # below this a scored set is labelled a small sample
MAX_EVAL_RECORDS = 5_000
EVAL_BATCH_SIZE = 8
CACHE_BATCH_SIZE = 4  # images per frozen forward while caching encoder outputs
VISION_BATCH_SIZE = 4  # images per vision-tower forward (bounds its activation memory)
GRAD_CLIP = 1.0
_TRAINABLE_FIRST_LAYER = DECODER_LAYERS - TRAINABLE_LAYERS
_TRAINABLE_PREFIXES = tuple(f"model.language_model.decoder.layers.{i}." for i in range(_TRAINABLE_FIRST_LAYER, DECODER_LAYERS)) + ("model.language_model.decoder.layernorm_embedding.",)


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check a local snapshot against its manifest; raise naming the first mismatch."""
    root = Path(path or DEFAULT_WEIGHTS_DIR)
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"snapshot manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {MODEL_ID!r}")
    if manifest.get("revision") != MODEL_REVISION:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {MODEL_REVISION!r}")
    for entry in manifest.get("files", []):
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
    return {"path": str(root), **manifest}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but
    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; "
            f"pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _weight_digest(root: Path) -> str | None:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        return None
    with open(manifest_path, encoding="utf-8") as handle:
        entries = json.load(handle).get("files", [])
    return next((e["sha256"] for e in entries if e["path"] == WEIGHTS_FILE), None)


def normalise_text(text: str) -> str:
    """The transcript form the corpus measures use: whitespace runs collapsed to one space, ends stripped."""
    return " ".join(str(text).split())


def edit_distance(reference: Sequence[Any], hypothesis: Sequence[Any]) -> int:
    """Levenshtein distance (insertions + deletions + substitutions, unit cost) between two sequences."""
    previous = list(range(len(hypothesis) + 1))
    for row_index, ref_item in enumerate(reference, 1):
        current = [row_index]
        for column_index, hyp_item in enumerate(hypothesis, 1):
            current.append(min(current[-1] + 1, previous[column_index] + 1, previous[column_index - 1] + (ref_item != hyp_item)))
        previous = current
    return previous[-1]


def character_error_rate(reference: str, hypothesis: str) -> float:
    """Character-level Levenshtein distance over reference length; for OCR against a known transcript."""
    if not isinstance(reference, str) or not isinstance(hypothesis, str):
        raise TypeError("reference and hypothesis must be str")
    if not reference:
        return 0.0 if not hypothesis else 1.0
    return edit_distance(reference, hypothesis) / len(reference)


def validate_image(image: Any) -> Image.Image:
    """The image contract every task and every dataset record shares; returns the RGB image."""
    if not isinstance(image, Image.Image):
        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")
    width, height = image.size
    if min(width, height) < MIN_IMAGE_SIDE or max(width, height) > MAX_IMAGE_SIDE:
        raise ValueError(f"image side outside {MIN_IMAGE_SIDE}..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: {image.size}")
    if width * height > MAX_IMAGE_PIXELS:
        raise ValueError(f"image area {width * height} px > MAX_IMAGE_PIXELS {MAX_IMAGE_PIXELS}")
    return image.convert("RGB")


INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one PIL.Image.Image (any mode, converted to RGB) plus one task prompt from TASKS; the tasks "
        "in TASKS_WITH_TEXT additionally require a caption string as text_input"
    ),
    "image_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],
    "image_pixels_max": MAX_IMAGE_PIXELS,
    "tasks": list(TASKS),
    "tasks_requiring_text_input": list(TASKS_WITH_TEXT),
    "text_input_chars": [1, MAX_TEXT_CHARS],
    "max_new_tokens": [1, MAX_NEW_TOKENS],
    "num_beams": (
        f"positive int; the snapshot's generation_config default is NUM_BEAMS={NUM_BEAMS} and decoding "
        "is deterministic beam search (do_sample=False)"
    ),
    "preprocessing": (
        "image converted to RGB; the processor resizes it to exactly 768x768 (bicubic, ImageNet "
        "mean/std), so aspect ratio is not preserved and nothing is cropped; region outputs are mapped "
        "back to input pixel coordinates. The prompt sent to the model is the task token followed by "
        "text_input when the task takes one."
    ),
}

# Which capability families have an intrinsic metric in this repository, and what the others need.
_OCR_TASK = "<OCR>"
_NEEDS: dict[str, str] = {
    "caption": (
        "reference captions for the same images plus a caption metric (for example CIDEr or SPICE), or "
        "human adequacy ratings; this repository ships neither the references nor a caption metric"
    ),
    "region": (
        "annotated boxes for the same images and the caller's own matching/mean-average-precision code; "
        "Florence-2 emits no per-box score, so there is also nothing to calibrate or threshold"
    ),
    "ocr": (
        "a known transcript for the image, passed as the reference, so character_error_rate can be "
        "computed"
    ),
    "grounding": (
        "annotated boxes for the phrases in the supplied caption and the caller's own matching code; no "
        "grounding metric ships with this repository"
    ),
}
_TASK_FAMILY: dict[str, str] = {
    "<CAPTION>": "caption",
    "<DETAILED_CAPTION>": "caption",
    "<MORE_DETAILED_CAPTION>": "caption",
    "<OD>": "region",
    "<DENSE_REGION_CAPTION>": "region",
    "<REGION_PROPOSAL>": "region",
    "<OCR>": "ocr",
    "<OCR_WITH_REGION>": "region",
    "<CAPTION_TO_PHRASE_GROUNDING>": "grounding",
}
_SCORE_SEMANTICS = (
    "Florence-2 emits no probability or confidence: captions and OCR are plain generated text, and "
    "region tasks return boxes with labels and no per-box score, so there is nothing to threshold or "
    "calibrate. Decoding is deterministic beam search, not a likelihood estimate."
)


def _check_inputs(
    image: Any, task: Any, text_input: Any, max_new_tokens: Any, num_beams: Any
) -> Image.Image:
    """Raise TypeError/ValueError naming the first violated ceiling; return the RGB image.

    ``Florence2Pipeline.run`` and ``validate_inputs`` both route through this function so their
    acceptance criteria cannot diverge.
    """
    rgb = validate_image(image)
    if task not in TASKS:
        raise ValueError(f"task must be one of TASKS {TASKS}, got {task!r}")
    if task in TASKS_WITH_TEXT:
        if not isinstance(text_input, str) or not text_input.strip():
            raise ValueError(f"task {task} requires a non-empty text_input")
        if len(text_input) > MAX_TEXT_CHARS:
            raise ValueError(f"text_input exceeds MAX_TEXT_CHARS={MAX_TEXT_CHARS}: {len(text_input)}")
    elif text_input is not None:
        raise ValueError(f"task {task} takes no text_input")
    if isinstance(max_new_tokens, bool) or not isinstance(max_new_tokens, int):
        raise TypeError("max_new_tokens must be an int")
    if not 1 <= max_new_tokens <= MAX_NEW_TOKENS:
        raise ValueError(f"max_new_tokens must be between 1 and MAX_NEW_TOKENS={MAX_NEW_TOKENS}")
    if isinstance(num_beams, bool) or not isinstance(num_beams, int) or num_beams < 1:
        raise TypeError("num_beams must be a positive int")
    return rgb


def validate_inputs(
    image: Image.Image,
    task: str = "<CAPTION>",
    text_input: str | None = None,
    *,
    max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
    num_beams: int = NUM_BEAMS,
    names: Sequence[str] | None = None,
) -> dict[str, Any]:
    """Validation stage: return the input manifest (schema, observations, request, verdict).

    Rejection is reported by raising exactly as ``run`` would; a caller that wants the finding
    recorded catches the exception and stores ``str(exc)`` under ``findings``.
    """
    _check_inputs(image, task, text_input, max_new_tokens, num_beams)
    if names is not None and len(names) != 1:
        raise ValueError("names must have exactly one entry (run takes one image)")
    return {
        "schema": dict(INPUT_SCHEMA),
        "inputs": [
            {
                "id": names[0] if names else "image-0",
                "mode": image.mode,
                "size": list(image.size),
            }
        ],
        "task": task,
        "task_requires_text_input": task in TASKS_WITH_TEXT,
        "text_input": text_input,
        "generation": {"max_new_tokens": max_new_tokens, "num_beams": num_beams, "do_sample": False},
        "verdict": "accepted",
        "findings": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }


def evaluation_report(
    result: Mapping[str, Any] | Sequence[Mapping[str, Any]],
    reference_text: str | None = None,
    *,
    sample_kind: str = "synthetic",
) -> dict[str, Any]:
    """Evaluation stage: a machine-readable report even when nothing is measurable.

    ``result`` is one ``run`` result, or a sequence of them for a multi-capability run. The only
    intrinsic metric in this repository is ``character_error_rate``, and it applies to ``<OCR>``
    when a known transcript is supplied as ``reference_text``; every other capability is
    ``not-measurable`` and the report says what labelled data would make it measurable.
    """
    if not isinstance(result, Mapping):
        subreports = [
            evaluation_report(item, reference_text, sample_kind=sample_kind) for item in result
        ]
        metrics = [
            {**metric, "task": sub["task"]} for sub in subreports for metric in sub["metrics"]
        ]
        return {
            "task": "multi-capability: " + ", ".join(sub["task"] for sub in subreports),
            "score_semantics": _SCORE_SEMANTICS,
            "sample_kind": sample_kind,
            "n_capabilities": len(subreports),
            "metrics": metrics,
            "baselines": [],
            "capabilities": subreports,
            "verdict": "sample-sanity" if metrics else "not-measurable",
            "reason": (
                f"{len(metrics)} capability metric(s) over {len(subreports)} capabilities on one "
                "tutorial sample; sanity evidence, not a benchmark"
                if metrics
                else f"none of the {len(subreports)} demonstrated capabilities has an intrinsic metric here"
            ),
            "needs": "; ".join(dict.fromkeys(sub["needs"] for sub in subreports)),
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }
    task = result["task"]
    base = {
        "task": task,
        "score_semantics": _SCORE_SEMANTICS,
        "sample_kind": sample_kind,
        "n_outputs": 1,
        "generation": dict(result.get("generation") or {}),
        "baselines": [],
        "model_id": MODEL_ID,
        "model_revision": MODEL_REVISION,
    }
    family = _TASK_FAMILY.get(task, "region")
    if task == _OCR_TASK and isinstance(reference_text, str) and reference_text.strip():
        reference = reference_text.strip()
        hypothesis = str(result["result"]).strip()
        return {
            **base,
            "metrics": [
                {
                    "id": "character_error_rate",
                    "value": character_error_rate(reference, hypothesis),
                    "reference": reference,
                    "hypothesis": hypothesis,
                    "estimation": "one image against a known transcript, no dispersion estimate",
                }
            ],
            "verdict": "sample-sanity",
            "reason": (
                "one image scored against a transcript the caller already knows; on the synthetic "
                "sample that transcript is text the notebook drew itself, so this is a code-path "
                "check on a rendered font, not an OCR benchmark"
            ),
            "needs": (
                "a labelled OCR corpus from the deployment domain for any generalisable "
                "character-error-rate claim"
            ),
        }
    return {
        **base,
        "metrics": [],
        "verdict": "not-measurable",
        "reason": (
            f"no intrinsic metric exists in this repository for the {family} capability {task}"
            if task != _OCR_TASK
            else "no reference transcript was supplied for the evaluated image"
        ),
        "needs": _NEEDS[family],
    }


def _trainable_names(model: Any) -> list[str]:
    """The last `TRAINABLE_LAYERS` BART decoder layers and the decoder's embedding layer norm; the DaViT vision tower,
    the projector, the BART encoder, the shared embeddings (tied to the output head) and the earlier decoder layers
    stay frozen."""
    return [name for name, _ in model.named_parameters() if name.startswith(_TRAINABLE_PREFIXES)]


def _check_artifact_manifest(manifest: Mapping[str, Any], artifact_dir: Path, base_sha256: str) -> None:
    """Refuse an adapter that names another base, another format or a file that does not match its digest."""
    if manifest.get("format") != ARTIFACT_FORMAT:
        raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
    base = manifest.get("base", {})
    if base.get("model_id") != MODEL_ID or base.get("revision") != MODEL_REVISION:
        raise ValueError(f"artifact was trained on {base.get('model_id')}@{base.get('revision')}, not {MODEL_ID}@{MODEL_REVISION}")
    if base.get("weight_sha256") != base_sha256:
        raise ValueError("artifact base weight digest does not match the verified snapshot")
    files = manifest.get("files") or []
    if len(files) != 1 or files[0].get("path") != ADAPTER_WEIGHTS:
        raise ValueError(f"artifact manifest must list exactly {ADAPTER_WEIGHTS}")
    weights = artifact_dir / ADAPTER_WEIGHTS
    if not weights.is_file():
        raise FileNotFoundError(f"artifact weights missing: {weights}")
    size = weights.stat().st_size
    if size != files[0].get("bytes"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: size {size} != manifest {files[0].get('bytes')}")
    digest = _sha256(weights)
    if digest != files[0].get("sha256"):
        raise ValueError(f"{ADAPTER_WEIGHTS}: sha256 {digest} != manifest {files[0].get('sha256')}")
    names = manifest.get("tensors") or []
    if not names or any(not str(n).startswith(_TRAINABLE_PREFIXES) for n in names):
        raise ValueError(f"artifact tensors must all belong to the last {TRAINABLE_LAYERS} decoder layers or the decoder embedding norm")


@dataclass
class Florence2Pipeline:
    """``_runner(image, prompt, task, max_new_tokens, num_beams)`` -> ``{"text": raw, "parsed": value}``; the optional
    ``_batch_runner(images, max_new_tokens, num_beams)`` -> one ``{"text", "new_tokens"}`` per image for ``<OCR>``."""

    _runner: Callable[..., dict[str, Any]]
    device: str = "cpu"
    source: str = "injected"
    dtype: str = "float32"
    _batch_runner: Callable[..., list[dict[str, Any]]] | None = field(default=None, repr=False)
    _model: Any = field(default=None, repr=False)
    _processor: Any = field(default=None, repr=False)
    weight_sha256: str | None = None
    adapter: dict[str, Any] | None = None

    @classmethod
    def from_pretrained(
        cls,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Florence2Pipeline:
        root = Path(weights_dir or DEFAULT_WEIGHTS_DIR)
        common: dict[str, Any] = {"trust_remote_code": False}
        if (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            verify_snapshot(root)
            location, common["local_files_only"], source = str(root), True, "local-snapshot"
        elif allow_download:
            location, common["revision"], source = MODEL_ID, MODEL_REVISION, "hf-hub"
        else:
            raise FileNotFoundError(
                f"no verified snapshot at {root} and allow_download=False; "
                f"stage it with: hf download {MODEL_ID} --revision {MODEL_REVISION} --local-dir {root}"
            )
        # Refuse invalid snapshots before importing model libraries.
        import torch
        from transformers import Florence2ForConditionalGeneration, Florence2Processor

        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
        # float32 on every device: the adapter is trained in float32 and overlays without a cast, and CPU,
        # Tesla-class and consumer GPUs then run the same arithmetic.
        dtype = torch.float32
        processor = Florence2Processor.from_pretrained(location, **common)
        model, info = Florence2ForConditionalGeneration.from_pretrained(
            location, dtype=dtype, output_loading_info=True, **common
        )
        bad = {k: v for k, v in info.items() if v}
        if bad:
            raise RuntimeError(f"checkpoint does not match the native Florence-2 architecture: {bad}")
        model = model.eval().to(resolved_device)
        for param in model.parameters():
            param.requires_grad_(False)
        vision_features = model.model.get_image_features

        def chunked_image_features(pixel_values: Any, **kwargs: Any) -> Any:
            """Bound the vision tower's activation memory: `VISION_BATCH_SIZE` images per forward, results concatenated."""
            if pixel_values.shape[0] <= VISION_BATCH_SIZE:
                return vision_features(pixel_values, **kwargs)
            chunks = [vision_features(pixel_values[i : i + VISION_BATCH_SIZE], **kwargs) for i in range(0, pixel_values.shape[0], VISION_BATCH_SIZE)]
            return torch.cat(chunks, dim=0)

        model.model.get_image_features = chunked_image_features
        eos_id = model.config.text_config.eos_token_id
        pad_id = processor.tokenizer.pad_token_id

        def runner(image: Image.Image, prompt: str, task: str, max_new_tokens: int, num_beams: int) -> dict:
            inputs = processor(text=prompt, images=image, return_tensors="pt").to(resolved_device, dtype)
            with torch.inference_mode():
                generated = model.generate(
                    **inputs, max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False
                )
            text = processor.batch_decode(generated, skip_special_tokens=False)[0]
            parsed = processor.post_process_generation(text, task=task, image_size=image.size)
            return {"text": text, "parsed": parsed[task]}

        def batch_runner(images: Sequence[Image.Image], max_new_tokens: int, num_beams: int) -> list[dict[str, Any]]:
            inputs = processor(text=[_OCR_TASK] * len(images), images=list(images), return_tensors="pt", padding=True)
            if not bool(inputs["attention_mask"].all()):
                raise RuntimeError("prompts in one batch differ in length; batched generation needs identical prompts")
            inputs = inputs.to(resolved_device, dtype)
            with torch.inference_mode():
                generated = model.generate(**inputs, max_new_tokens=max_new_tokens, num_beams=num_beams, do_sample=False)
            texts = processor.batch_decode(generated, skip_special_tokens=True)
            out = []
            for row, text in zip(generated, texts, strict=True):
                ids = row.tolist()[2:]  # decoder start + <s>
                n_new = len(ids)
                for position, token in enumerate(ids):
                    if token in (eos_id, pad_id):
                        n_new = position + (token == eos_id)
                        break
                out.append({"text": text, "new_tokens": int(n_new)})
            return out

        return cls(runner, resolved_device, source, str(dtype).removeprefix("torch."), batch_runner, model, processor, _weight_digest(root))

    def _validate(
        self, image: Any, task: str, text_input: str | None, max_new_tokens: int, num_beams: int
    ) -> None:
        _check_inputs(image, task, text_input, max_new_tokens, num_beams)

    def run(
        self,
        image: Image.Image,
        task: str = "<CAPTION>",
        text_input: str | None = None,
        *,
        max_new_tokens: int = DEFAULT_MAX_NEW_TOKENS,
        num_beams: int = NUM_BEAMS,
    ) -> dict[str, Any]:
        """Run one task prompt on one image; ``result`` is the task-parsed value (text, or boxes + labels)."""
        self._validate(image, task, text_input, max_new_tokens, num_beams)
        prompt = task + (text_input or "")
        raw = self._runner(image.convert("RGB"), prompt, task, max_new_tokens, num_beams)
        if not isinstance(raw, dict) or "text" not in raw or "parsed" not in raw:
            raise RuntimeError("runner must return a dict with 'text' and 'parsed'")
        return {
            "task": task,
            "text_input": text_input,
            "result": raw["parsed"],
            "generated_text": str(raw["text"]),
            "image_size": list(image.size),
            "generation": {"max_new_tokens": max_new_tokens, "num_beams": num_beams, "do_sample": False},
            "device": self.device,
            "source": self.source,
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }


    # ------------------------------------------------------------------------------------------------------
    # Adaptation contract (the <OCR> task on transcribed text lines)
    # ------------------------------------------------------------------------------------------------------

    def transcribe(
        self,
        images: Sequence[Image.Image],
        *,
        max_new_tokens: int = DEFAULT_LINE_MAX_NEW_TOKENS,
        num_beams: int = NUM_BEAMS,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> list[dict[str, Any]]:
        """Run ``<OCR>`` on many images in batches (every image shares the same prompt, so no padding is involved);
        one ``{text, new_tokens, truncated}`` per image, in order. With an injected runner and no batch runner the
        images are recognised one by one through ``run``."""
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 64:
            raise ValueError("batch_size must be an int in 1..64")
        checked = [_check_inputs(image, _OCR_TASK, None, max_new_tokens, num_beams) for image in images]
        out: list[dict[str, Any]] = []
        for start in range(0, len(checked), batch_size):
            batch = checked[start : start + batch_size]
            if self._batch_runner is not None:
                raw = self._batch_runner(batch, max_new_tokens, num_beams)
            else:
                raw = []
                for image in batch:
                    single = self._runner(image, _OCR_TASK, _OCR_TASK, max_new_tokens, num_beams)
                    raw.append({"text": single["parsed"], "new_tokens": int(single.get("new_tokens", 0))})
            if not isinstance(raw, list) or len(raw) != len(batch) or any(not isinstance(r, dict) or "text" not in r for r in raw):
                raise RuntimeError("batch runner must return one dict with 'text' per image")
            for item in raw:
                new_tokens = int(item.get("new_tokens", 0))
                out.append({"text": normalise_text(str(item["text"])), "new_tokens": new_tokens, "truncated": new_tokens >= max_new_tokens})
            if progress is not None:
                progress(len(out), len(checked))
        return out

    def _require_model(self) -> tuple[Any, Any]:
        if self._model is None or self._processor is None:
            raise RuntimeError("this pipeline has no loaded model (injected runner); use from_pretrained for adapt/save_artifact/load_artifact")
        return self._model, self._processor

    def evaluate(
        self,
        records: Sequence[Mapping[str, Any]],
        *,
        max_new_tokens: int = DEFAULT_LINE_MAX_NEW_TOKENS,
        num_beams: int = NUM_BEAMS,
        batch_size: int = EVAL_BATCH_SIZE,
        progress: Callable[[int, int], None] | None = None,
    ) -> dict[str, Any]:
        """Transcribe every validated record with ``<OCR>`` and score the hypotheses with ``metrics.ocr_metrics``
        (micro and macro CER / WER, exact match). Works with an injected runner too."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import ocr_metrics` removed — names are kernel globals defined by the carried modules
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]
        started = time.perf_counter()
        items = self.transcribe([r["image"] for r in checked], max_new_tokens=max_new_tokens, num_beams=num_beams, batch_size=batch_size, progress=progress)
        hypotheses = [item["text"] for item in items]
        metrics = ocr_metrics(hypotheses, checked)
        metrics.update(
            {
                "hypotheses": hypotheses,
                "truncated": sum(item["truncated"] for item in items),
                "new_tokens": sum(item["new_tokens"] for item in items),
                "max_new_tokens": max_new_tokens,
                "num_beams": num_beams,
                "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",
                "adapted": self.adapter is not None,
                "seconds": round(time.perf_counter() - started, 3),
                "model_id": MODEL_ID,
                "model_revision": MODEL_REVISION,
            }
        )
        return metrics

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None,
        *,
        epochs: int = 6,
        lr: float = 5e-5,
        batch_size: int = 8,
        seed: int = 0,
        max_new_tokens: int = DEFAULT_LINE_MAX_NEW_TOKENS,
        num_beams: int = NUM_BEAMS,
        progress: Callable[[Mapping[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning of the last `TRAINABLE_LAYERS` BART decoder layers and the decoder's embedding layer norm
        on transcribed lines with the sequence-to-sequence loss over the transcript tokens (``<s>`` … transcript …
        ``</s>``, teacher-forced from the decoder start token) — the model's own objective. The frozen prefix — DaViT
        vision tower, projector and the BART encoder over the ``<OCR>`` prompt — is run once per line under no gradient
        and its encoder output cached, so each step runs only the decoder on those cached states; the loss equals the
        full model's loss exactly. AdamW (no weight decay), gradient clipping at `GRAD_CLIP`, seeded shuffling, no
        scheduler, no augmentation. Epoch 0 records the frozen model's validation metrics; the epoch with the lowest
        validation CER is kept (the final one without a validation split). On any exception the frozen weights are
        restored.

        An already adapted pipeline (or one with a loaded artifact) is refused: training would start from the adapted
        weights and record them as the "frozen model" at epoch 0 (review FL-m5). Build a fresh pipeline with
        `from_pretrained()` first."""
        model, processor = self._require_model()  # refuse before importing torch
        if self.adapter is not None:
            raise ValueError(
                "this pipeline is already adapted; adapt() starts from the pretrained base, so build a fresh "
                "pipeline with from_pretrained() first"
            )
        import torch
        from transformers.modeling_outputs import BaseModelOutput

        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-2:
            raise ValueError("lr must be in (0, 1e-2]")
        if isinstance(batch_size, bool) or not isinstance(batch_size, int) or not 1 <= batch_size <= 32:
            raise ValueError("batch_size must be an int in 1..32")
        train_checked = validate_dataset(train)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val is not None else None
        names = _trainable_names(model)
        name_set = set(names)
        device = torch.device(self.device)
        tokenizer = processor.tokenizer
        start_id = int(model.config.text_config.decoder_start_token_id)
        bos_id = int(tokenizer.bos_token_id)
        eos_id = int(tokenizer.eos_token_id)
        pad_id = int(tokenizer.pad_token_id)
        frozen_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
        previous_adapter = self.adapter
        cudnn_flags = (torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark)
        torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = True, False  # repeatable on one device
        history: list[dict[str, Any]] = []
        started = time.perf_counter()

        def _val() -> dict[str, Any] | None:
            if val_checked is None:
                return None
            result = self.evaluate(val_checked, max_new_tokens=max_new_tokens, num_beams=num_beams)
            return {"cer": result["cer"], "wer": result["wer"], "cer_macro": result["cer_macro"], "exact_match": result["exact_match"], "n": result["n"]}

        def _targets(batch: Sequence[Mapping[str, Any]]) -> tuple[Any, Any]:
            seqs = [[start_id, bos_id, *tokenizer(r["text"], add_special_tokens=False)["input_ids"], eos_id] for r in batch]
            length = max(len(seq) for seq in seqs) - 1
            decoder_input = torch.full((len(batch), length), pad_id, dtype=torch.long)
            labels = torch.full((len(batch), length), -100, dtype=torch.long)
            for i, seq in enumerate(seqs):
                decoder_input[i, : len(seq) - 1] = torch.tensor(seq[:-1])
                labels[i, : len(seq) - 1] = torch.tensor(seq[1:])
            return decoder_input, labels

        try:
            # 1. cache the frozen prefix: the encoder output for every training line (identical prompt lengths)
            cache: list[tuple[Any, Any, Any]] = []
            for start in range(0, len(train_checked), CACHE_BATCH_SIZE):
                batch = train_checked[start : start + CACHE_BATCH_SIZE]
                inputs = processor(text=[_OCR_TASK] * len(batch), images=[r["image"] for r in batch], return_tensors="pt", padding=True)
                if not bool(inputs["attention_mask"].all()):
                    raise RuntimeError("prompts in one batch differ in length")
                inputs = inputs.to(device)
                with torch.no_grad():
                    encoded = model.model(**inputs, decoder_input_ids=torch.full((len(batch), 1), start_id, dtype=torch.long, device=device)).encoder_last_hidden_state
                decoder_input, labels = _targets(batch)
                for k in range(len(batch)):
                    cache.append((encoded[k].detach().to("cpu"), decoder_input[k], labels[k]))
                del encoded
            cache_seconds = round(time.perf_counter() - started, 3)
            # 2. train the decoder tail on the cached encoder outputs
            params = []
            for name, param in model.named_parameters():
                if name in name_set:
                    param.requires_grad_(True)
                    params.append(param)
            n_trainable = sum(p.numel() for p in params)
            entry = {"epoch": 0, "train_loss": None, "val": _val(), "note": "frozen model"}
            history.append(entry)
            if progress is not None:
                progress(entry)
            best_epoch, best_score = 0, (history[0]["val"] or {}).get("cer", float("inf"))
            best_state = frozen_state
            optimizer = torch.optim.AdamW(params, lr=float(lr), weight_decay=0.0)
            rng = random.Random(seed)
            torch.manual_seed(seed)
            for epoch in range(1, epochs + 1):
                model.train()
                order = list(range(len(cache)))
                rng.shuffle(order)
                losses = []
                for start in range(0, len(order), batch_size):
                    items = [cache[k] for k in order[start : start + batch_size]]
                    encoded = torch.stack([h for h, _, _ in items]).to(device)
                    length = max(int(d.shape[0]) for _, d, _ in items)
                    decoder_input = torch.full((len(items), length), pad_id, dtype=torch.long)
                    labels = torch.full((len(items), length), -100, dtype=torch.long)
                    for k, (_, d, lab) in enumerate(items):
                        decoder_input[k, : d.shape[0]] = d
                        labels[k, : lab.shape[0]] = lab
                    attention = torch.ones((len(items), encoded.shape[1]), dtype=torch.long, device=device)
                    loss = model(encoder_outputs=BaseModelOutput(last_hidden_state=encoded), attention_mask=attention, decoder_input_ids=decoder_input.to(device), labels=labels.to(device)).loss
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
                    optimizer.step()
                    losses.append(float(loss.detach()))
                model.eval()
                entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val": _val()}
                history.append(entry)
                if progress is not None:
                    progress(entry)
                if val_checked is None or entry["val"]["cer"] < best_score:
                    best_epoch, best_score = epoch, (entry["val"] or {}).get("cer", float("inf"))
                    best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in name_set}
            model.load_state_dict(best_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
        except BaseException:
            model.load_state_dict(frozen_state, strict=False)
            for param in model.parameters():
                param.requires_grad_(False)
            model.eval()
            self.adapter = previous_adapter
            raise
        finally:
            torch.backends.cudnn.deterministic, torch.backends.cudnn.benchmark = cudnn_flags
        self.adapter = {
            "task": _OCR_TASK,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "first_trainable_layer": _TRAINABLE_FIRST_LAYER,
            "epochs": epochs,
            "batch_size": batch_size,
            "best_epoch": best_epoch,
            "selection": "lowest validation CER" if val_checked is not None else "final epoch (no validation split)",
            "loss": "sequence-to-sequence cross-entropy over the transcript tokens and the end token, teacher-forced from the decoder start token; computed on the cached frozen encoder outputs",
            "lr": float(lr),
            "seed": seed,
            "max_new_tokens": max_new_tokens,
            "num_beams": num_beams,
            "n_train": len(train_checked),
            "n_val": len(val_checked) if val_checked is not None else 0,
            "cache_seconds": cache_seconds,
            "history": history,
            "seconds": round(time.perf_counter() - started, 3),
        }
        return dict(self.adapter)

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the trained tensors as safetensors plus a manifest naming the base, the digests and the training
        configuration. Requires a prior `adapt`."""
        model, _processor = self._require_model()  # refuse before importing torch
        import torch
        from safetensors.torch import save_file

        if self.adapter is None:
            raise RuntimeError("nothing to save: call adapt() first")
        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = list(self.adapter["trainable_names"])
        state = model.state_dict()
        tensors = {name: state[name].detach().cpu().contiguous() for name in names}
        weights = out / ADAPTER_WEIGHTS
        save_file(tensors, str(weights), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "version": ARTIFACT_VERSION,
            "base": {"model_id": MODEL_ID, "revision": MODEL_REVISION, "weight_file": WEIGHTS_FILE, "weight_sha256": self.weight_sha256},
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": names,
            "files": [{"path": ADAPTER_WEIGHTS, "bytes": weights.stat().st_size, "sha256": _sha256(weights)}],
            "torch": torch.__version__,
            "metadata": dict(metadata or {}),
        }
        with open(out / ADAPTER_MANIFEST, "w", encoding="utf-8") as handle:
            json.dump(manifest, handle, indent=2, ensure_ascii=False)
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Overlay a saved adapter onto this (freshly loaded) pipeline after checking its manifest, digest and exact
        tensor set. Refuses tensors outside the last decoder layers and the decoder embedding norm."""
        model, _processor = self._require_model()  # refuse before importing safetensors
        from safetensors.torch import load_file

        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        _check_artifact_manifest(manifest, artifact, self.weight_sha256 or "")
        expected = _trainable_names(model)
        if sorted(manifest["tensors"]) != sorted(expected):
            raise ValueError("artifact tensor set does not match its recorded configuration")
        tensors = load_file(str(artifact / ADAPTER_WEIGHTS))
        if sorted(tensors) != sorted(expected):
            raise ValueError("artifact tensor names differ from the manifest")
        state = model.state_dict()
        for name, tensor in tensors.items():
            if tuple(tensor.shape) != tuple(state[name].shape):
                raise ValueError(f"artifact tensor {name} has shape {tuple(tensor.shape)}, base has {tuple(state[name].shape)}")
        model.load_state_dict({k: v.to(state[k].device, state[k].dtype) for k, v in tensors.items()}, strict=False)
        model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": expected, "history": manifest.get("history", [])}
        return dict(self.adapter)

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str | None = None,
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
    ) -> Florence2Pipeline:
        """Check the adapter manifest against the base snapshot's recorded weight digest, load the verified base, then
        overlay the adapter (checked again, and the tensor set, before deserialising). A refused manifest never loads
        a model."""
        artifact = Path(artifact_dir)
        manifest_path = artifact / ADAPTER_MANIFEST
        if not manifest_path.is_file():
            raise FileNotFoundError(f"artifact manifest missing: {manifest_path}")
        with open(manifest_path, encoding="utf-8") as handle:
            manifest = json.load(handle)
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        _check_artifact_manifest(manifest, artifact, _weight_digest(root) or "")
        pipe = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)
        pipe.load_artifact(artifact_dir)
        return pipe

**Module 2/3:** `src/florence2_vision_language_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
# @title Infrastructure: carried module `src/florence2_vision_language_pipeline/metrics.py` (you may run it without reading it)
"""Corpus-level text-recognition measures and two non-neural baselines, in plain Python.

``ocr_metrics`` scores one hypothesis string per record against ``record['text']``: the character error rate and the
word error rate as **micro** averages (total Levenshtein edits over total reference characters or words — the
corpus CER/WER of the handwriting-recognition literature) and as **macro** averages (mean per-line rate), the exact-
match rate, and the counts behind them. Neither rate is capped: a hypothesis longer than its reference can push a
rate above 1.0, which is the signal that the model is generating text the image does not carry. The baselines
answer without looking at the image — the empty string, or one constant training transcript — and are scored by
the same function.
"""
# ruff: noqa: E501  -- adaptation-contract lines are kept at the fleet width

from __future__ import annotations

from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import edit_distance, normalise_text` removed — names are kernel globals defined by the carried modules

METRIC_DEFINITIONS = {
    "cer": "character error rate, micro: total character edits (insertions + deletions + substitutions) over total reference characters; 0 is perfect, values above 1 mean over-generation",
    "wer": "word error rate, micro: total word edits over total reference words after lower-casing and whitespace tokenisation; punctuation kept",
    "cer_macro": "mean of the per-line character error rates (each line weighted equally regardless of length)",
    "wer_macro": "mean of the per-line word error rates",
    "exact_match": "fraction of lines whose hypothesis equals the reference after whitespace normalisation",
}
MEDOID_POOL = 120  # training transcripts considered when choosing the constant baseline (quadratic cost)


def _words(text: str) -> list[str]:
    return text.lower().split()


def ocr_metrics(hypotheses: Sequence[str], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Score one hypothesis per record against `record['text']`; per-line rows plus the corpus rates. Raises when
    the lengths differ or nothing is scored."""
    if len(hypotheses) != len(records) or not records:
        raise ValueError("hypotheses and records must be non-empty and the same length")
    rows = []
    char_edits = word_edits = ref_chars = ref_words = hyp_chars = exact = 0
    for hypothesis, record in zip(hypotheses, records, strict=True):
        reference = normalise_text(record["text"])
        hyp = normalise_text(str(hypothesis))
        if not reference:
            raise ValueError(f"record {record.get('id')!r} has an empty reference")
        ce = edit_distance(reference, hyp)
        we = edit_distance(_words(reference), _words(hyp))
        n_words = len(_words(reference))
        rows.append({"id": record.get("id"), "ref_chars": len(reference), "hyp_chars": len(hyp), "char_edits": ce, "cer": ce / len(reference), "word_edits": we, "wer": we / n_words, "exact": hyp == reference})
        char_edits += ce
        word_edits += we
        ref_chars += len(reference)
        ref_words += n_words
        hyp_chars += len(hyp)
        exact += hyp == reference
    n = len(rows)
    return {
        "n": n,
        "cer": char_edits / ref_chars,
        "wer": word_edits / ref_words,
        "cer_macro": sum(r["cer"] for r in rows) / n,
        "wer_macro": sum(r["wer"] for r in rows) / n,
        "exact_match": exact / n,
        "char_edits": char_edits,
        "ref_chars": ref_chars,
        "hyp_chars": hyp_chars,
        "word_edits": word_edits,
        "ref_words": ref_words,
        "rows": rows,
        "definitions": dict(METRIC_DEFINITIONS),
    }


def empty_baseline(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Predict the empty string for every line: CER and WER are exactly 1.0 by construction (every reference
    character is a deletion). The floor any recogniser must beat to be doing better than silence."""
    result = ocr_metrics([""] * len(records), records)
    result["baseline"] = "empty string for every line"
    return result


def medoid_transcript(train: Sequence[Mapping[str, Any]], *, pool: int = MEDOID_POOL) -> str:
    """The training transcript (among the first `pool`) with the smallest summed character error rate to the others:
    the single constant string that best matches the corpus on average."""
    texts = [normalise_text(r["text"]) for r in train[:pool]]
    texts = [t for t in texts if t]
    if not texts:
        raise ValueError("no non-empty training transcripts")
    return min(texts, key=lambda candidate: sum(edit_distance(other, candidate) / len(other) for other in texts))


def constant_baseline(train: Sequence[Mapping[str, Any]], records: Sequence[Mapping[str, Any]], *, pool: int = MEDOID_POOL) -> dict[str, Any]:
    """Predict one constant training transcript (the medoid) for every line: what corpus statistics alone buy
    without reading the image — usually a CER near 1.0, since edits are dominated by substitutions."""
    text = medoid_transcript(train, pool=pool)
    result = ocr_metrics([text] * len(records), records)
    result["baseline"] = f"constant training transcript {text!r}"
    result["transcript"] = text
    return result

**Module 3/3:** `src/florence2_vision_language_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
# @title Infrastructure: carried module `src/florence2_vision_language_pipeline/samples.py` (you may run it without reading it)
"""Labelled text-line datasets for the adaptation contract: the digest-pinned Belfort sample, the record contract
and its structural validation, image-disjoint splitting, and the BYOD loader.

A record is ``{id, image, text}`` where ``image`` is a PIL image of one text line (or any region whose transcript is
known; sides within the pipeline's ceilings) and ``text`` its transcript — the string the model is asked to
generate for that image. Whitespace runs in ``text`` are collapsed; case and punctuation are kept.

The default sample is drawn from the Belfort-line dataset (Teklia; the minutes of the Belfort municipal council,
19th–20th century French handwriting transcribed by a crowdsourcing campaign; Tarride et al. 2023, **MIT**) as
converted to parquet by the Hugging Face Hub at an immutable revision: the first ``CORPUS_ROW_GROUPS`` row groups of
the test shard are read with HTTPS range requests (about 5.5 MB each; the shard's declared size is checked first and
every row group's decoded content is refused unless its SHA-256 matches the pin). Every line image is 128 px tall;
widths run from about 145 to 9,000 px. The domain gap to the model's printed-text training distribution is the point
of the sample: the frozen model reads almost none of it.
"""
# ruff: noqa: E501  -- record and pin literals are kept on single lines

from __future__ import annotations

import csv
import hashlib
import io
import random
import re
import urllib.request
import zipfile
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

from PIL import Image

# standalone rewrite (build_notebook.py): `from .pipeline import MODEL_ID, normalise_text, validate_image` removed — names are kernel globals defined by the carried modules

CORPUS_NAME = "Belfort-line (test split), first eight parquet row groups"
CORPUS_REPO = "Teklia/Belfort-line"
CORPUS_REVISION = "c4a74bbd39f2df314752e7e6026649a39d365cbb"  # refs/convert/parquet commit on the Hub
CORPUS_FILE = "default/test/0000.parquet"
CORPUS_BYTES = 210_579_166
CORPUS_ROWS = 3_819
CORPUS_ROW_GROUPS = 8  # of 39; 100 lines each
CORPUS_LICENSE = "MIT (Teklia; Belfort municipal council minutes, Zenodo record 8041668; Tarride et al. 2023, https://doi.org/10.1145/3604951.3605517)"
CORPUS_LANGUAGE = "fr"
CORPUS_URL = f"https://huggingface.co/datasets/{CORPUS_REPO}/resolve/{CORPUS_REVISION}/{CORPUS_FILE}"
# SHA-256 over the concatenated image bytes + UTF-8 transcript of each row group, in row order, and that byte total.
ROW_GROUP_PINS: dict[int, tuple[str, int]] = {
    0: ("1dc3141e4809ea628b17c3ca7b81d64e6ca92bce18dd5765ecd618bfc7867954", 5_481_145),
    1: ("c9d3b52013933c803f4886edbce68da0ae483347a4a6ff3e0f5ced1db4a7e653", 5_465_901),
    2: ("6e0578a90a07a9e25e65b765881d3fa33d6a797624425e01026980d7287f0bf6", 5_379_166),
    3: ("00cdfb7aabe924f31b9f1bb1ba4849040051e5619567b68bf99fdcbcab15131a", 5_821_303),
    4: ("fc063442fb20e7a60c2533ab44dcc69a22ad59f5ce24921fe6af5f53ceab7e1a", 5_163_559),
    5: ("2d7e29331bd4e93e0c8a1caa9a83b8f1ede9b17af6dae9377b83f56c56f05689", 4_713_140),
    6: ("49423d91780cb184c4b0069630e85acccb314a113256ced9692a5138c4782ef1", 5_069_794),
    7: ("3a010831456f185399579b4ecf9d46222f95368c3cdbbc2ff103a258b9c16e2f", 5_309_891),
}
DEFAULT_CACHE_DIR = Path("weights") / "belfort"

SAMPLE_SEED = 42
SAMPLE_SPLIT = {"train": 600, "validation": 60, "test": 140}  # of the 800 lines the eight row groups hold
SAMPLE_DIGEST = "b7e1dd684691a0eedb63a609311f4964e7732e5c1a8d254fe4e1293a8cd0964d"  # dataset_digest over the three default splits together; tests pin it
MIN_RECORDS = 8
MAX_RECORDS = 5_000
# BYOD (review FL-m2): the notebook selects the epoch on validation and scores and reload-checks the test split, so a
# split dataset needs at least this many validation and test records besides MIN_RECORDS training records.
MIN_VAL_RECORDS = 2
MIN_TEST_RECORDS = 2
MIN_TEXT_CHARS = 1
# The transcript bound. Named apart from pipeline.MAX_TEXT_CHARS (the phrase-grounding `text_input` bound, 1000): the
# notebook carries both modules into one namespace, where one name for both silently lowered the grounding ceiling
# to 512 (review FL-m1).
MAX_TRANSCRIPT_CHARS = 512
_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")


def _sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


class _HttpRangeFile(io.RawIOBase):
    """A seekable read-only view of one HTTPS object served with `Range` requests (what `pyarrow` needs to read a
    parquet footer and a few row groups without downloading the file)."""

    def __init__(self, url: str, size: int) -> None:
        self.url, self.size, self.pos = url, size, 0
        self.fetched = 0

    def readable(self) -> bool:
        return True

    def seekable(self) -> bool:
        return True

    def tell(self) -> int:
        return self.pos

    def seek(self, offset: int, whence: int = 0) -> int:
        base = {0: 0, 1: self.pos, 2: self.size}[whence]
        self.pos = max(0, base + offset)
        return self.pos

    def read(self, n: int = -1) -> bytes:
        if n is None or n < 0:
            n = self.size - self.pos
        if n <= 0 or self.pos >= self.size:
            return b""
        end = min(self.size, self.pos + n) - 1
        request = urllib.request.Request(self.url, headers={"Range": f"bytes={self.pos}-{end}", "User-Agent": "florence2-vision-language-pipeline"})
        with urllib.request.urlopen(request, timeout=300) as response:  # noqa: S310 (pinned https URL)
            if response.status != 206:
                raise ValueError(f"{self.url}: server ignored the Range request (HTTP {response.status})")
            data = response.read()
        self.fetched += len(data)
        self.pos += len(data)
        return data

    def readinto(self, buffer: Any) -> int:
        data = self.read(len(buffer))
        buffer[: len(data)] = data
        return len(data)


def _declared_size(url: str) -> int:
    request = urllib.request.Request(url, method="HEAD", headers={"User-Agent": "florence2-vision-language-pipeline"})
    with urllib.request.urlopen(request, timeout=60) as response:  # noqa: S310 (pinned https URL)
        length = response.headers.get("Content-Length")
    if length is None:
        raise ValueError(f"{url}: no Content-Length in the HEAD response")
    return int(length)


def _group_digest(rows: Sequence[Mapping[str, Any]]) -> tuple[str, int]:
    digest, total = hashlib.sha256(), 0
    for row in rows:
        data = row["image"]["bytes"]
        text = str(row["text"]).encode("utf-8")
        digest.update(data)
        digest.update(text)
        total += len(data) + len(text)
    return digest.hexdigest(), total


def fetch_corpus(
    *, cache_dir: str | Path | None = None, groups: Sequence[int] | None = None, opener: Any = None
) -> dict[int, list[dict[str, Any]]]:
    """Return the pinned row groups as lists of `{image, text}` (JPEG bytes, transcript), from the cache (one parquet
    file per row group) or the Hub (footer + the row groups it needs, over range requests). Every row group's decoded
    content is refused unless its SHA-256 and byte total match `ROW_GROUP_PINS`; a fresh fetch also checks the shard's
    declared size and row count."""
    import pyarrow.parquet as pq

    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    wanted = list(groups) if groups is not None else sorted(ROW_GROUP_PINS)
    out: dict[int, list[dict[str, Any]]] = {}
    reader = None
    for group in wanted:
        if group not in ROW_GROUP_PINS:
            raise ValueError(f"row group {group} has no pin; pinned groups are {sorted(ROW_GROUP_PINS)}")
        local = cache / f"test-rg{group}.parquet"
        rows: list[dict[str, Any]] | None = None
        if local.is_file():
            rows = pq.read_table(local).to_pylist()
            if _group_digest(rows) != ROW_GROUP_PINS[group]:
                rows = None  # stale or corrupt cache: refetch
        if rows is None:
            if reader is None:
                if opener is not None:
                    reader = pq.ParquetFile(opener(CORPUS_URL))
                else:
                    declared = _declared_size(CORPUS_URL)
                    if declared != CORPUS_BYTES:
                        raise ValueError(f"{CORPUS_FILE}: declared size {declared} != pinned {CORPUS_BYTES}")
                    reader = pq.ParquetFile(_HttpRangeFile(CORPUS_URL, CORPUS_BYTES))
                if reader.metadata.num_rows != CORPUS_ROWS:
                    raise ValueError(f"{CORPUS_FILE}: {reader.metadata.num_rows} rows, pinned {CORPUS_ROWS}")
            table = reader.read_row_group(group, columns=["image", "text"])
            rows = table.to_pylist()
            digest, total = _group_digest(rows)
            if (digest, total) != ROW_GROUP_PINS[group]:
                raise ValueError(f"{CORPUS_FILE} row group {group}: sha256 {digest} / {total} bytes != pinned {ROW_GROUP_PINS[group]}")
            pq.write_table(table, local)
        out[group] = [{"image": r["image"]["bytes"], "text": str(r["text"])} for r in rows]
    return out


def read_corpus(groups: Mapping[int, Sequence[Mapping[str, Any]]]) -> list[dict[str, Any]]:
    """Decode the verified row groups into records (one per line; lines with an empty transcript are skipped)."""
    out = []
    for group in sorted(groups):
        for index, row in enumerate(groups[group]):
            text = normalise_text(row["text"])
            if not text:
                continue
            image = Image.open(io.BytesIO(row["image"]))
            image.load()
            out.append({"id": f"belfort-test-{group * 100 + index}", "image": image.convert("RGB"), "text": text, "source_row_group": group})
    return out


def build_sample_dataset(
    records: Sequence[Mapping[str, Any]], *, seed: int = SAMPLE_SEED, sizes: Mapping[str, int] | None = None
) -> dict[str, list[dict[str, Any]]]:
    """Seeded line-level draw: shuffle the records and cut `sizes` (train / validation / test) in order.

    The split unit is the line image: no image is in two splits, but the parquet carries no page or writer field, so
    lines cut from one page can land in different splits and the held-out rates are a same-collection estimate, not a
    page-disjoint one (review FL-M3; `transcript_overlap` reports the verbatim transcript repeats)."""
    sizes = dict(sizes or SAMPLE_SPLIT)
    pool = [dict(r) for r in records]
    random.Random(seed).shuffle(pool)
    needed = sum(sizes.values())
    if len(pool) < needed:
        raise ValueError(f"only {len(pool)} records available, need {needed}")
    out, cursor = {}, 0
    for name, count in sizes.items():
        out[name] = pool[cursor : cursor + count]
        cursor += count
    return out


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, seed: int = SAMPLE_SEED) -> dict[str, list[dict[str, Any]]]:
    return build_sample_dataset(read_corpus(fetch_corpus(cache_dir=cache_dir)), seed=seed)


# ---------------------------------------------------------------------------------------------------------
# Record contract
# ---------------------------------------------------------------------------------------------------------


def _open(image: Any, where: str) -> Image.Image:
    if isinstance(image, str | Path):
        path = Path(image)
        if not path.is_file():
            raise ValueError(f"{where}: image file not found: {path}")
        image = Image.open(path)
        image.load()
    if not isinstance(image, Image.Image):
        raise ValueError(f"{where}: must be a PIL.Image.Image or a file path")
    return image


def _check_record(record: Any, index: int) -> dict[str, Any]:
    where = f"records[{index}]"
    if not isinstance(record, Mapping):
        raise ValueError(f"{where} must be a mapping with id/image/text")
    for key in ("id", "image", "text"):
        if key not in record:
            raise ValueError(f"{where} is missing {key!r}")
    rid = record["id"]
    if not isinstance(rid, str) or not _ID_RE.match(rid):
        raise ValueError(f"{where}: id must match {_ID_RE.pattern}")
    try:
        image = validate_image(_open(record["image"], f"{where}.image"))
    except TypeError as exc:
        raise ValueError(f"{where}: {exc}") from exc
    except ValueError as exc:
        raise ValueError(f"{where}: {exc}") from exc
    if not isinstance(record["text"], str):
        raise ValueError(f"{where}: text must be a str")
    text = normalise_text(record["text"])
    if not MIN_TEXT_CHARS <= len(text) <= MAX_TRANSCRIPT_CHARS:
        raise ValueError(f"{where}: text has {len(text)} characters after whitespace normalisation; {MIN_TEXT_CHARS}..{MAX_TRANSCRIPT_CHARS} are required")
    item = {"id": rid, "image": image, "text": text}
    if "source_row_group" in record:
        item["source_row_group"] = record["source_row_group"]
    return item


def validate_dataset(
    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS
) -> dict[str, Any]:
    """Structural validation of a text-line dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):
        raise ValueError("records must be a list of {id, image, text} mappings")
    if not min_records <= len(records) <= max_records:
        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")
    checked, ids = [], set()
    for index, record in enumerate(records):
        item = _check_record(record, index)
        if item["id"] in ids:
            raise ValueError(f"duplicate id {item['id']!r}")
        ids.add(item["id"])
        checked.append(item)
    chars = [len(r["text"]) for r in checked]
    words = [len(r["text"].split()) for r in checked]
    widths = [r["image"].width for r in checked]
    heights = [r["image"].height for r in checked]
    return {
        "records": checked,
        "n_records": len(checked),
        "text_chars": {"min": min(chars), "max": max(chars), "total": sum(chars)},
        "text_words": {"total": sum(words)},
        "image_width": {"min": min(widths), "max": max(widths)},
        "image_height": {"min": min(heights), "max": max(heights)},
        "digest": dataset_digest(checked),
        "model_id": MODEL_ID,
    }


def image_digest(image: Image.Image) -> str:
    """SHA-256 of the decoded RGB pixels (size-prefixed) — the identity a split is made disjoint on."""
    rgb = image.convert("RGB")
    return _sha256_bytes(f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes())


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """Order-independent SHA-256 over (id, image digest, normalised text)."""
    parts = sorted(f"{r['id']}:{image_digest(r['image'])}:{normalise_text(r['text'])}" for r in records)
    return _sha256_bytes("\n".join(parts).encode("utf-8"))


def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:
    """Assert no image (by decoded-pixel digest) appears in two splits (leakage check)."""
    seen: dict[str, str] = {}
    for name, records in splits.items():
        for record in records:
            key = image_digest(record["image"])
            if key in seen and seen[key] != name:
                raise ValueError(f"image {record['id']!r} appears in both {seen[key]} and {name}")
            seen[key] = name
    return {name: len(records) for name, records in splits.items()}


def drop_duplicate_images(records: Sequence[Mapping[str, Any]]) -> tuple[list[dict[str, Any]], list[str]]:
    """Keep the first record of every decoded image; return the kept records and the ids of the dropped duplicates
    (review FL-m2: a caller can report what `split_dataset` would otherwise drop silently)."""
    seen: set[str] = set()
    kept, dropped = [], []
    for record in records:
        key = image_digest(record["image"])
        if key in seen:
            dropped.append(str(record["id"]))
        else:
            seen.add(key)
            kept.append(dict(record))
    return kept, dropped


def _split_counts(n: int, val_fraction: float, test_fraction: float) -> tuple[int, int, int]:
    n_test = max(1, round(n * test_fraction))
    n_val = round(n * val_fraction)
    return n - n_test - n_val, n_val, n_test


def byod_record_limits(val_fraction: float = 0.15, test_fraction: float = 0.2) -> tuple[int, int]:
    """Smallest and largest number of distinct images `split_dataset` accepts at these fractions: at least
    MIN_RECORDS training, MIN_VAL_RECORDS validation and MIN_TEST_RECORDS test records, at most MAX_RECORDS in all."""
    for n in range(1, MAX_RECORDS + 1):
        train, val, test = _split_counts(n, val_fraction, test_fraction)
        if train >= MIN_RECORDS and val >= MIN_VAL_RECORDS and test >= MIN_TEST_RECORDS:
            return n, MAX_RECORDS
    raise ValueError("no dataset size gives every split its minimum at these fractions")


def split_dataset(
    records: Sequence[Mapping[str, Any]], *, val_fraction: float = 0.15, test_fraction: float = 0.2, seed: int = 0
) -> dict[str, list[dict[str, Any]]]:
    """Seeded shuffle of a BYOD dataset into train/validation/test after de-duplicating images.

    The split must leave at least MIN_RECORDS training, MIN_VAL_RECORDS validation and MIN_TEST_RECORDS test records;
    a refusal names the split, its count and the dataset size that works (`byod_record_limits`) before any model runs
    (review FL-m2)."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records, min_records=1)["records"]
    unique, dropped = drop_duplicate_images(checked)
    random.Random(seed).shuffle(unique)
    n = len(unique)
    n_train, n_val, n_test = _split_counts(n, val_fraction, test_fraction)
    needed = (("train", n_train, MIN_RECORDS), ("validation", n_val, MIN_VAL_RECORDS), ("test", n_test, MIN_TEST_RECORDS))
    short = [(name, have, least) for name, have, least in needed if have < least]
    if short:
        name, have, least = short[0]
        try:
            low, _high = byod_record_limits(val_fraction, test_fraction)
            sizes = f"a dataset needs at least {low} distinct images"
        except ValueError:
            sizes = "no dataset size gives every split its minimum at these fractions"
        duplicates = f" ({len(dropped)} duplicate image(s) dropped)" if dropped else ""
        raise ValueError(
            f"the {name} split would hold {max(have, 0)} records (at least {least} are required): {len(checked)} records, "
            f"{n} distinct images{duplicates}, split into train/validation/test as {max(n_train, 0)}/{n_val}/{n_test}; "
            f"{sizes}. Add records"
        )
    return {"test": unique[:n_test], "validation": unique[n_test : n_test + n_val], "train": unique[n_test + n_val :]}


def transcript_overlap(train: Sequence[Mapping[str, Any]], other: Sequence[Mapping[str, Any]]) -> list[str]:
    """Ids of the `other` records whose normalised transcript also occurs verbatim in `train` (review FL-M3: formulaic
    lines such as "Le Conseil Municipal" repeat across pages, so an image-disjoint split can still test text it trained
    on)."""
    seen = {normalise_text(r["text"]) for r in train}
    return [str(r["id"]) for r in other if normalise_text(r["text"]) in seen]


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Records from a directory or zip holding line images and a `transcripts.csv` with the columns `file` and `text`
    (and optionally `id`); every image file must have a transcript row and every row an image."""
    source = Path(path)
    members: dict[str, bytes] = {}
    origin: dict[str, str] = {}

    def add(name: str, where: str, data: bytes) -> None:
        # The archive is flattened (no extractall): two files with one base name would overwrite each other (FL-m2).
        if name in members:
            raise ValueError(f"two files are named {name!r} ({origin[name]} and {where}); the BYOD archive is read without its folders, so every file name must be unique - rename one")
        members[name], origin[name] = data, where

    if source.is_dir():
        for file in sorted(source.rglob("*")):
            if file.is_file() and "__MACOSX" not in file.parts:
                add(file.name, str(file.relative_to(source)), file.read_bytes())
    elif zipfile.is_zipfile(source):
        with zipfile.ZipFile(source) as archive:
            for info in archive.infolist():
                if not info.is_dir() and not info.filename.startswith("__MACOSX/"):
                    add(Path(info.filename).name, info.filename, archive.read(info))
    else:
        raise ValueError(f"{source} is neither a directory nor a zip file")
    if "transcripts.csv" not in members:
        raise ValueError("BYOD data must include transcripts.csv with the columns file and text")
    rows = list(csv.DictReader(io.StringIO(members["transcripts.csv"].decode("utf-8-sig"))))
    if not rows or "file" not in rows[0] or "text" not in rows[0]:
        raise ValueError("transcripts.csv must have the columns file and text")
    out = []
    for row in rows:
        name = Path(str(row.get("file", "")).strip()).name
        if name not in members:
            raise ValueError(f"transcripts.csv names a missing image: {name}")
        try:
            image = Image.open(io.BytesIO(members[name]))
            image.load()
        except Exception as exc:  # noqa: BLE001
            raise ValueError(f"BYOD file is not a decodable image: {name}") from exc
        rid = str(row.get("id", "") or "").strip()
        out.append({"id": rid or re.sub(r"[^A-Za-z0-9_.:-]", "_", Path(name).stem)[:64], "image": image.convert("RGB"), "text": str(row.get("text", ""))})
    listed = {Path(str(r.get("file", "")).strip()).name for r in rows}
    unlisted = [n for n in members if n != "transcripts.csv" and n not in listed]
    if unlisted:
        raise ValueError(f"{len(unlisted)} image file(s) have no transcripts.csv row, e.g. {unlisted[0]}")
    return out


def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """A summary table (id, image size, transcript length, transcript, provenance) in the BYOD `transcripts.csv`
    column layout plus extras (`file` names the id; the images themselves are not written)."""
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    with open(out, "w", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        writer.writerow(["id", "file", "width", "height", "chars", "words", "text", "source_row_group"])
        for r in records:
            text = normalise_text(r["text"])
            writer.writerow([r["id"], f"{r['id']}.jpg", r["image"].width, r["image"].height, len(text), len(text.split()), text, r.get("source_row_group", "")])
    return out

## 3. Pin, stage and verify the model

> **Infrastructure.** You may run this section without studying its implementation; the learning activities start after Section 3.

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `12`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `4271c66b88cd…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `Florence2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
# @title Infrastructure: pin, stage and verify the model (you may run it without reading it)
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "florence-2-large-community",
  "modelId": "florence-community/Florence-2-large",
  "revision": "4271c66b88cdbc05735372ec13b2360108de5317",
  "files": [
    {
      "path": "README.md",
      "bytes": 13445,
      "sha256": "5662d3853b245397062aa0b1853958f23305e0b9518071293c5154c7eeb415d2"
    },
    {
      "path": "added_tokens.json",
      "bytes": 22430,
      "sha256": "1d75deda84dfa81fb6c09301f3fed00f9695059568bbb1403a6bf299cd84fc37"
    },
    {
      "path": "config.json",
      "bytes": 2396,
      "sha256": "8412483f687f2f71587328a38c6fa70a68d9488f28e90607be3c182641f60f2c"
    },
    {
      "path": "generation_config.json",
      "bytes": 292,
      "sha256": "0251459c49cc358ac033b5d4b8569e61ac22bde5d76be404b97a44cfb33fb12e"
    },
    {
      "path": "merges.txt",
      "bytes": 456318,
      "sha256": "1ce1664773c50f3e0cc8842619a93edc4624525b728b188a9e0be33b7726adc5"
    },
    {
      "path": "model.safetensors",
      "bytes": 1553541016,
      "sha256": "7715423d6549bf1e71188bdd84f4ac960cc0597886af24a5ef7b66f128660685"
    },
    {
      "path": "preprocessor_config.json",
      "bytes": 603,
      "sha256": "1396ec5a0a7adfe1c04fb777b09e8ba753be6dbb5868212ab3c3ef39d91fe031"
    },
    {
      "path": "processor_config.json",
      "bytes": 2264,
      "sha256": "cd0e3bf41a39b1276503fbd273bc03b9afc70d7a15ea681a92a1b4b77f858ee6"
    },
    {
      "path": "special_tokens_map.json",
      "bytes": 146627,
      "sha256": "72ff172dc769bc1551b1b4211628ce3271643bc60379e4da45d85a9be9332c39"
    },
    {
      "path": "tokenizer.json",
      "bytes": 3748144,
      "sha256": "3ad7001f773409abe6bba33eac92662611a73d72f459bda2f00d2a221dd31ce4"
    },
    {
      "path": "tokenizer_config.json",
      "bytes": 197922,
      "sha256": "cb5f80bd9afa767bb1bb798ee5f0a79eae45239b8e72506166b4218175a16723"
    },
    {
      "path": "vocab.json",
      "bytes": 798293,
      "sha256": "ed19656ea1707df69134c4af35c8ceda2cc9860bf2c3495026153a133670ab5e"
    }
  ],
  "totalBytes": 1558929750
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = Florence2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

{'model_id': 'florence-community/Florence-2-large', 'revision': '4271c66b88cdbc05735372ec13b2360108de5317', 'license': 'mit', 'files': 12, 'total_bytes': 1558929750}


{'weights_dir': '/content/weights/florence-2-large-community', 'fetched': ['README.md', 'added_tokens.json', 'config.json', 'generation_config.json', 'merges.txt', 'model.safetensors', 'preprocessor_config.json', 'processor_config.json', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'vocab.json']}


{'verified_files': 12, 'revision': '4271c66b88cdbc05735372ec13b2360108de5317'}


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.


{'device': 'cuda:0', 'source': 'local-snapshot'}


## 4. Belfort lines, the transcripts and the split

`fetch_corpus` returns the eight pinned row groups from the cache under `weights/belfort/` or the Hub at the pinned parquet-conversion revision — `pyarrow` reads the shard's footer and exactly those row groups over HTTPS range requests; every cached file is re-hashed and every fetched row group refused on any SHA-256 or byte-total mismatch — and `read_corpus` turns each row into a record: the line image (128 px tall, 145 to 8,956 px wide) and its crowdsourced transcript with whitespace runs collapsed. `build_sample_dataset` draws a seeded split **by line image** (600 / 60 / 140). `validate_dataset` then checks every record against the contract, `check_split_disjoint` asserts no image (by decoded-pixel digest) is shared, and the training split's summary table is written to `outputs/florence2_vision_language_train.csv`.

**What the split does and does not protect against.** No line image is in two splits. But the Belfort parquet has only two columns, `image` and `text` — no page, writer or register — so the split cannot keep a page's lines together: lines cut from one page can sit in training and in test, in the same hand. Nor would cutting the corpus into runs of consecutive lines help, because the shard is not in page order (neighbouring lines come from different years and registers: in the 800 lines, six of the eight pairs of year-bearing lines at most three rows apart name years more than two years apart, and the width of one line image predicts the next one's with a correlation of 0.02). The held-out rates in Section 8 are therefore a **same-collection estimate**: they measure lines from pages and hands the model may have trained on, and are likely optimistic for a new register or a new hand. A second, smaller overlap is textual: formulaic lines such as "Le Conseil Municipal" repeat across pages, so a few test transcripts also occur verbatim in training. The cell counts them (`transcript_overlap`), and Section 8 reports the rates with and without those lines.

**Bring your own data:** set `USE_BYOD = True`, either leave `BYOD_PATH` empty to get an upload dialog (Colab only) or set it to a zip or folder already in this runtime, then select this cell and choose **Runtime → Run after**. An empty or cancelled upload, a runtime without the Colab upload dialog and no `BYOD_PATH`, or a path that does not exist stops with a message saying what to supply. `load_byod_dataset` refuses two files with one name (the archive is read without its folders), a missing `transcripts.csv`, an image without a row or a row without an image; pixel-duplicate images are counted and named before the split drops them; and `split_dataset` refuses a dataset that leaves fewer than 8 training, 2 validation or 2 test lines, naming the split (at least 12 distinct images). Under BYOD the printed provenance and `result.json` describe your upload, not the Belfort corpus.

Look for: 800 lines and 33,117 reference characters, three digests, the number of validation and test transcripts that also occur in training, and four refusal probes — a duplicate id, an empty transcript, an image above the side ceiling, and a dataset too small to use — each rejected before the model does anything.

**Predict before running:** of the 140 test lines, how many do you expect to have a transcript that also appears, word for word, among the 600 training lines — none, a handful, or dozens?

In [ ]:
import hashlib
import json
import time

import numpy as np
from PIL import Image, ImageDraw, ImageFont

USE_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
SPLIT_SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
byod = None
if USE_BYOD:
    if BYOD_PATH.strip():
        byod_source = Path(BYOD_PATH.strip()).expanduser()
        if not byod_source.exists():
            raise FileNotFoundError(f'BYOD_PATH = {BYOD_PATH!r} does not exist in this runtime: fix the path, or leave BYOD_PATH empty to upload a zip in Colab.')
        file_name, payload = byod_source.name, (byod_source.read_bytes() if byod_source.is_file() else None)
    else:
        try:
            from google.colab import files
        except ImportError:
            raise RuntimeError('The upload dialog needs Google Colab. Elsewhere, put the zip (or folder) in this runtime and set BYOD_PATH to its path.') from None
        uploaded = files.upload()
        if len(uploaded) != 1:
            raise ValueError(f'Upload exactly one .zip file (received {len(uploaded)}). Run this cell again and choose a file, or set BYOD_PATH to a zip or folder already in this runtime.')
        file_name, payload = next(iter(uploaded.items()))
        byod_source = Path('work') / 'byod.zip'
        byod_source.parent.mkdir(parents=True, exist_ok=True)
        byod_source.write_bytes(payload)
    records = load_byod_dataset(byod_source)
    unique_records, dropped_duplicates = drop_duplicate_images(records)
    print({'byod_records': len(records), 'distinct_images': len(unique_records), 'pixel_duplicates_dropped': len(dropped_duplicates), 'dropped_ids': dropped_duplicates[:20], 'needed': f'at least {byod_record_limits()[0]} distinct images'})
    splits = split_dataset(unique_records, seed=SPLIT_SEED)
    data_source = 'BYOD (' + file_name + ')'
    raw_rows = {'byod': len(records), 'distinct_images': len(unique_records)}
    byod = {'file': file_name, 'records': len(records), 'distinct_images': len(unique_records), 'duplicates_dropped': dropped_duplicates, 'zip_sha256': hashlib.sha256(payload).hexdigest() if payload is not None else None}
else:
    t0 = time.perf_counter()
    corpus_groups = fetch_corpus(cache_dir='weights/belfort')
    corpus = read_corpus(corpus_groups)
    splits = build_sample_dataset(corpus, seed=SPLIT_SEED)
    data_source = f'{CORPUS_NAME} @ {CORPUS_REVISION[:12]} ({CORPUS_LICENSE})'
    raw_rows = {'row_groups': len(corpus_groups), 'lines': sum(len(v) for v in corpus_groups.values()), 'bytes': sum(len(r['image']) + len(r['text'].encode('utf-8')) for v in corpus_groups.values() for r in v), 'seconds': round(time.perf_counter() - t0, 1)}
dataset_manifests = {name: validate_dataset(part, min_records=MIN_RECORDS if name == 'train' else 1) for name, part in splits.items()}
splits = {name: manifest['records'] for name, manifest in dataset_manifests.items()}
disjoint = check_split_disjoint(splits)
train_records, val_records, test_records = splits['train'], splits['validation'], splits['test']
write_dataset_csv(train_records, 'outputs/florence2_vision_language_train.csv')
print({'data_source': data_source, 'raw_rows': raw_rows, 'splits': disjoint, 'split_unit': 'line image (no page or writer field)'})
for name, manifest in dataset_manifests.items():
    print({name: {'n': manifest['n_records'], 'chars': manifest['text_chars'], 'words': manifest['text_words']['total'], 'width': manifest['image_width'], 'height': manifest['image_height'], 'digest': manifest['digest'][:16] + '...'}})
overlap = {'validation': transcript_overlap(train_records, val_records), 'test': transcript_overlap(train_records, test_records)}
test_by_id = {r['id']: r for r in test_records}
print({'transcripts_also_in_train': {name: len(ids) for name, ids in overlap.items()}, 'test_examples': [test_by_id[i]['text'] for i in overlap['test'][:5]]})
example = train_records[0]
example['image'].save('outputs/florence2_vision_language_example_line.png')
print({'example': {'id': example['id'], 'image': list(example['image'].size), 'text': example['text']}})

probes = {
    'duplicate id': [{**r, 'id': 'same'} for r in train_records[:8]],
    'empty transcript': [{**train_records[0], 'text': '   '}, *train_records[1:8]],
    'image above the side ceiling': [{**train_records[0], 'image': Image.new('RGB', (MAX_IMAGE_SIDE + 1, 8))}, *train_records[1:8]],
    'too small': train_records[:3],
}
for name, probe in probes.items():
    try:
        validate_dataset(probe)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

{'data_source': 'Belfort-line (test split), first eight parquet row groups @ c4a74bbd39f2 (MIT (Teklia; Belfort municipal council minutes, Zenodo record 8041668; Tarride et al. 2023, https://doi.org/10.1145/3604951.3605517))', 'raw_rows': {'row_groups': 8, 'lines': 800, 'bytes': 42403899, 'seconds': 16.4}, 'splits': {'train': 600, 'validation': 60, 'test': 140}, 'split_unit': 'line image (no page or writer field)'}
{'train': {'n': 600, 'chars': {'min': 1, 'max': 120, 'total': 24494}, 'words': 4150, 'width': {'min': 145, 'max': 8956}, 'height': {'min': 128, 'max': 128}, 'digest': '64186dce18ca12d3...'}}
{'validation': {'n': 60, 'chars': {'min': 3, 'max': 98, 'total': 2454}, 'words': 412, 'width': {'min': 306, 'max': 7543}, 'height': {'min': 128, 'max': 128}, 'digest': '5768e17faca8ca13...'}}
{'test': {'n': 140, 'chars': {'min': 2, 'max': 105, 'total': 6169}, 'words': 1035, 'width': {'min': 197, 'max': 7872}, 'height': {'min': 128, 'max': 128}, 'digest': '0e815da406cccff9...'}}
{'transcr

**What to notice:** the three split sizes, `split_unit`, the transcript-overlap counts with their examples, and four rejected probes.

<details><summary>Check your reasoning</summary>A handful. With the default seed, 5 of the 140 test lines and 2 of the 60 validation lines have a transcript that also occurs in training: formulaic lines ("Le Conseil Municipal", "Après en avoir délibéré, à l'unanimité :") and bare numbers ("100", "440"). They are 84 of the 6,169 reference characters of the test split, so they cannot move the corpus CER much, and Section 8 shows the rates without them. The larger, unmeasurable overlap is the one the dataset cannot show: lines of the same page and hand on both sides of the split.</details>

## 5. Run three capabilities on a synthetic drawing through the inference contract

The inference contract is exercised as the multi-capability tutorial exercised it: a 512 × 512 white canvas drawn in code — a filled red square, a filled blue circle and the text `DIMER 2026` in Pillow's bundled font — whose drawn text is the **known OCR reference**; a different image family from the handwritten lines, and a drawing the adapted model will see again in Section 9. `validate_inputs` applies exactly the checks `run` applies (image sides `MIN_IMAGE_SIDE`..`MAX_IMAGE_SIDE` and at most `MAX_IMAGE_PIXELS`, a task token inside `TASKS`, a `text_input` only for the tasks that take one, `max_new_tokens` in 1..`MAX_NEW_TOKENS`, a positive `num_beams`) and one combined manifest records the requests; an unsupported task token is validated too and its rejection recorded as a finding. Each entry of `CAPABILITIES` may carry a `text_input`, which is passed to `validate_inputs` and `run` (only `<CAPTION_TO_PHRASE_GROUNDING>` takes one). `run` returns the task-parsed `result` — a caption string, `{bboxes, labels}` in input pixels with **no confidence scores**, an OCR string — the raw `generated_text` and the generation settings; structural checks assert each output's contract. The per-image `evaluation_report` over the results scores OCR against the drawn text (`sample-sanity`) and reports the caption and the detections `not-measurable` — plumbing evidence, not a measurement; whether the model is *good at handwriting* is what Section 6 measures on 140 lines. The printed ceilings name both text bounds: `MAX_TEXT_CHARS` (1000) for a grounding `text_input` and `MAX_TRANSCRIPT_CHARS` (512) for a training transcript.

**Every pass starts from the pretrained model.** Section 7 changes the model in memory. If this cell, Section 6 or Section 7 runs again after Section 7 (a BYOD run or one of the optional experiments), it first reloads the pretrained model from the verified snapshot with `reset_to_pretrained()`, so a *frozen* number is never read from the adapted model.

**Predict before running:** the drawing holds a red square, a blue circle and two words. How many boxes will `<OD>` return, and what labels? Will `<OCR>` read `DIMER 2026` exactly?

In [ ]:
import gc


def reset_to_pretrained():
    """Sections 5-7 start from the pretrained model: if an earlier pass adapted `pipe`, reload it from the verified snapshot."""
    global pipe
    if pipe.adapter is None:
        return
    pipe = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    pipe = Florence2Pipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
    print({'reloaded': 'the pretrained model, from the verified snapshot', 'reason': 'an earlier pass had adapted the model in memory'})


reset_to_pretrained()
drawing = Image.new('RGB', (512, 512), (255, 255, 255))
draw = ImageDraw.Draw(drawing)
drawn_square = (64, 64, 224, 224)
draw.rectangle(drawn_square, fill=(220, 30, 30))
draw.ellipse((300, 96, 460, 256), fill=(30, 60, 220))
drawing_reference = 'DIMER 2026'
draw.text((96, 360), drawing_reference, fill=(0, 0, 0), font=ImageFont.load_default(size=48))
drawing_name = 'synthetic_shapes_text_512'
drawing_sha256 = hashlib.sha256(np.asarray(drawing).tobytes()).hexdigest()
print({'ceilings': {'MIN_IMAGE_SIDE': MIN_IMAGE_SIDE, 'MAX_IMAGE_SIDE': MAX_IMAGE_SIDE, 'MAX_IMAGE_PIXELS': MAX_IMAGE_PIXELS, 'MAX_NEW_TOKENS': MAX_NEW_TOKENS, 'DEFAULT_MAX_NEW_TOKENS': DEFAULT_MAX_NEW_TOKENS, 'DEFAULT_LINE_MAX_NEW_TOKENS': DEFAULT_LINE_MAX_NEW_TOKENS, 'MAX_TEXT_CHARS (grounding text_input)': MAX_TEXT_CHARS, 'MAX_TRANSCRIPT_CHARS (training transcript)': MAX_TRANSCRIPT_CHARS, 'NUM_BEAMS': NUM_BEAMS, 'MIN_RECORDS': MIN_RECORDS, 'MAX_RECORDS': MAX_RECORDS, 'device': pipe.device, 'dtype': pipe.dtype}})
print({'TASKS_WITHOUT_TEXT': TASKS_WITHOUT_TEXT, 'TASKS_WITH_TEXT': TASKS_WITH_TEXT})
CAPABILITIES = {
    '<CAPTION>': {'input': 'image only', 'output': 'result: str (one short caption); no score', 'max_new_tokens': 64},
    '<OD>': {'input': 'image only', 'output': "result: {'bboxes': [[x1, y1, x2, y2], ...] in input pixels, 'labels': [str, ...]}; no per-box score", 'max_new_tokens': DEFAULT_MAX_NEW_TOKENS},
    '<OCR>': {'input': 'image only', 'output': 'result: str (transcribed text, reading order chosen by the model); no score', 'max_new_tokens': DEFAULT_LINE_MAX_NEW_TOKENS},
}
manifests = {task: validate_inputs(drawing, task, contract.get('text_input'), max_new_tokens=contract['max_new_tokens'], num_beams=NUM_BEAMS, names=[drawing_name]) for task, contract in CAPABILITIES.items()}
input_manifest = {**manifests['<CAPTION>'], 'task': 'multi-capability: ' + ', '.join(CAPABILITIES), 'tasks': list(CAPABILITIES), 'findings': [], 'capabilities': manifests}
try:
    validate_inputs(drawing, '<REFERRING_EXPRESSION_SEGMENTATION>')
except ValueError as exc:
    input_manifest['findings'].append({'input': 'unsupported-task-probe', 'task': '<REFERRING_EXPRESSION_SEGMENTATION>', 'verdict': 'rejected', 'message': str(exc)})
with open('outputs/florence2_vision_language_input_manifest.json', 'w', encoding='utf-8') as handle:
    json.dump(input_manifest, handle, indent=2, ensure_ascii=False)
print({'drawing': drawing_name, 'sha256': drawing_sha256[:16] + '...', 'manifest_verdict': input_manifest['verdict'], 'findings': len(input_manifest['findings'])})


def run_capabilities(pipeline, label):
    results, timings = {}, {}
    for task, contract in CAPABILITIES.items():
        started = time.perf_counter()
        results[task] = pipeline.run(drawing, task, contract.get('text_input'), max_new_tokens=contract['max_new_tokens'], num_beams=NUM_BEAMS)
        timings[task] = round(time.perf_counter() - started, 3)
    caption, detections, ocr_text = results['<CAPTION>']['result'], results['<OD>']['result'], results['<OCR>']['result']
    checks = {
        'caption_is_text': isinstance(caption, str),
        'od_boxes_and_labels_aligned': isinstance(detections, dict) and len(detections.get('bboxes', [])) == len(detections.get('labels', [])),
        'od_boxes_inside_image': all(0 <= x1 <= x2 <= drawing.width and 0 <= y1 <= y2 <= drawing.height for x1, y1, x2, y2 in detections.get('bboxes', [])),
        'ocr_is_text': isinstance(ocr_text, str),
        'deterministic_settings': all(r['generation']['do_sample'] is False and r['generation']['num_beams'] == NUM_BEAMS for r in results.values()),
        'identity_reported': all(r['model_id'] == MODEL_ID and r['model_revision'] == MODEL_REVISION for r in results.values()),
    }
    if not all(checks.values()):
        raise RuntimeError(f'capability output failed a sanity check: {checks}')
    report = evaluation_report(list(results.values()), drawing_reference, sample_kind='synthetic (drawn in this notebook)')
    preview = drawing.copy()
    marker = ImageDraw.Draw(preview)
    for (x1, y1, x2, y2), name in zip(detections.get('bboxes', []), detections.get('labels', [])):
        marker.rectangle((x1, y1, x2, y2), outline=(0, 160, 0), width=3)
        marker.text((x1 + 4, y1 + 4), name, fill=(0, 160, 0))
    preview.save(f'outputs/florence2_vision_language_preview_{label}.png')
    extra = {task: {'text_input': CAPABILITIES[task].get('text_input'), 'result': r['result']} for task, r in results.items() if task not in ('<CAPTION>', '<OD>', '<OCR>')}
    print({label: {'seconds': timings, 'checks': checks, 'caption': caption, 'detections': list(zip(detections.get('labels', []), [[round(v) for v in b] for b in detections.get('bboxes', [])])), 'ocr': ocr_text, 'ocr_cer': {m['id']: round(m['value'], 4) for m in report['metrics']}, 'verdict': report['verdict'], **({'other_tasks': extra} if extra else {})}})
    return results, timings, checks, report


frozen_results, frozen_timings, frozen_checks, frozen_report = run_capabilities(pipe, 'frozen')

{'ceilings': {'MIN_IMAGE_SIDE': 1, 'MAX_IMAGE_SIDE': 16384, 'MAX_IMAGE_PIXELS': 16777216, 'MAX_NEW_TOKENS': 1024, 'DEFAULT_MAX_NEW_TOKENS': 256, 'DEFAULT_LINE_MAX_NEW_TOKENS': 128, 'MAX_TEXT_CHARS (grounding text_input)': 1000, 'MAX_TRANSCRIPT_CHARS (training transcript)': 512, 'NUM_BEAMS': 3, 'MIN_RECORDS': 8, 'MAX_RECORDS': 5000, 'device': 'cuda:0', 'dtype': 'float32'}}
{'TASKS_WITHOUT_TEXT': ('<CAPTION>', '<DETAILED_CAPTION>', '<MORE_DETAILED_CAPTION>', '<OD>', '<DENSE_REGION_CAPTION>', '<REGION_PROPOSAL>', '<OCR>', '<OCR_WITH_REGION>'), 'TASKS_WITH_TEXT': ('<CAPTION_TO_PHRASE_GROUNDING>',)}
{'drawing': 'synthetic_shapes_text_512', 'sha256': '950f2e0f69919f36...', 'manifest_verdict': 'accepted', 'findings': 1}


{'frozen': {'seconds': {'<CAPTION>': 2.511, '<OD>': 0.756, '<OCR>': 0.739}, 'checks': {'caption_is_text': True, 'od_boxes_and_labels_aligned': True, 'od_boxes_inside_image': True, 'ocr_is_text': True, 'deterministic_settings': True, 'identity_reported': True}, 'caption': 'a red and blue circle with the word dimer 2026 on it', 'detections': [('poster', [0, 0, 511, 511])], 'ocr': 'DIMER 2026', 'ocr_cer': {'character_error_rate': 0.0}, 'verdict': 'sample-sanity'}}


**What to notice:** the two text bounds in the ceilings, the three `checks`, the caption, the box list with its labels, the OCR string and its CER against the drawn text.

<details><summary>Check your reasoning</summary>In the recorded Tesla T4 run of the previous notebook version (same model stages, float32) the frozen model captioned the drawing `a red and blue circle with the word dimer 2026 on it`, returned **one** box covering the whole drawing (`[0, 0, 511, 511]`) labelled `poster` — not one box per shape — and read `DIMER 2026` exactly (CER 0.000). One box around everything is a legitimate `<OD>` answer for a poster-like image; nothing in the output tells you it is less useful than three boxes, because there is no score. Printed text in a clean font is what this model was trained on; the handwriting in Section 6 is not.</details>

## 6. Baselines and the frozen model on the test lines

Two non-adapted baselines frame the adaptation, each scored by `ocr_metrics` (carried in `metrics.py`): the **character error rate** and **word error rate** as micro averages — total Levenshtein edits over total reference characters or words, the corpus CER/WER of the handwriting-recognition literature — beside the macro (per-line mean) rates and the exact-match rate. Neither rate is capped: a hypothesis longer than its reference pushes the rate **above 1.0**, the signal that the model is generating text the line does not carry. The **empty-string** baseline predicts nothing and scores CER 1.0 exactly (every reference character is a deletion) — the floor any recogniser must beat to do better than silence. The **constant-transcript** baseline predicts one training transcript — the medoid, the line closest on average to the others — for every test line: what corpus statistics buy without reading the image. The **frozen model** is scored by `pipe.evaluate`, which runs `<OCR>` in batches of `EVAL_BATCH_SIZE` with `NUM_BEAMS` beams under a `LINE_MAX_NEW_TOKENS` budget and returns the hypotheses with the rates. The cell reloads the pretrained model first if an earlier pass adapted it, and refuses to report an adapted model as frozen; four hypotheses are printed under their references.

**Predict before running:** the frozen model reads printed text well (Section 5). On cursive nineteenth-century French, will its CER land nearer the empty baseline's 1.0, or nearer 0? Will its hypotheses be shorter or longer than the references?

In [ ]:
METRICS = ('cer', 'wer', 'cer_macro', 'exact_match')
LINE_MAX_NEW_TOKENS = 128  # @param {type:"integer"}

reset_to_pretrained()
baseline_empty = empty_baseline(test_records)
baseline_constant = constant_baseline(train_records, test_records)
print({'empty_baseline': {k: round(baseline_empty[k], 3) for k in METRICS}, 'n': baseline_empty['n'], 'note': baseline_empty['baseline']})
print({'constant_baseline': {k: round(baseline_constant[k], 3) for k in METRICS}, 'note': baseline_constant['baseline']})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_records, max_new_tokens=LINE_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, batch_size=EVAL_BATCH_SIZE)
if frozen_test['adapted']:
    raise RuntimeError('Section 6 must score the pretrained model, but the model in memory is adapted. Restart the session and choose Run all.')
print({'frozen_model_test': {k: round(frozen_test[k], 3) for k in METRICS}, 'n': frozen_test['n'], 'ref_chars': frozen_test['ref_chars'], 'hyp_chars': frozen_test['hyp_chars'], 'truncated': frozen_test['truncated'], 'verdict': frozen_test['verdict'], 'adapted': frozen_test['adapted'], 'seconds': round(time.perf_counter() - t0, 1)})
print({'definitions': frozen_test['definitions']})
for record, hypothesis in zip(test_records[:4], frozen_test['hypotheses'][:4], strict=True):
    print({'id': record['id'], 'reference': record['text'], 'frozen': hypothesis[:120]})

{'empty_baseline': {'cer': 1.0, 'wer': 1.0, 'cer_macro': 1.0, 'exact_match': 0.0}, 'n': 140, 'note': 'empty string for every line'}
{'constant_baseline': {'cer': 0.944, 'wer': 0.999, 'cer_macro': 0.952, 'exact_match': 0.0}, 'note': "constant training transcript 'idem'"}


{'frozen_model_test': {'cer': 0.992, 'wer': 1.002, 'cer_macro': 0.962, 'exact_match': 0.0}, 'n': 140, 'ref_chars': 6169, 'hyp_chars': 250, 'truncated': 0, 'verdict': 'measured', 'adapted': False, 'seconds': 87.7}
{'definitions': {'cer': 'character error rate, micro: total character edits (insertions + deletions + substitutions) over total reference characters; 0 is perfect, values above 1 mean over-generation', 'wer': 'word error rate, micro: total word edits over total reference words after lower-casing and whitespace tokenisation; punctuation kept', 'cer_macro': 'mean of the per-line character error rates (each line weighted equally regardless of length)', 'wer_macro': 'mean of the per-line word error rates', 'exact_match': 'fraction of lines whose hypothesis equals the reference after whitespace normalisation'}}
{'id': 'belfort-test-761', 'reference': "régulariser l'achat", 'frozen': '1'}
{'id': 'belfort-test-763', 'reference': 'Renvoi à la Commission des travaux de diverses questio

**What to notice:** `adapted: False`, the frozen CER beside the two baselines, `hyp_chars` against `ref_chars`, and the four printed hypotheses.

<details><summary>Check your reasoning</summary>Nearer 1.0, and much shorter. In the recorded T4 run the empty baseline scored CER 1.000, the constant transcript 0.944 and the frozen model **0.992** — essentially silence: its hypotheses held about 0.04 times as many characters as the references (a dash or nothing for most lines). The constant transcript beats it because a guessed common line shares some characters with many references. A model that reads print perfectly can read nothing in a hand it never saw: that is the gap the fine-tuning in Section 7 tries to close.</details>

## 7. Bounded fine-tuning of the last decoder layers

`pipe.adapt` trains only the last four of the 12 BART decoder layers and the decoder's embedding layer norm — 67,188,736 of 776,505,344 parameters — while the DaViT vision tower, the projector, the BART encoder, the shared embeddings (tied to the output head) and the first eight decoder layers stay frozen. Each training line is the `<OCR>` prompt with its 577 visual tokens on the encoder side and, on the decoder side, the transcript's tokens between the start and end tokens; the loss is the **sequence-to-sequence cross-entropy** over those target tokens, teacher-forced (the decoder is fed the correct previous tokens, not its own guesses) — the checkpoint's own training objective. Because everything before the decoder is frozen, the encoder output for every training line is computed once under no gradient and cached (the **frozen encoder cache**, 587 × 1024 numbers per line), and each step runs only the decoder on those cached states — the loss equals the full model's loss exactly, at a fraction of the cost. AdamW without weight decay at a fixed learning rate, gradient clipping at 1.0, seeded shuffling (seed 0, printed below), no scheduler, no augmentation. Epoch 0 records the frozen model's validation rates; every epoch is scored on the 60 validation lines with the same beam search, and the epoch with the **lowest validation CER** is kept — the test split plays no part in that choice. If no epoch beats the frozen model on validation, the selector keeps epoch 0 and that outcome is reported.

**Adaptation always starts from the pretrained model.** If this cell runs again (for example with `LEARNING_RATE = 1e-4` in Section 10), it first reloads the pretrained model from the verified snapshot, so the new training never stacks on the previous one, epoch 0 really is the frozen model, and the exported adapter holds exactly what this run trained; `pipe.adapt` itself refuses a model that is already adapted.

**Run-to-run spread.** Beam search is deterministic only for fixed weights, device and dtype, and GPU training is not bit-reproducible across GPU models or library builds: expect your epoch rates to differ from the recorded run in the second or third decimal (the release record treats a few hundredths as the expected spread), and the kept epoch to move by one when two epochs are that close.

**Predict before running:** will the validation CER fall in every epoch? Will the training loss and the validation CER keep moving together?

In [ ]:
EPOCHS = 6  # @param {type:"integer"}
LEARNING_RATE = 5e-5  # @param {type:"number"}
BATCH_SIZE = 8  # @param {type:"integer"}


def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4)}
    if entry.get('val'):
        row.update({'val_' + k: round(entry['val'][k], 3) for k in METRICS})
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)


reset_to_pretrained()
t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, max_new_tokens=LINE_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'task': adapt_result['task'], 'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'first_trainable_layer': adapt_result['first_trainable_layer'], 'learning_rate': adapt_result['lr'], 'seed': adapt_result['seed'], 'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'selection_split': 'validation', 'loss': adapt_result['loss'], 'cache_seconds': adapt_result['cache_seconds'], 'seconds': adapt_seconds})

{'epoch': 0, 'train_loss': None, 'val_cer': 0.969, 'val_wer': 0.981, 'val_cer_macro': 0.94, 'val_exact_match': 0.017, 'note': 'frozen model'}


{'epoch': 1, 'train_loss': 6.6341, 'val_cer': 0.969, 'val_wer': 1.036, 'val_cer_macro': 1.01, 'val_exact_match': 0.017}


{'epoch': 2, 'train_loss': 5.8761, 'val_cer': 0.821, 'val_wer': 1.015, 'val_cer_macro': 0.85, 'val_exact_match': 0.0}


{'epoch': 3, 'train_loss': 5.2364, 'val_cer': 0.852, 'val_wer': 1.005, 'val_cer_macro': 0.878, 'val_exact_match': 0.017}


{'epoch': 4, 'train_loss': 4.5353, 'val_cer': 0.801, 'val_wer': 1.087, 'val_cer_macro': 0.84, 'val_exact_match': 0.0}


{'epoch': 5, 'train_loss': 3.8282, 'val_cer': 0.808, 'val_wer': 1.024, 'val_cer_macro': 0.838, 'val_exact_match': 0.0}


{'epoch': 6, 'train_loss': 3.1267, 'val_cer': 0.799, 'val_wer': 1.032, 'val_cer_macro': 0.835, 'val_exact_match': 0.0}
{'task': '<OCR>', 'trainable_parameters': 67188736, 'total_parameters': 776505344, 'first_trainable_layer': 8, 'learning_rate': 5e-05, 'seed': 0, 'best_epoch': 6, 'selection': 'lowest validation CER', 'selection_split': 'validation', 'loss': 'sequence-to-sequence cross-entropy over the transcript tokens and the end token, teacher-forced from the decoder start token; computed on the cached frozen encoder outputs', 'cache_seconds': 351.656, 'seconds': 786.5}


**What to notice:** epoch 0 is labelled the frozen model, how the training loss and the validation CER move from epoch to epoch, and which epoch is kept.

<details><summary>Check your reasoning</summary>No, and no. In the recorded T4 run the validation CER fell from 0.969 (epoch 0, the frozen model; epoch 1 also 0.969) to 0.799 at epoch 6, but not in every epoch — 0.821 at epoch 2, then 0.852, 0.801, 0.808, 0.799 — while the training loss fell steadily from 6.63 (epoch 1) to 3.13. The validation curve flattened near 0.80 while the loss kept falling: the decoder fits the 600 training transcripts better than it learns to read. Four decoder layers move the model from silence to something that is not yet reading: the frozen encoder's representation of a 128-px cursive line squashed into 768 × 768 is the bottleneck, not the optimiser. The build record's faster rates (1e-4 and 2e-4 for eight epochs) reached the same plateau (test CER 0.798 and 0.807). The sibling GOT-OCR 2.0 row reached 0.759 on the same split with the same design; the two models are compared in the card, not here.</details>

## 8. Held-out evaluation

The test lines were never used for training or epoch selection, and no test image appears in the training or validation splits — but, as Section 4 explained, lines of the same page and hand can, so this is a same-collection estimate. The adapted model is scored exactly as the frozen model was in Section 6 and the four systems are put side by side. Read it in this order: **CER** first (the measure the epoch was selected on), then **WER** (whether whole words are read), then the hypothesis length (`hyp_chars` against `ref_chars`), then the exact-match rate. The cell then repeats the frozen and adapted rates on the test lines whose transcript does **not** occur in training, so you can see whether the repeated formulaic lines carry the result. `adapted_beats_frozen` and `adapted_beats_both_baselines` record the outcome; a result where the adapted model does not beat the frozen one (possible on your own data, for example printed text the frozen model already reads) is reported in the evaluation report and the run continues to the export — it is a finding, not an error. A **run history** keeps one row per pass through this cell, so an experiment can be compared with the default run (Section 10). One hundred and forty lines from one seeded split give **no dispersion estimate**, and the run-to-run spread of Section 7 applies here too; the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark, and a result on one French council's minutes says nothing about other hands, other languages or other scripts until you measure them.

**Predict before running:** if the CER falls, will the WER fall with it? Will any of the 140 lines be read exactly?

In [ ]:
adapted_test = pipe.evaluate(test_records, max_new_tokens=LINE_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, batch_size=EVAL_BATCH_SIZE)
adapted_val = pipe.evaluate(val_records, max_new_tokens=LINE_MAX_NEW_TOKENS, num_beams=NUM_BEAMS, batch_size=EVAL_BATCH_SIZE)
comparison = {metric: {'empty': round(baseline_empty[metric], 3), 'constant': round(baseline_constant[metric], 3), 'frozen': round(frozen_test[metric], 3), 'adapted': round(adapted_test[metric], 3)} for metric in METRICS}
comparison['delta_vs_frozen'] = {metric: round(adapted_test[metric] - frozen_test[metric], 3) for metric in METRICS}
comparison['hypothesis_length'] = {'ref_chars': adapted_test['ref_chars'], 'frozen_hyp_chars': frozen_test['hyp_chars'], 'adapted_hyp_chars': adapted_test['hyp_chars'], 'frozen_truncated': frozen_test['truncated'], 'adapted_truncated': adapted_test['truncated']}
for key, row in comparison.items():
    print({key: row})

# FL-M3: the same rates without the test lines whose transcript also occurs in training (Section 4).
seen_ids = set(overlap['test'])
unseen = [i for i, r in enumerate(test_records) if r['id'] not in seen_ids]
transcript_breakdown = {'all_test_lines': len(test_records), 'transcript_also_in_train': len(seen_ids), 'transcript_unseen': len(unseen)}
if unseen and seen_ids:
    unseen_records = [test_records[i] for i in unseen]
    for label, run in (('frozen', frozen_test), ('adapted', adapted_test)):
        rates = ocr_metrics([run['hypotheses'][i] for i in unseen], unseen_records)
        transcript_breakdown[label + '_unseen_transcripts'] = {'cer': round(rates['cer'], 3), 'wer': round(rates['wer'], 3), 'n': rates['n']}
        transcript_breakdown[label + '_all'] = {'cer': round(run['cer'], 3), 'wer': round(run['wer'], 3), 'n': run['n']}
print({'transcript_breakdown': transcript_breakdown})

adapted_beats_frozen = adapted_test['cer'] < frozen_test['cer']
adapted_beats_both_baselines = adapted_test['cer'] < min(baseline_empty['cer'], baseline_constant['cer'])
if adapt_result['best_epoch'] == 0:
    reading = 'The selector kept epoch 0, so the adapted model is the frozen model: no epoch beat it on validation.'
elif not adapted_beats_frozen:
    reading = 'The adapted CER is not below the frozen CER on the test lines: this adaptation did not help here. The export and reload below still run.'
elif adapted_test['wer'] >= frozen_test['wer']:
    reading = 'CER fell but WER did not: the gain is in characters inside words, and no more whole words are read than before.'
else:
    reading = 'CER and WER both fell: more characters and more whole words are read than by the frozen model.'
print({'adapted_beats_frozen': adapted_beats_frozen, 'adapted_beats_both_baselines': adapted_beats_both_baselines, 'best_epoch': adapt_result['best_epoch']})
print('Reading: ' + reading)
run_history = globals().get('run_history', [])
run_history.append({'run': len(run_history) + 1, 'data': 'BYOD' if USE_BYOD else 'sample', 'epochs': EPOCHS, 'learning_rate': LEARNING_RATE, 'best_epoch': adapt_result['best_epoch'], 'frozen_cer': round(frozen_test['cer'], 3), 'adapted_cer': round(adapted_test['cer'], 3), 'adapted_wer': round(adapted_test['wer'], 3)})
print(f"{'run':>3} {'data':>6} {'epochs':>6} {'lr':>8} {'best':>4} {'frozen':>7} {'adapted':>7} {'wer':>7}")
for h in run_history:
    print(f"{h['run']:>3} {h['data']:>6} {h['epochs']:>6} {h['learning_rate']:>8g} {h['best_epoch']:>4} {h['frozen_cer']:>7.3f} {h['adapted_cer']:>7.3f} {h['adapted_wer']:>7.3f}")
evaluation_report_payload = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'task': '<OCR>',
    'data_source': data_source,
    'dataset_digests': {name: manifest['digest'] for name, manifest in dataset_manifests.items()},
    'splits': disjoint,
    'split_unit': 'line image; no page or writer field, so lines of one page can be in training and test (same-collection estimate)',
    'transcript_overlap': overlap,
    'transcript_breakdown': transcript_breakdown,
    'baselines': {'empty': {k: v for k, v in baseline_empty.items() if k != 'rows'}, 'constant': {k: v for k, v in baseline_constant.items() if k != 'rows'}},
    'frozen_test': {k: v for k, v in frozen_test.items() if k != 'rows'},
    'validation_metrics': {k: v for k, v in adapted_val.items() if k != 'rows'},
    'test_metrics': {k: v for k, v in adapted_test.items() if k != 'rows'},
    'per_line': [{**frozen_row, 'frozen_hypothesis': frozen_hyp, 'adapted_cer': adapted_row['cer'], 'adapted_hypothesis': adapted_hyp} for frozen_row, frozen_hyp, adapted_row, adapted_hyp in zip(frozen_test['rows'], frozen_test['hypotheses'], adapted_test['rows'], adapted_test['hypotheses'], strict=True)],
    'comparison': comparison,
    'adapted_beats_frozen': adapted_beats_frozen,
    'adapted_beats_both_baselines': adapted_beats_both_baselines,
    'reading': reading,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/florence2_vision_language_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report_payload, f, indent=2, ensure_ascii=False)
print({'report': 'outputs/florence2_vision_language_evaluation_report.json'})

{'cer': {'empty': 1.0, 'constant': 0.944, 'frozen': 0.992, 'adapted': 0.797}}
{'wer': {'empty': 1.0, 'constant': 0.999, 'frozen': 1.002, 'adapted': 1.012}}
{'cer_macro': {'empty': 1.0, 'constant': 0.952, 'frozen': 0.962, 'adapted': 0.824}}
{'exact_match': {'empty': 0.0, 'constant': 0.0, 'frozen': 0.0, 'adapted': 0.0}}
{'delta_vs_frozen': {'cer': -0.195, 'wer': 0.01, 'cer_macro': -0.138, 'exact_match': 0.0}}
{'hypothesis_length': {'ref_chars': 6169, 'frozen_hyp_chars': 250, 'adapted_hyp_chars': 3570, 'frozen_truncated': 0, 'adapted_truncated': 0}}
{'transcript_breakdown': {'all_test_lines': 140, 'transcript_also_in_train': 5, 'transcript_unseen': 135, 'frozen_unseen_transcripts': {'cer': 0.992, 'wer': 1.002, 'n': 135}, 'frozen_all': {'cer': 0.992, 'wer': 1.002, 'n': 140}, 'adapted_unseen_transcripts': {'cer': 0.796, 'wer': 1.012, 'n': 135}, 'adapted_all': {'cer': 0.797, 'wer': 1.012, 'n': 140}}}
{'adapted_beats_frozen': True, 'adapted_beats_both_baselines': True, 'best_epoch': 6}
Readin

**What to notice:** the delta row, the hypothesis length, the transcript breakdown, the printed reading and the run history.

<details><summary>Check your reasoning</summary>No, and no. In the recorded T4 run the CER fell 0.992 → **0.797**, past both baselines — one fifth of the characters right — while the WER went 1.002 → 1.012: it did not fall, so the character gain sits inside words and no whole word is read. The hypotheses grew from 0.04 to 0.58 times the reference length (the adapted model writes something for every line, but not enough of it), and 0 of the 140 lines were read exactly. The transcript breakdown from that run's per-line hypotheses: on the 135 test lines whose transcript does not occur in training the adapted CER is also **0.797** (frozen 0.992); the 5 repeated lines (84 characters) scored 0.821, so the verbatim repeats do not carry the result. What the breakdown cannot remove is the shared pages and hands (Section 4).</details>

## 9. Look at the lines, re-run the three capabilities, export the adapter and reload it

Six held-out lines are written as panels (`outputs/florence2_vision_language_examples/`: the line image with the reference, the frozen transcript and the adapted transcript beneath it) so the numbers can be checked by eye — compare the adapted rows with the references, not only with the empty frozen rows. The drawing from Section 5 is then run again through all three capabilities by the adapted model — the decoder that was tuned serves every task token, so this is a small look at what the adaptation did *outside* its task and its corpus (one drawing of evidence, not a measurement).

`pipe.save_artifact` writes the trained tensors — the four decoder layers and the embedding norm, 106 tensors and about 269 MB in float32 — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digest of the base `model.safetensors`, the tensor names, the file size and SHA-256, the task, the training configuration and the epoch history (OUT8). An **adapter** is only the changed tensors; it is meaningless without the pinned base. `Florence2Pipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest, its digest and its exact tensor set **before** deserialising, refuses any tensor outside the last four decoder layers and the embedding norm, and overlays the tensors onto a freshly loaded base — a new object from files, not the in-memory model (VER2). The cell checks for identical transcripts on eight test lines (**reload parity**, VER4) and stops with an explanation if they differ. Under BYOD `result.json` records your upload in place of the Belfort corpus block.

**Predict before running:** after tuning the decoder on handwriting, will the drawing's caption, its box and its OCR stay the same? Will reload parity hold?

In [ ]:
import shutil

examples_dir = Path('outputs/florence2_vision_language_examples')
shutil.rmtree(examples_dir, ignore_errors=True)
examples_dir.mkdir(parents=True)
caption_font = ImageFont.load_default(size=18)
for record, frozen_hyp, adapted_hyp in zip(test_records[:6], frozen_test['hypotheses'][:6], adapted_test['hypotheses'][:6], strict=True):
    line = record['image']
    width = min(1400, line.width)
    line = line.resize((width, max(1, round(line.height * width / record['image'].width))))
    sheet = Image.new('RGB', (max(width, 1400), line.height + 96), (255, 255, 255))
    sheet.paste(line, (0, 0))
    marker = ImageDraw.Draw(sheet)
    for i, (tag, text) in enumerate((('REF', record['text']), ('FROZEN', frozen_hyp), ('ADAPTED', adapted_hyp))):
        marker.text((8, line.height + 6 + i * 28), f'{tag}: {text[:140]}', fill=(20, 20, 20) if tag != 'FROZEN' else (150, 40, 40), font=caption_font)
    sheet.save(examples_dir / f"{record['id']}.png")
print({'examples': sorted(p.name for p in examples_dir.iterdir()), 'rows': ['reference', 'frozen transcript', 'adapted transcript']})

adapted_results, adapted_timings, adapted_checks, adapted_report = run_capabilities(pipe, 'adapted')

artifact_dir = Path('outputs/florence2_vision_language_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'florence2_vision_language', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...', 'task': artifact_manifest['adapter']['task'], 'best_epoch': artifact_manifest['adapter']['best_epoch']})

reloaded = Florence2Pipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = [item['text'] for item in pipe.transcribe([r['image'] for r in test_records[:8]], max_new_tokens=LINE_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)]
after = [item['text'] for item in reloaded.transcribe([r['image'] for r in test_records[:8]], max_new_tokens=LINE_MAX_NEW_TOKENS, num_beams=NUM_BEAMS)]
parity = {'identical_lines': sum(a == b for a, b in zip(before, after, strict=True)), 'of': len(before)}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
reloaded = None  # free the second copy of the model before any re-run
gc.collect()
if parity['identical_lines'] != parity['of']:
    raise RuntimeError(f'Reload parity failed: {parity}. The saved adapter does not reproduce the model that was evaluated in Section 8. Re-run from Section 7 (it reloads the pretrained model before adapting), or restart the session and choose Run all.')

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'model_license': MODEL_LICENSE,
    'snapshot': {'path': str(WEIGHTS_DIR), 'files': snapshot['files'], 'total_bytes': snapshot.get('total_bytes'), 'fetched_this_run': fetched, 'weight_file': WEIGHTS_FILE, 'weight_format': 'safetensors, digest-verified', 'weight_sha256': pipe.weight_sha256},
    'data_source': data_source,
    'corpus': None if USE_BYOD else {'name': CORPUS_NAME, 'repo': CORPUS_REPO, 'revision': CORPUS_REVISION, 'file': CORPUS_FILE, 'license': CORPUS_LICENSE, 'language': CORPUS_LANGUAGE, 'row_groups': sorted(ROW_GROUP_PINS), 'shard_bytes': CORPUS_BYTES},
    'byod': byod,
    'split_unit': 'line image (no page or writer field)',
    'inference_contract': {'input_manifest': input_manifest, 'drawing': {'name': drawing_name, 'sha256': drawing_sha256, 'ocr_reference': drawing_reference, 'drawn_square': drawn_square}, 'tasks_exposed': list(TASKS), 'frozen': {'capabilities': {task: {'result': r['result'], 'generated_text': r['generated_text'], 'generation': r['generation'], 'text_input': CAPABILITIES[task].get('text_input'), 'seconds': frozen_timings[task]} for task, r in frozen_results.items()}, 'checks': frozen_checks, 'report': frozen_report}, 'adapted': {'capabilities': {task: {'result': r['result'], 'generated_text': r['generated_text'], 'generation': r['generation'], 'text_input': CAPABILITIES[task].get('text_input'), 'seconds': adapted_timings[task]} for task, r in adapted_results.items()}, 'checks': adapted_checks, 'report': adapted_report}, 'output_files': ['outputs/florence2_vision_language_preview_frozen.png', 'outputs/florence2_vision_language_preview_adapted.png']},
    'comparison': comparison,
    'transcript_breakdown': transcript_breakdown,
    'adapted_beats_frozen': adapted_beats_frozen,
    'adapted_beats_both_baselines': adapted_beats_both_baselines,
    'reading': reading,
    'adaptation': {'best_epoch': adapt_result['best_epoch'], 'selection': adapt_result['selection'], 'learning_rate': LEARNING_RATE, 'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'seed': adapt_result['seed']},
    'examples': 'outputs/florence2_vision_language_examples',
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes'], 'tensors': len(artifact_manifest['tensors'])},
    'reload_parity': parity,
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'transformers': transformers.__version__, 'pillow': PIL.__version__, 'device': pipe.device, 'source': pipe.source, 'dtype': pipe.dtype},
}
with open('outputs/florence2_vision_language_result.json', 'w', encoding='utf-8') as handle:
    json.dump(result_payload, handle, indent=2, ensure_ascii=False)
print(sorted(os.listdir('outputs')))

{'examples': ['belfort-test-112.png', 'belfort-test-156.png', 'belfort-test-48.png', 'belfort-test-642.png', 'belfort-test-761.png', 'belfort-test-763.png'], 'rows': ['reference', 'frozen transcript', 'adapted transcript']}


{'adapted': {'seconds': {'<CAPTION>': 0.843, '<OD>': 0.82, '<OCR>': 0.747}, 'checks': {'caption_is_text': True, 'od_boxes_and_labels_aligned': True, 'od_boxes_inside_image': True, 'ocr_is_text': True, 'deterministic_settings': True, 'identity_reported': True}, 'caption': 'Dim 2026 logo with a red and blue circle.', 'detections': [('poster', [0, 0, 511, 511])], 'ocr': 'DIMER 2026', 'ocr_cer': {'character_error_rate': 0.0}, 'verdict': 'sample-sanity'}}


{'artifact': 'outputs/florence2_vision_language_adapter', 'format': 'org.valcorza.florence-2-large-community.adapter.v1', 'tensors': 106, 'bytes': 268769008, 'sha256': '3c79728c44b45675...', 'task': '<OCR>', 'best_epoch': 6}


{'reload_parity': {'identical_lines': 8, 'of': 8}, 'reloaded_best_epoch': 6}
['florence2_vision_language_adapter', 'florence2_vision_language_evaluation_report.json', 'florence2_vision_language_example_line.png', 'florence2_vision_language_examples', 'florence2_vision_language_input_manifest.json', 'florence2_vision_language_preview_adapted.png', 'florence2_vision_language_preview_frozen.png', 'florence2_vision_language_result.json', 'florence2_vision_language_train.csv']


**What to notice:** the panels (references against adapted transcripts), what changed in the drawing's caption, box and OCR, the artifact's tensor count and size, and the reload-parity line.

<details><summary>Check your reasoning</summary>Partly, and yes. In the recorded T4 run the caption changed from `a red and blue circle with the word dimer 2026 on it` to `Dim 2026 logo with a red and blue circle.`, the single `poster` box covering the drawing stayed, and the OCR still read `DIMER 2026` exactly (CER 0.000) — tuning the shared decoder on handwriting touched the caption task it was never trained on, which is one drawing of evidence, not a measurement. The artifact held 106 tensors (4 decoder layers × 26 plus the 2 tensors of the embedding layer norm), 268,769,008 bytes, and reload parity was 8/8: the exported adapter is the model that was evaluated. Parity holds because the adaptation started from the pretrained base, so the adapter carries every tensor that differs from it.</details>

## 10. Your turn — change one thing: a higher learning rate

Optional; **Predict → Change one thing → Run → Observe → Explain**. It retrains, so it takes about as long as Sections 7–9 did (Section 7 alone took 822 s on the T4 record).

1. **Predict:** with `LEARNING_RATE = 1e-4` (twice the default), will the validation CER fall faster in the first epochs, will it end lower than the default run's plateau near 0.80, and will the training loss end lower? Write your guesses next to the default run's row in the Section 8 run history.
2. **Change:** in Section 7 set `LEARNING_RATE = 1e-4`. Change nothing else.
3. **Run:** select the Section 7 cell and choose **Runtime → Run after**. Section 7 reloads the pretrained model before adapting, Section 8 compares against the same Section 6 frozen score and adds a row to the run history, and Section 9 exports and reloads the new adapter.
4. **Observe:** in Section 7, epoch 0 shows the *frozen* validation CER again (0.969 in the recorded run), not the adapted one — the sign that the model was reset; read how the loss and the validation CER move. In Section 8, compare the two rows of the run history and the reading line.
5. **Explain:** in one or two sentences, did doubling the learning rate move the held-out CER, and is the difference between the two runs larger than the run-to-run spread and what 140 lines can resolve?

<details><summary>Check your reasoning</summary>The build record ran this rate for eight epochs (not six) and reached a test CER of 0.798 — the same plateau as the default's 0.797 — and 2e-4 reached 0.807. Expect the training loss to fall faster and the validation CER to reach the plateau sooner, and then stop. That is the point of the activity: when every rate tried lands on the same plateau, the limit is what the frozen encoder passes to the decoder, not how fast the decoder learns. Your six-epoch numbers are your own result; a difference of a few thousandths between the two rows is not evidence that one rate is better.</details>

## Interpretation and limits

A prompt-driven document model reads the print it was trained on, and nineteenth-century cursive French is not among it: the frozen `<OCR>` scores a character error rate of 0.992 on the Belfort lines — silence, a dash or nothing for almost every line. A bounded fine-tuning of the last four decoder layers on 600 transcribed lines moves it off the floor but not far (0.797 CER and 1.012 WER in the build record, 0 of the held-out lines exact; the constant-transcript baseline scores 0.944) and the validation curve plateaus there at every rate tried, with a 269 MB adapter that reloads line-for-line. That is the claim: the adaptation contract works end to end on one task of a multi-task model with a real labelled set, and the numbers it produces are read as micro and macro rates, against two non-adapted baselines and the frozen model, with the hypothesis length beside them rather than in isolation — and they say that a decoder-only adapter cannot make this encoder read cursive. The obvious next experiment, adapting the BART encoder's last layers too (or the vision tower), is not in this repository.

The test split is 140 lines from one seeded draw of one 800-line sample, the validation split that picks the epoch is 60, and both rates are corpus edit distances over one crowdsourced transcription — not a benchmark, not a measure of reading order or layout. **The split is by line image, not by page:** the dataset has no page or writer field, so the held-out lines can share pages and hands with the training lines, and the held-out rate is a same-collection estimate that is likely optimistic for a new register or a new hand (the verbatim-repeated transcripts, which Section 8 removes, do not change it). So a gain here says the contract works on one council's minutes, not that the adapted model handles other hands, other languages, other scripts or your scans. The decoder that was tuned serves every task token: the drawing re-run in Section 9 is one image of evidence about what the tuning did to `<CAPTION>` and `<OD>`, not a measurement, and a deployment that needs the other tasks must measure them after adapting. The decoder was adapted, not the vision tower: what the encoder cannot resolve in a 128-px line squashed into a 768 × 768 square stays unread.

Three things to carry to real data. **Baselines first:** the empty and constant-transcript rates on *your* transcripts, and the frozen model's hypothesis length, are the numbers to read before any adapted one. **Rates above 1.0:** an uncapped CER tells you the model is generating, not reading; a capped one would hide it. **Leakage:** keep every image in one split (the contract de-duplicates by decoded pixels), and when your data records the page, writer or volume of each line, split by that — lines cut from the same page share a hand, and this notebook's split cannot keep them together because its dataset does not say which page a line came from.

Successful execution proves that the recorded repository revision's package, carried in this standalone notebook, can acquire and digest-verify the pinned model snapshot, fetch and digest-verify a real labelled line set, validate the demonstrated dataset contract with no image in two splits, execute the inference contract for three task tokens and a bounded fine-tuning of one of them with the model's own objective, evaluate against two non-adapted baselines and the frozen model on a held-out split of line images, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, recognition quality on any other hand, language or document family, caption or detection quality after adaptation, or production fitness.

**Optional experiments (they do not affect the default path).** Each one changes one thing; note the numbers you want to compare first (the Section 8 run history keeps one row per pass). In Colab, **Runtime → Run after** runs the selected cell and every cell below it; Sections 5, 6 and 7 reload the pretrained model before they use it, so no experiment inherits an earlier adaptation.

- **A higher learning rate:** Section 10.
- **More epochs:** set `EPOCHS = 10` in Section 7, select Section 7 and choose **Run after**. Does the validation CER keep falling after epoch 6, and which epoch does the selector keep?
- **Greedy decoding:** in Section 6 add the line `NUM_BEAMS = 1` under `LINE_MAX_NEW_TOKENS`, select Section 6 and choose **Run after**. What does giving up beam search cost the frozen and the adapted CER? (Restart and **Run all** to return to 3 beams.)
- **A smaller token budget:** set `LINE_MAX_NEW_TOKENS = 64` in Section 6, select Section 6 and choose **Run after**. How does the `truncated` count change, and what does it do to the CER?
- **Phrase grounding:** in Section 5 add the entry `'<CAPTION_TO_PHRASE_GROUNDING>': {'input': 'image + text', 'output': 'boxes for the phrase', 'max_new_tokens': DEFAULT_MAX_NEW_TOKENS, 'text_input': 'a red square'}` to `CAPABILITIES`, select Section 5 and choose **Run after**. Where does the box for the phrase land, and does the adapted model in Section 9 still find it? (`run_capabilities` passes each entry's `text_input`, and `result.json` records it.)
- **Another split:** change `SPLIT_SEED` in Section 4, select Section 4 and choose **Run after**. How far do the frozen and adapted CER move with a different draw of 140 test lines — more or less than the default run's delta?
- **Your own lines:** BYOD (Section 4, then **Run after**). Before the adapted number, how do the two baselines and the frozen model score on your transcripts?

## Troubleshooting

- **Section 1 stops with "needs a Linux x86_64 runtime".** The locked environment is built from manylinux wheels. Use Google Colab, Kaggle or a Linux Jupyter server; Windows and macOS kernels are not supported.
- **Section 1 fails while downloading** (`uv` wheel, Python build or packages). The runtime needs PyPI and the python-build-standalone download; run the cell again once the network is back. A "size/SHA-256" or "does not match its digest" error means a file was altered: do not edit the cell, regenerate the notebook from the repository.
- **The run is very slow, or the session ends before Section 9.** On CPU the default path takes hours (Prerequisites). Switch to a GPU runtime (**Runtime → Change runtime type**) and choose **Run all** again.
- **"The isolated environment's Python process exited".** Usually the runtime ran out of memory (the model is about 3.1 GB in float32, and Section 9 briefly holds two copies). Restart the session and choose **Run all**. A CUDA out-of-memory error in Section 7 can be avoided with `BATCH_SIZE = 4`.
- **`FileNotFoundError: snapshot file missing` or a `sha256`/`size` `ValueError` in Section 3.** A staged file is incomplete or altered — delete it from `weights/florence-2-large-community/` and run Section 3 again. A row-group digest mismatch in Section 4 is the same for the corpus cache under `weights/belfort/`.
- **`RuntimeError: checkpoint does not match the native Florence-2 architecture` in Section 3.** The staged weights are not the pinned converted checkpoint — re-stage.
- **BYOD is refused in Section 4.** The message names what to fix: a split with too few lines (at least 12 distinct images in all), two files with one name, a missing `transcripts.csv`, an image without a row or a row without an image, a transcript over 512 characters, or a duplicate id. Fix the zip and run Section 4 with **Run after** again. An empty or cancelled upload, or a runtime without the Colab upload dialog, asks you to choose a file or set `BYOD_PATH`.
- **Section 8 reads "did not help" or "The selector kept epoch 0".** Not an error: under these settings the adaptation did not beat the frozen model. Read the epoch history in Section 7; the export and reload in Section 9 still run.
- **Reload parity fails in Section 9.** The saved adapter does not reproduce the evaluated model. Re-run from Section 7 (it reloads the pretrained model first), or restart the session and choose **Run all**.
- **Numbers differ slightly from the recorded run.** GPU models and library builds differ, and because decoding is autoregressive one different token changes the rest of a line; differences in the second or third decimal are expected. Larger differences on the default settings mean a form field was changed.
- **A "slow image processor" notice** from `transformers` is expected and harmless.
- **You re-ran the routing cell in Section 1.** It starts a new isolated process and discards every variable created so far; choose **Run all** again.

## Glossary

- **Task prompt token:** the token such as `<OCR>` or `<CAPTION>` that selects what Florence-2 generates.
- **DaViT / BART:** the vision tower that turns the image into 577 visual tokens, and the encoder–decoder language model that reads them with the prompt and generates text.
- **Beam search:** decoding that keeps the `num_beams` most likely partial outputs at each step and returns the best; deterministic for fixed weights, device and dtype. Greedy decoding is one beam.
- **Levenshtein distance:** the minimum number of character (or word) insertions, deletions and substitutions that turn one string into another.
- **CER / WER (micro):** total character (word) edits over total reference characters (words) across the split; **macro** is the mean of the per-line rates. Not capped at 1.0.
- **Empty / constant-transcript baseline:** systems that never look at the image — predicting nothing, or one training transcript (the **medoid**, the line closest on average to the others) for every line.
- **Teacher forcing:** during training the decoder is fed the correct previous tokens of the transcript, not its own predictions.
- **Frozen encoder cache:** the encoder output for each training line, computed once without gradients and reused every epoch because nothing before the decoder is trained.
- **Epoch / validation selection:** one pass over the training lines; the epoch kept is the one with the lowest validation CER (epoch 0 = the frozen model).
- **Training / validation / test split:** lines used to train, to choose the epoch, and to report the final result once; split by line image, so no image is in two splits — but not by page (Section 4).
- **Same-collection estimate:** a held-out score on data drawn from the same pages, hands and registers as the training data; usually optimistic for new sources.
- **safetensors / adapter / reload parity:** a tensor file format that cannot run code when loaded; the saved trained tensors; and the check that a pipeline rebuilt from them and the pinned base gives the same transcripts.

## Conclusion (your notes)

Optional personal notes, not a required submission. Complete each sentence from your own run:

- The task: from ___ to ___. On the test lines the empty baseline scored CER ___, the constant transcript ___, the pretrained model ___ and the adapted model ___ (kept epoch ___).
- WER moved from ___ to ___, so the change in CER reflects ___ (characters inside words / whole words).
- Without the test lines whose transcript is also in training, the adapted CER was ___; the split cannot rule out ___ because ___.
- One failure mode I saw in the panels: ___. On the drawing, the adaptation changed ___.
- A higher learning rate (Section 10) changed ___; the difference between the runs is / is not larger than one seeded split can resolve, because ___.
- Before trusting this adaptation on my own documents I would measure ___ on ___, split by ___, with the baselines first.

## References

- Repository README: https://github.com/kurtvalcorza/florence2-vision-language-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/florence2-vision-language-pipeline/blob/main/MODEL_CARD.md
- Weight provenance and pin history: https://github.com/kurtvalcorza/florence2-vision-language-pipeline/blob/main/docs/WEIGHTS.md
- Pinned converted checkpoint: https://huggingface.co/florence-community/Florence-2-large
- Original weights and licence: https://huggingface.co/microsoft/Florence-2-large
- Florence-2 paper (Xiao et al., 2023): https://arxiv.org/abs/2311.06242
- Transformers Florence-2 documentation: https://huggingface.co/docs/transformers/model_doc/florence2
- Belfort-line dataset (Teklia, MIT): https://huggingface.co/datasets/Teklia/Belfort-line — Tarride et al., Handwritten Text Recognition from Crowdsourced Annotations (HIP 2023): https://doi.org/10.1145/3604951.3605517
- Sibling row on the same split: https://github.com/kurtvalcorza/got-ocr2-pipeline
- DIMER Notebook Specification 2.2 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)